US Post-Market

1. Settings

In [1]:
import html
import io
import logging
import os
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, wait
from datetime import datetime, timedelta, timezone
from functools import partial
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv
from IPython.display import Markdown, display
from requests.adapters import HTTPAdapter

# 《股市》區塊
INDICES = {"^DJI": "道瓊", "^GSPC": "標普", "^IXIC": "納指", "^SOX": "費半", "^VIX": "VIX"}
ADRS = {"TSM": "台積", "UMC": "聯電", "ASX": "日月光"}           # 台股 ADR
# GICS 類股：維基百科名稱 → (中文, S&P 500 類股指數, 備援 SPDR ETF)
SECTORS = {
    "Information Technology": ("資訊科技", "^SP500-45", "XLK"),
    "Communication Services": ("通訊服務", "^SP500-50", "XLC"),
    "Consumer Discretionary": ("非必需消費", "^SP500-25", "XLY"),
    "Consumer Staples": ("必需消費", "^SP500-30", "XLP"),
    "Energy": ("能源", "^GSPE", "XLE"),             # Yahoo 的能源類股指數代號是 ^GSPE（沒有 ^SP500-10）
    "Financials": ("金融", "^SP500-40", "XLF"),
    "Health Care": ("醫療保健", "^SP500-35", "XLV"),
    "Industrials": ("工業", "^SP500-20", "XLI"),
    "Materials": ("原材料", "^SP500-15", "XLB"),
    "Utilities": ("公用事業", "^SP500-55", "XLU"),
    "Real Estate": ("不動產", "^SP500-60", "XLRE"),
}
FOCUS = {"GS": "Goldman Sachs", "MSFT": "Microsoft", "GOOGL": "Google", "MS": "Morgan Stanley", "AMZN": "Amazon",
         "JPM": "JP Morgan", "META": "Meta", "AAPL": "Apple", "TSLA": "Tesla", "AMD": "AMD", "NVDA": "NVIDIA",
         "SMCI": "SuperMicro"}
ASSETS = {"^TNX": "美債", "DX-Y.NYB": "美元", "GC=F": "黃金", "CL=F": "西德州", "BZ=F": "布蘭特"}
BREADTH = ["RSP", "SPY"]           # 等權重 vs 市值加權
# 科技與 AI 觀察名單：不論是否為 S&P 500 成分股，每天都列在第 4 格；財報會進第 5 格的財報表。可自行增減
AI_WATCH = {"AVGO": "Broadcom", "MU": "Micron", "INTC": "Intel", "MRVL": "Marvell", "ORCL": "Oracle", "PLTR": "Palantir",
            "DELL": "Dell", "ARM": "Arm", "ASML": "ASML", "CRWV": "CoreWeave", "NBIS": "Nebius", "SPCX": "SpaceX",
            "CBRS": "Cerebras", "ALAB": "Astera Labs", "LITE": "Lumentum"}
ALWAYS = ["Information Technology"]   # 第 4 格一定列出的類股（範例 12 篇都寫資訊科技，不論排名）
logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # 隱藏 yfinance 的逐檔錯誤訊息，改由第 2 格整理
warnings.filterwarnings("ignore", module=r"yfinance")      # yfinance 內部的 pandas 棄用警告（與資料無關，會洗版）

MOVERS_PER_SECTOR = 3   # 強弱類股各列出幾檔領漲 / 領跌
TECH_MOVERS = 5         # 資訊科技列出幾檔領漲 / 領跌
# S&P 500 以外的個股（Yahoo 篩選器）：(市值下限美元, 漲跌幅門檻 %)
SCREEN_TECH = (20e9, 4)     # 科技、通訊服務
SCREEN_OTHER = (10e9, 6)    # 不分產業
SCREEN_MAX = 8          # 第 4 格每一行最多列幾檔
EXTRA_NEWS = 6          # 非成分股、觀察名單各最多幾檔進第 5 格（查評等與新聞）
WATCH_MOVE = 3          # 觀察名單漲跌超過幾 % 才進第 5 格
STOCK_NEWS_MAX = 40     # 第 5 格最多幾檔抓 Finnhub 新聞（免費版每分鐘 60 次，要留給第 7、8 格）
NEWS_HOURS = 20         # 新聞抓最近幾小時
NEWS_PER_GROUP = 8
SP500_CACHE = "sp500_constituents.csv"
TIMEOUT = 15
YAHOO_WORKERS = 6       # 同時對 Yahoo 發出的請求數（實測 6 沒問題；太多會被限流）
FINNHUB_WORKERS = 8     # Finnhub 免費版：每分鐘 60 次、每秒 30 次
TW = ZoneInfo("Asia/Taipei")
NY = ZoneInfo("America/New_York")
UA = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")}

# 共用連線：連同一個網站多次時不必每次重新建立 HTTPS 連線
HTTP = requests.Session()
HTTP.headers.update(UA)
HTTP.mount("https://", HTTPAdapter(pool_connections=20, pool_maxsize=20))

# 金鑰放在同資料夾的 .env（不要寫進筆記本）
load_dotenv(".env")
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "").strip()   # 個股新聞、財報、綜合新聞
FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()         # 美債殖利率備援
print("Finnhub：", "已設定" if FINNHUB_API_KEY else "未設定", "｜FRED：", "已設定" if FRED_API_KEY else "未設定")


def clean(text, limit=200):
    text = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html.unescape(text or ""))).strip()   # &#10; 這類 HTML 字元也還原
    return text if len(text) <= limit else text[:limit].rstrip() + "…"


# 不使用的來源（Reuters、Bloomberg 與其他付費牆媒體）：網址或來源名稱符合就略過
PAYWALLED = ("reuters.com", "bloomberg.com", "wsj.com", "ft.com", "barrons.com", "marketwatch.com",
             "economist.com", "nytimes.com", "investors.com", "washingtonpost.com")
PAYWALLED_NAMES = ("reuters", "bloomberg", "wall street journal", "wsj", "financial times", "barron", "marketwatch",
                   "the economist", "new york times", "investor's business daily", "washington post")


def blocked(url="", source=""):
    """Finnhub 的 Reuters 新聞網址是 news.google.com，只看網址擋不掉，所以來源名稱也要比對"""
    u, s = (url or "").lower(), (source or "").lower()
    return any(p in u for p in PAYWALLED) or any(p in s for p in PAYWALLED_NAMES)


# ---- 新聞分類與過濾（第 5、7、8 格共用）----
MACRO_RE = re.compile(   # 總經與聯準會
    r"\b(fed|fomc|federal reserve|central banks?|inflation|cpi|ppi|pce|consumer prices?|producer prices?|wholesale prices?|"
    r"jobs report|payrolls?|jobless|unemployment|labor market|job openings|jolts|retail sales|gdp|ism|pmi|"
    r"consumer (confidence|sentiment)|durable goods|housing starts|home sales|rate hikes?|rate cuts?|interest rates?|"
    r"treasur(y|ies)|yields?|tariffs?|recession|economy|economic|mortgage rates?)\b", re.I)
SPEAK_RE = re.compile(   # Fed 官員談話：標題慣用 "Fed's Goolsbee…"、"Dallas Fed's Logan…"、"Fed Governor Waller…"
    r"\bFed['’]s (?!(Inflation|Rate|Interest|Policy|Decision|Meeting|Move|Next|First|Latest|New|Big|Hawkish|Dovish|Reaction|"
    r"Balance|Preferred|Favorite|Beige|Fight|Path|Dilemma|Independence|Mistake|Outlook|Stance|Plan|Target|January|February|"
    r"March|April|May|June|July|August|September|October|November|December)\b)[A-Z][a-z]+|"
    r"(?i:\bfed (chair|vice chair|governor|president|chief|officials?)\b)")
TECH_RE = re.compile(    # 科技與 AI
    r"\b(ai|a\.i\.|artificial intelligence|chips?|chipmakers?|semiconductors?|gpus?|data ?centers?|cloud|hyperscalers?|"
    r"neoclouds?|software|tech|technology|nvidia|amd|broadcom|tsmc|micron|intel|coreweave|openai|anthropic|microsoft|"
    r"alphabet|google|meta|amazon|aws|apple|tesla|oracle|palantir|super micro|dell|spacex|nasdaq)\b", re.I)
JUNK_RE = re.compile(    # 理財建議、加密貨幣、他國匯率與金價、通訊社稿件的轉載
    r"should you buy|stocks? to buy|i['’]d buy|better buy|buy and hold|never sell|screaming buy|millionaire|"
    r"price prediction|prediction:|\b(two|three|four|five|\d+) (\w+ )?stocks\b|(outpaced|outpaces) the (stock )?market|"
    r"bitcoin|\bbtc\b|ethereum|\bxrp\b|dogecoin|altcoin|capital gains tax|\b13f\b|dividend stocks?|etfs? (are|is) buying|"
    r"\$[\d,]+ (investment|invested)|social security|rupee|ringgit|naira|rupiah|\bpeso\b|gold (price|rate)s? in |\bsjc\b|"
    r"^(forex|treasuries|precious|global markets|us stocks|morning bid|breakingviews)\b|[-–—|] ?(reuters|bloomberg)$", re.I)
JUNK_SOURCES = ("motley fool", "insider monkey", "coin", "crypto")
MOVE_RE = re.compile(    # 標題在講股價異動的原因（財報、財測、評等、併購、大漲大跌）
    r"\b(earnings|results|revenue|guidance|forecast|outlook|beat|miss|upgrad|downgrad|price target|soar|surg|jump|rall|"
    r"climb|plung|sink|tumbl|slid|slump|skyrocket|crash|acqui|merger|deal)\w*|\b(falls?|drops?|rises?|gains?)\b", re.I)
PREFERRED = ("cnbc", "yahoo", "associated press", "ap news", "axios", "seeking alpha", "investing.com", "fxstreet", "kitco",
             "morningstar", "business insider", "fortune", "forbes", "quartz", "investinglive", "benzinga", "fox business",
             "cnn", "npr", "pbs", "the hill", "politico", "techcrunch", "the verge", "barchart", "oilprice", "thestreet",
             "cbs news", "nbc news", "abc news", "forex.com")


def junk(title, source=""):
    return bool(JUNK_RE.search(title or "")) or any(s in (source or "").lower() for s in JUNK_SOURCES)


def preferred(source):
    """專屬來源（CNBC、Fed、OilPrice、鉅亨網）或 Google News／Finnhub 轉來的主要財經媒體"""
    return "/" not in source or any(p in source.lower() for p in PREFERRED)


def dedupe(items, title=lambda it: it[1]):
    """標題相同，或用字有一半以上重疊（同一則消息的不同轉載）的只留第一則"""
    kept, keys, sigs = [], set(), []
    for it in items:
        key = re.sub(r"\W+", "", title(it).lower())
        w = set(re.findall(r"[a-z0-9]{4,}", title(it).lower()))
        if not key or key in keys or (len(w) >= 4 and any(len(w & s) >= 0.5 * min(len(w), len(s)) for s in sigs)):
            continue
        keys.add(key)
        kept.append(it)
        if len(w) >= 4:
            sigs.append(w)
    return kept


def gnews(query):
    return ("https://news.google.com/rss/search?q=" + quote_plus(f"{query} when:1d")
            + "&hl=en-US&gl=US&ceid=US:en")


# Google News 查詢：Fed 官員談話（標題慣用 "Fed's Goolsbee says…"）、利率政策、經濟數據、科技股
GN_FED = gnews('"Fed\'s" OR "Fed president" OR "Fed governor" OR "Fed chair"')
GN_POLICY = gnews("intitle:Fed inflation OR rates OR CPI OR hike OR cut")
GN_DATA = gnews('"jobs report" OR "jobless claims" OR "inflation report" OR "CPI report" OR "consumer prices" OR '
                '"producer prices" OR "retail sales" OR "consumer confidence" OR "consumer sentiment" OR '
                '"ISM manufacturing" OR "ISM services" OR "PCE inflation" OR "job openings" OR "durable goods" OR '
                '"housing starts" OR "GDP report"')
GN_TECH = gnews('"AI stocks" OR "chip stocks" OR "tech stocks" OR "semiconductor stocks"')


def rss(name, url):
    """RSS → [(時間, 標題, 連結, 來源, 摘要)]。Google News 會帶出原始媒體名稱，付費牆媒體在這裡略過"""
    r = HTTP.get(url, timeout=TIMEOUT)
    r.raise_for_status()
    items = []
    for e in feedparser.parse(r.content).entries:
        t = e.get("published_parsed") or e.get("updated_parsed")
        src = e.get("source") or {}
        if src and blocked(src.get("href"), src.get("title")):
            continue
        title, publisher = clean(e.get("title"), 300), src.get("title", "")
        if publisher and title.endswith(" - " + publisher):
            title = title[: -(len(publisher) + 3)]
        items.append((datetime(*t[:6], tzinfo=timezone.utc) if t else None, title, e.get("link", ""),
                      f"{name} / {publisher}" if publisher else name, "" if publisher else clean(e.get("summary"))))
    return items


def safe_err(e):
    """錯誤訊息遮蔽金鑰"""
    msg = f"{type(e).__name__}: {e}"
    for k in (FINNHUB_API_KEY, FRED_API_KEY):
        if k:
            msg = msg.replace(k, "***")
    return clean(msg, 120)


def attempt(fn, *args, **kwargs):
    """回傳 (結果, 錯誤訊息)：平行抓取時一項失敗不影響其他項"""
    try:
        return fn(*args, **kwargs), None
    except Exception as e:
        return None, safe_err(e)


def gather(tasks, workers, deadline=8):
    """平行執行 tasks = {名稱: (連線池, 無參數函式)}，workers = {連線池: 同時請求數}。
    超過 deadline 秒還沒回來的重送一次（Yahoo 偶爾有請求卡 30 秒才逾時，整格不必跟著等）。
    回傳 {名稱: (結果, 錯誤訊息)}"""
    pools = {p: ThreadPoolExecutor(n) for p, n in workers.items()}
    jobs = {k: pools[p].submit(attempt, fn) for k, (p, fn) in tasks.items()}
    wait(jobs.values(), timeout=deadline)
    for k, f in jobs.items():
        if not f.done():
            p, fn = tasks[k]
            jobs[k] = pools[p].submit(attempt, fn)
    wait(jobs.values(), timeout=deadline)
    for ex in pools.values():
        ex.shutdown(wait=False, cancel_futures=True)     # 卡住的請求留在背景自行逾時
    return {k: f.result() if f.done() else (None, f"逾時（超過 {2 * deadline} 秒）") for k, f in jobs.items()}


FINNHUB_CACHE = {}   # 5 分鐘內重跑同一格直接用上次結果（免費版每分鐘只有 60 次，重跑很容易超過）


def finnhub(path, **params):
    if not FINNHUB_API_KEY:
        raise ValueError("未設定 FINNHUB_API_KEY")
    key = (path, tuple(sorted(params.items())))
    if key in FINNHUB_CACHE and time.time() - FINNHUB_CACHE[key][0] < 300:
        return FINNHUB_CACHE[key][1]
    try:
        for i in range(3):
            # 金鑰放在標頭而不是網址：網址會出現在錯誤訊息裡
            r = HTTP.get(f"https://finnhub.io/api/v1/{path}", params=params,
                         headers={"X-Finnhub-Token": FINNHUB_API_KEY}, timeout=TIMEOUT)
            if r.status_code != 429 or i == 2:
                break
            # 429：第一次多半是每秒上限，等 1.5 秒；再不行就是每分鐘上限，依標頭等到額度重置（最多 5 秒）
            reset = float(r.headers.get("X-Ratelimit-Reset") or 0) - time.time()
            time.sleep(1.5 if i == 0 else min(max(reset + 0.5, 1.5), 5))
        if r.status_code == 429:
            raise RuntimeError("Finnhub 超過免費版每分鐘 60 次上限，約 1 分鐘後重跑本格")
        r.raise_for_status()
        data = r.json()
    except requests.RequestException as e:
        raise RuntimeError(safe_err(e)) from None      # 不論在哪裡被呼叫，錯誤訊息都先遮蔽金鑰
    if isinstance(data, dict) and data.get("error"):
        raise ValueError(data["error"])
    FINNHUB_CACHE[key] = (time.time(), data)
    return data


def company_news(sym, day):
    """Finnhub 個股新聞（day 前後各一天）。第 5、8 格用同一組參數，才能共用 5 分鐘快取"""
    return finnhub("company-news", symbol=sym, **{"from": f"{day - timedelta(days=1):%Y-%m-%d}",
                                                  "to": f"{day + timedelta(days=1):%Y-%m-%d}"})


def fred(path, **params):
    if not FRED_API_KEY:
        raise ValueError("未設定 FRED_API_KEY")
    try:
        r = HTTP.get(f"https://api.stlouisfed.org/fred/{path}", timeout=TIMEOUT,
                     params={**params, "api_key": FRED_API_KEY, "file_type": "json"})
        r.raise_for_status()
    except requests.RequestException as e:
        raise RuntimeError(safe_err(e)) from None      # FRED 的金鑰只能放在網址，錯誤訊息要先遮蔽
    return r.json()

Finnhub： 已設定 ｜FRED： 已設定


2. Prices

In [2]:
try:
    from yfinance.data import YfData    # yfinance 自己的 Yahoo 連線（處理 cookie / crumb）
except ImportError:                     # yfinance 改版找不到時，closes() 全部改用 yf.download
    YfData = None

SPARK = "https://query1.finance.yahoo.com/v7/finance/spark"   # Yahoo 批次端點，一次最多 20 檔
FUTURES = [s for s in ASSETS if s != "^TNX"]   # 幾乎 24 小時交易的黃金、原油、美元指數
FUTURES_CUTOFF = 17                            # 這幾檔的「收盤」＝報告日美東 17:00 前最後一筆價格


def spark_raw(batch, rng, interval):
    """{代號: (時間戳, 收盤序列, meta)}"""
    r = YfData().get(SPARK, params={"symbols": ",".join(batch), "range": rng, "interval": interval}, timeout=TIMEOUT)
    r.raise_for_status()
    out = {}
    for item in r.json()["spark"]["result"] or []:
        resp = (item.get("response") or [{}])[0]
        stamps = resp.get("timestamp") or []
        close = (resp.get("indicators", {}).get("quote") or [{}])[0].get("close") or []
        if stamps and len(stamps) == len(close):
            out[item["symbol"]] = (stamps, close, resp.get("meta") or {})
    return out


def spark(batch):
    """{代號: (近 10 日收盤序列, 第一筆之前的收盤)}，索引是美東日期。
    實測：收盤後幾個小時內，Yahoo 個股 / ETF 的當日日 K 收盤是空值，真正的收盤價放在 meta.regularMarketPrice。
    這裡把它補回去；不補的話當日那筆會被丟掉，整份報告的個股會悄悄變成前一天的數字。"""
    out = {}
    for sym, (stamps, close, meta) in spark_raw(batch, "10d", "1d").items():
        days = pd.to_datetime([datetime.fromtimestamp(x, NY).date() for x in stamps])
        s = pd.Series(close, index=days, dtype=float).dropna()
        s = s[~s.index.duplicated(keep="last")]
        px, t = meta.get("regularMarketPrice"), meta.get("regularMarketTime")
        if px and t:
            day = pd.Timestamp(datetime.fromtimestamp(t, NY).date())
            if s.empty or day > s.index[-1]:
                s[day] = float(px)
        if not s.empty:
            out[sym] = (s, meta.get("chartPreviousClose"))
    return out


def futures_close(symbols, day):
    """{代號: 報告日美東 17:00 前最後一筆價格}（30 分 K，一次請求）。
    期貨的當日日 K 在美股收盤後會變成夜盤即時價，晚一點重跑數字就不一樣；固定截在 17:00，什麼時候跑都相同。
    清晨 5–6 點（美東 16–17 點）執行時，這就是當下的最新價。"""
    end = datetime.combine(day, datetime.min.time(), NY).replace(hour=FUTURES_CUTOFF).timestamp()
    out = {}
    for sym, (stamps, close, _) in spark_raw(symbols, "5d", "30m").items():
        v = [c for t, c in zip(stamps, close) if end - FUTURES_CUTOFF * 3600 <= t < end and c is not None]
        if v:
            out[sym] = float(v[-1])
    return out


def closes(symbols):
    """近 10 個交易日收盤價（列：美東交易日，欄：代號）。
    先用 Yahoo 批次端點（20 檔一組、多組同時抓，500 檔約 2–3 秒）；抓不到的再用 yf.download 補（較慢）。"""
    symbols = list(dict.fromkeys(symbols))
    data = {}
    if YfData is not None:
        batches = [symbols[i:i + 20] for i in range(0, len(symbols), 20)]
        res = gather({i: ("yahoo", partial(spark, b)) for i, b in enumerate(batches)}, {"yahoo": YAHOO_WORKERS})
        for part, _ in res.values():
            data.update(part or {})
    wanted = set(symbols)
    c = pd.DataFrame({s: v[0] for s, v in data.items() if s in wanted})
    missing = [s for s in symbols if s not in c]
    if missing:
        df = yf.download(missing, period="10d", interval="1d", auto_adjust=False, progress=False)
        if not df.empty and "Close" in df:
            extra = df["Close"]
            extra = extra.to_frame(missing[0]) if isinstance(extra, pd.Series) else extra
            c = c.join(extra.dropna(axis=1, how="all"), how="outer") if not c.empty else extra
    if c.empty:
        raise RuntimeError("Yahoo Finance 沒有回傳資料（請檢查網路）")
    c = c.sort_index()
    c.attrs["prev_close"] = {s: v[1] for s, v in data.items()}
    return c


def last_two(c, as_of):
    """{代號: (報告日收盤, 前一日收盤)}。
    沒有報告日當天價格的代號不放進來（會列在「缺資料」）：寧可缺，也不拿前一天的數字頂替。
    Yahoo 只給最新一筆日 K 的代號（S&P 500 類股指數），用該筆之前的收盤當前一日收盤。"""
    prev0 = c.attrs.get("prev_close", {})
    out = {}
    for s, col in c.items():
        v = col.dropna().loc[:as_of]
        if v.empty or v.index[-1] != as_of:
            continue
        if len(v) >= 2:
            out[s] = (float(v.iloc[-1]), float(v.iloc[-2]))
        elif prev0.get(s):
            out[s] = (float(v.iloc[-1]), float(prev0[s]))
    return out


def chg(s, q=None):
    last, prev = (q or Q)[s]
    return last - prev, (last / prev - 1) * 100, last


symbols = [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH, *AI_WATCH] + [x for _, idx, etf in SECTORS.values() for x in (idx, etf) if x]
C = closes(symbols)
days = C[[s for s in INDICES if s in C]].dropna(how="all").index
AS_OF, PREV = days[-1], days[-2]          # 報告日、前一個交易日
fut, fut_err = attempt(futures_close, [s for s in FUTURES if s in C], AS_OF.date())
for s, px in (fut or {}).items():
    C.loc[AS_OF, s] = px
Q = last_two(C, AS_OF)
# 類股漲跌：優先用 S&P 500 類股指數，沒有資料才用 SPDR ETF
SECTOR_PCT, via_etf, no_sector = {}, [], []
for en, (zh, idx, etf) in SECTORS.items():
    s = idx if idx in Q else etf if etf in Q else None
    if s:
        SECTOR_PCT[en] = chg(s)[1]
        if s == etf:
            via_etf.append(f"{zh}（{etf}）")
    else:
        no_sector.append(zh)
RANKED = sorted(SECTOR_PCT, key=SECTOR_PCT.get, reverse=True)
missing = [s for s in [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH] if s not in Q] + no_sector
now = datetime.now(NY)
print(f"資料日期：{AS_OF:%Y-%m-%d}（美東收盤）｜執行時間：台北 {now.astimezone(TW):%m/%d %H:%M}＝美東 {now:%m/%d %H:%M}")
if now.date() == AS_OF.date() and now.hour < 16:
    print("⚠️ 美股還沒收盤（美東 16:00），以下是盤中數字，不是收盤價")
no_cut = [s for s in FUTURES if s in Q and s not in (fut or {})]
if no_cut:
    print(f"⚠️ {'、'.join(ASSETS[s] for s in no_cut)} 抓不到美東 {FUTURES_CUTOFF}:00 的價格"
          f"{f'（{fut_err}）' if fut_err else ''}，改用日線最新價（美東 18:00 後執行會是夜盤價）")
print("類股改用 ETF：" + "、".join(via_etf) if via_etf else "類股全部使用 S&P 500 類股指數")
print("缺資料（沒有報告日當天的價格，稍後重跑本格）：" + "、".join(missing) if missing else "全部抓取成功")

資料日期：2026-11-11（美東收盤）｜執行時間：台北 11/12 10:40＝美東 11/11 21:40
類股全部使用 S&P 500 類股指數
全部抓取成功


3. Report Data Blocks

In [3]:
def updown(d, digits=2):
    return f"{'漲' if d >= 0 else '跌'}{abs(d):.{digits}f}"


out = ["《股市》", ""]
for s, name in {**INDICES, **ADRS}.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
        continue
    d, p, last = chg(s)
    if s == "^VIX":
        out += [f"🔘{name} {updown(d, 1)} ({p:+.2f}%) 收 {last:.2f}", ""]
    elif s in INDICES:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.0f}", ""]
    else:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.2f}", ""]
out += ["（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）", "", "（類股評論：最強類股 → 次強 → 最弱，各一段）", ""]
out += ["🔹強勢類股：", ""] + [x for s in RANKED[:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
out += ["🔹弱勢類股：", ""] + [x for s in RANKED[::-1][:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
focus = sorted((s for s in FOCUS if s in Q), key=lambda s: chg(s)[1], reverse=True)
out += ["🔹焦點股： " + "、".join(f"{FOCUS[s]} {chg(s)[1]:+.2f}%" for s in focus), ""]
out += ["《美債/美元/黃金/原油》", ""]
for s, name in ASSETS.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
    elif s == "^TNX":
        last, prev = Q[s]
        out += [f"🔘{name} {round((last - prev) * 100):+d}bps 收{last:.2f}%", ""]
    else:
        d, p, last = chg(s)
        out += [f"🔘{name} {p:+.2f}% 收 {last:.2f}", ""]
out += [".", ""] + [f"🔹{k}：（待撰寫）\n" for k in ("美債", "美元", "黃金", "原油")]
out += ["《重點訊息》", "", "🔹利多：", "", "1. （待撰寫）", "", "🔹利空：", "", "1. （待撰寫）"]
print("\n".join(out))

《股市》

🔘道瓊 漲254.60 (+0.49%) 收 52486

🔘標普 漲98.45 (+1.31%) 收 7619

🔘納指 漲400.32 (+1.60%) 收 25466

🔘費半 漲565.78 (+5.66%) 收 10564

🔘VIX 跌2.0 (-10.12%) 收 17.49

🔘台積 漲16.87 (+4.25%) 收 413.35

🔘聯電 漲1.04 (+6.91%) 收 16.08

🔘日月光 漲2.34 (+6.60%) 收 37.82

（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）

（類股評論：最強類股 → 次強 → 最弱，各一段）

🔹強勢類股：

資訊科技 +2.71%

能源 +1.83%

工業 +1.32%

🔹弱勢類股：

非必需消費 -0.36%

不動產 +0.03%

醫療保健 +0.06%

🔹焦點股： SuperMicro +9.50%、AMD +7.32%、Tesla +2.91%、NVIDIA +2.87%、Google +2.11%、Apple +2.04%、Goldman Sachs +1.83%、Morgan Stanley +1.63%、JP Morgan +0.39%、Amazon -1.74%、Meta -1.97%、Microsoft -2.26%

《美債/美元/黃金/原油》

🔘美債 +2bps 收5.00%

🔘美元 +0.67% 收 102.91

🔘黃金 -0.89% 收 3994.30

🔘西德州 +0.44% 收 86.42

🔘布蘭特 +1.25% 收 92.63

.

🔹美債：（待撰寫）

🔹美元：（待撰寫）

🔹黃金：（待撰寫）

🔹原油：（待撰寫）

《重點訊息》

🔹利多：

1. （待撰寫）

🔹利空：

1. （待撰寫）


4. Sector detail, top movers, market breadth, tech & AI names outside the S&P 500

In [4]:
def sp500():
    """S&P 500 成分股（維基百科），快取 7 天"""
    if os.path.exists(SP500_CACHE) and time.time() - os.path.getmtime(SP500_CACHE) < 7 * 86400:
        return pd.read_csv(SP500_CACHE)
    r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=TIMEOUT)
    r.raise_for_status()
    t = next(t for t in pd.read_html(io.StringIO(r.text)) if {"Symbol", "GICS Sector"} <= set(t.columns))
    df = t.rename(columns={"Symbol": "symbol", "Security": "name", "GICS Sector": "sector",
                           "GICS Sub-Industry": "industry"})[["symbol", "name", "sector", "industry"]]
    df["symbol"] = df["symbol"].astype(str).str.replace(".", "-", regex=False)   # BRK.B → BRK-B
    df.to_csv(SP500_CACHE, index=False)
    return df


def tag(r):
    return f"{r['name']}（{r['symbol']} {r['chg']:+.2f}%）"


def short_name(name):
    """Yahoo 的公司全名去掉 Inc.、Corporation、Holdings 等字尾"""
    return re.sub(r",?\s+(Inc|Incorporated|Corp|Corporation|Holdings?|Ltd|Limited|plc|N\.V|S\.A|AG|Group|Company|Co)\b\.?.*$",
                  "", name, flags=re.I).strip() or name


def screen(min_cap, min_move, sectors=None):
    """Yahoo 篩選器：美國掛牌、市值 ≥ min_cap 美元、當日漲跌 ≥ min_move% 的個股（不限 S&P 500），依漲跌幅由高到低。
    S&P 500 以外的大漲大跌（如 CoreWeave）靠這裡抓；只留報價日期等於報告日的，避免混進別天的數字"""
    EQ = yf.EquityQuery
    cond = [EQ("eq", ["region", "us"]), EQ("is-in", ["exchange", "NMS", "NYQ", "NGM", "ASE"]),
            EQ("gte", ["intradaymarketcap", min_cap]),
            EQ("or", [EQ("gt", ["percentchange", min_move]), EQ("lt", ["percentchange", -min_move])])]
    if sectors:
        cond.append(EQ("is-in", ["sector", *sectors]))
    quotes = yf.screen(EQ("and", cond), sortField="percentchange", sortAsc=False, size=250)["quotes"]
    rows = [{"symbol": q["symbol"], "name": short_name(q.get("longName") or q.get("shortName") or q["symbol"]),
             "chg": q["regularMarketChangePercent"]} for q in quotes
            if q.get("regularMarketChangePercent") is not None
            and datetime.fromtimestamp(q.get("regularMarketTime") or 0, NY).date() == AS_OF.date()]
    return pd.DataFrame(rows, columns=["symbol", "name", "chg"])


UNI = sp500()
SP500 = set(UNI["symbol"])
QU = last_two(closes(UNI["symbol"]), AS_OF)
UNI["chg"] = [chg(s, QU)[1] if s in QU else None for s in UNI["symbol"]]
n_missing = int(UNI["chg"].isna().sum())
UNI = UNI.dropna(subset=["chg"])

adv, dec = int((UNI["chg"] > 0).sum()), int((UNI["chg"] < 0).sum())
lines = [f"**市場廣度**：上漲 {adv} 家、下跌 {dec} 家（漲跌家數比 {adv / max(dec, 1):.2f}）"]
if all(s in Q for s in BREADTH):
    lines[0] += f"｜等權重 RSP {chg('RSP')[1]:+.2f}% vs 市值加權 SPY {chg('SPY')[1]:+.2f}%"
if n_missing:
    lines[0] += f"｜⚠️ {n_missing} 檔沒有報告日當天的價格，未計入"

# ---- 類股：前三強、後三弱，加上一定要寫的資訊科技 ----
MOVERS = []     # 第 5 格要查評等與新聞的個股
fmt_ind = lambda rows: "、".join(f"{i} {m:+.2f}%（{int(n)}檔）" for i, (m, n) in rows.iterrows()) or "—"
shown = RANKED[:3] + [s for s in ALWAYS if s in SECTOR_PCT and s not in RANKED[:3] + RANKED[-3:]] + RANKED[::-1][:3]
for en in shown:
    sub = UNI[UNI["sector"] == en].sort_values("chg", ascending=False)
    if sub.empty:
        continue
    ind = sub.groupby("industry")["chg"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    n = TECH_MOVERS if en in ALWAYS else MOVERS_PER_SECTOR
    top = sub[sub["chg"] > 0].head(n)
    bottom = sub[sub["chg"] < 0].tail(n).iloc[::-1]
    for _, r in pd.concat([top, bottom]).iterrows():
        if r["symbol"] not in {m["symbol"] for m in MOVERS}:
            MOVERS.append(dict(r, group=SECTORS[en][0]))
    rank = RANKED.index(en) + 1
    lines += ["", f"### {SECTORS[en][0]} {SECTOR_PCT[en]:+.2f}%（11 類股第 {rank} 名）"]
    if en in ALWAYS:        # 資訊科技：子產業全部列出，不只前後三名
        lines.append(f"- 子產業（強 → 弱）：{fmt_ind(ind)}")
    else:
        lines += [f"- 子產業最強：{fmt_ind(ind[ind['mean'] > 0].head(3))}",
                  f"- 子產業最弱：{fmt_ind(ind[ind['mean'] < 0].tail(3).iloc[::-1])}"]
    lines += [f"- 領漲：{'、'.join(tag(r) for _, r in top.iterrows()) or '—'}",
              f"- 領跌：{'、'.join(tag(r) for _, r in bottom.iterrows()) or '—'}"]

# ---- 科技與 AI：觀察名單 + S&P 500 以外的個股（Yahoo 篩選器，兩次請求）----
res = gather({"tech": ("yahoo", partial(screen, *SCREEN_TECH, sectors=["Technology", "Communication Services"])),
              "other": ("yahoo", partial(screen, *SCREEN_OTHER))}, {"yahoo": 2})
empty = pd.DataFrame(columns=["symbol", "name", "chg"])
tech = res["tech"][0] if res["tech"][0] is not None else empty
other = res["other"][0] if res["other"][0] is not None else empty
tech = tech[~tech["symbol"].isin(SP500 | set(ADRS))]             # 成分股在上面的類股裡，台股 ADR 在第 3 格
other = other[~other["symbol"].isin(SP500 | set(ADRS) | set(tech["symbol"]))]
watch = pd.DataFrame([{"symbol": s, "name": n, "chg": chg(s)[1]} for s, n in AI_WATCH.items() if s in Q],
                     columns=["symbol", "name", "chg"]).sort_values("chg", ascending=False)


def movers_line(df, up):
    """上漲（或下跌）的前 SCREEN_MAX 檔；超過時註明總數"""
    d = df[df["chg"] > 0] if up else df[df["chg"] < 0].iloc[::-1]
    more = f"（共 {len(d)} 檔，列前 {SCREEN_MAX}）" if len(d) > SCREEN_MAX else ""
    return ("、".join(tag(r) for _, r in d.head(SCREEN_MAX).iterrows()) or "—") + more


cap = lambda v: f"{v / 1e8:,.0f} 億美元"
lines += ["", "### 科技與 AI：觀察名單與 S&P 500 以外的個股",
          f"- AI 觀察名單：{'、'.join(tag(r) for _, r in watch.iterrows()) or '—'}",
          f"- 非成分股上漲（科技、通訊服務；市值 ≥ {cap(SCREEN_TECH[0])}、漲跌 ≥ {SCREEN_TECH[1]}%）：{movers_line(tech, True)}",
          f"- 非成分股下跌：{movers_line(tech, False)}",
          "", f"### 其他 S&P 500 以外的大型股（不分產業；市值 ≥ {cap(SCREEN_OTHER[0])}、漲跌 ≥ {SCREEN_OTHER[1]}%）",
          f"- 上漲：{movers_line(other, True)}", f"- 下跌：{movers_line(other, False)}"]
no_watch = [n for s, n in AI_WATCH.items() if s not in Q]
if no_watch:
    lines.append(f"\n⚠️ 觀察名單缺資料：{'、'.join(no_watch)}")
for k, label in (("tech", "科技"), ("other", "不分產業")):
    if res[k][1]:
        lines.append(f"\n⚠️ Yahoo 篩選器（{label}）無法取得：{res[k][1]}（稍後重跑本格）")

# 第 5 格多查這幾檔：非成分股與觀察名單裡漲跌最大的（排在最前面）
biggest = lambda df, n: df.reindex(df["chg"].abs().sort_values(ascending=False).index).head(n)
picked = biggest(tech, EXTRA_NEWS)
rest = watch[(watch["chg"].abs() >= WATCH_MOVE) & ~watch["symbol"].isin({m["symbol"] for m in MOVERS} | set(picked["symbol"]))]
MOVERS = ([dict(r, group="科技與 AI（S&P 500 以外＋觀察名單）") for _, r in pd.concat([picked, biggest(rest, EXTRA_NEWS)]).iterrows()]
          + [dict(r, group="其他 S&P 500 以外的大型股") for _, r in biggest(other, EXTRA_NEWS // 2).iterrows()] + MOVERS)
# {代號: (名稱, 當日漲跌 %)}：第 5 格的財報表用（S&P 500 + 非成分股異動 + 觀察名單 + 焦點股）
STOCKS = {r["symbol"]: (r["name"], r["chg"]) for df in (UNI, tech, other, watch) for _, r in df.iterrows()}
STOCKS.update({s: (n, chg(s)[1]) for s, n in FOCUS.items() if s in Q and s not in STOCKS})
display(Markdown("\n".join(lines)))

**市場廣度**：上漲 332 家、下跌 168 家（漲跌家數比 1.98）｜等權重 RSP +0.80% vs 市值加權 SPY +1.32%

### 資訊科技 +2.71%（11 類股第 1 名）
- 子產業（強 → 弱）：Technology Hardware, Storage & Peripherals +7.05%（10檔）、Semiconductor Materials & Equipment +5.61%（5檔）、Electronic Components +5.29%（3檔）、Communications Equipment +5.13%（6檔）、Semiconductors +5.12%（15檔）、Electronic Manufacturing Services +2.72%（3檔）、Technology Distributors +2.20%（1檔）、Electronic Equipment & Instruments +2.05%（4檔）、Systems Software +1.87%（7檔）、Internet Services & Infrastructure +1.62%（3檔）、Application Software +1.61%（15檔）、IT Consulting & Other Services +0.23%（4檔）
- 領漲：Sandisk（SNDK +13.69%）、Micron Technology（MU +11.89%）、Dell Technologies（DELL +11.75%）、Lumentum（LITE +10.67%）、Seagate Technology（STX +9.99%）
- 領跌：Microsoft（MSFT -2.26%）、Accenture（ACN -1.88%）、First Solar（FSLR -1.59%）、Cognizant（CTSH -0.88%）、Amphenol（APH -0.80%）

### 能源 +1.83%（11 類股第 2 名）
- 子產業最強：Oil & Gas Refining & Marketing +3.43%（3檔）、Oil & Gas Equipment & Services +2.97%（3檔）、Oil & Gas Storage & Transportation +2.02%（4檔）
- 子產業最弱：—
- 領漲：Targa Resources（TRGP +3.72%）、Valero Energy（VLO +3.71%）、APA Corporation（APA +3.59%）
- 領跌：—

### 工業 +1.32%（11 類股第 3 名）
- 子產業最強：Heavy Electrical Equipment +4.06%（2檔）、Electrical Components & Equipment +3.95%（6檔）、Construction & Engineering +3.21%（4檔）
- 子產業最弱：Passenger Airlines -0.81%（3檔）、Diversified Support Services -0.65%（3檔）、Industrial Conglomerates -0.44%（3檔）
- 領漲：Bloom Energy（BE +12.97%）、Generac（GNRC +6.72%）、Vertiv（VRT +6.33%）
- 領跌：Leidos（LDOS -2.44%）、United Airlines Holdings（UAL -1.82%）、Xylem Inc.（XYL -1.79%）

### 非必需消費 -0.36%（11 類股第 11 名）
- 子產業最強：Consumer Electronics +1.83%（1檔）、Leisure Products +1.81%（1檔）、Automotive Retail +1.03%（3檔）
- 子產業最弱：Apparel, Accessories & Luxury Goods -1.66%（4檔）、Footwear -1.57%（1檔）、Other Specialty Retail -1.41%（2檔）
- 領漲：Tesla, Inc.（TSLA +2.91%）、Expedia Group（EXPE +2.37%）、Chipotle Mexican Grill（CMG +2.00%）
- 領跌：Airbnb（ABNB -3.09%）、Lululemon Athletica（LULU -2.65%）、McDonald's（MCD -2.61%）

### 不動產 +0.03%（11 類股第 10 名）
- 子產業最強：Other Specialized REITs +1.49%（1檔）、Data Center REITs +0.93%（2檔）、Office REITs +0.35%（2檔）
- 子產業最弱：Multi-Family Residential REITs -0.80%（5檔）、Single-Family Residential REITs -0.78%（1檔）、Hotel & Resort REITs -0.39%（2檔）
- 領漲：Equinix（EQIX +1.72%）、Iron Mountain（IRM +1.49%）、Alexandria Real Estate Equities（ARE +0.85%）
- 領跌：Vivmark Residential（VMRK -1.70%）、Camden Property Trust（CPT -0.80%）、Invitation Homes（INVH -0.78%）

### 醫療保健 +0.06%（11 類股第 9 名）
- 子產業最強：Biotechnology +0.72%（8檔）、Health Care Services +0.57%（5檔）、Health Care Facilities +0.51%（2檔）
- 子產業最弱：Health Care Technology -0.94%（2檔）、Health Care Supplies -0.73%（3檔）、Life Sciences Tools & Services -0.46%（9檔）
- 領漲：Regeneron Pharmaceuticals（REGN +4.73%）、DaVita（DVA +2.35%）、ResMed（RMD +1.89%）
- 領跌：Stryker Corporation（SYK -1.70%）、Solventum（SOLV -1.62%）、GE HealthCare（GEHC -1.26%）

### 科技與 AI：觀察名單與 S&P 500 以外的個股
- AI 觀察名單：CoreWeave（CRWV +24.71%）、Nebius（NBIS +15.44%）、Arm（ARM +12.05%）、Micron（MU +11.89%）、Dell（DELL +11.75%）、Lumentum（LITE +10.67%）、Marvell（MRVL +9.33%）、Oracle（ORCL +8.82%）、Cerebras（CBRS +7.92%）、Intel（INTC +7.52%）、Astera Labs（ALAB +6.81%）、Broadcom（AVGO +3.99%）、Palantir（PLTR +2.12%）、ASML（ASML +1.76%）、SpaceX（SPCX -2.33%）
- 非成分股上漲（科技、通訊服務；市值 ≥ 200 億美元、漲跌 ≥ 4%）：CoreWeave（CRWV +24.71%）、Nebius（NBIS +15.44%）、Arm（ARM +12.05%）、Entegris（ENTG +11.29%）、SK hynix（SKHY +9.71%）、Credo Technology（CRDO +8.94%）、Celestica（CLS +8.50%）、Cerebras Systems（CBRS +7.92%）（共 11 檔，列前 8）
- 非成分股下跌：NetEase（NTES -4.61%）

### 其他 S&P 500 以外的大型股（不分產業；市值 ≥ 100 億美元、漲跌 ≥ 6%）
- 上漲：Lattice Semiconductor（LSCC +10.36%）、Semtech（SMTC +10.34%）、MACOM Technology Solutions（MTSI +10.32%）、Innio（INIO +9.66%）、DigitalOcean（DOCN +9.60%）、Onto Innovation（ONTO +8.89%）、AST SpaceMobile（ASTS +8.70%）、Tower Semiconductor（TSEM +7.64%）（共 11 檔，列前 8）
- 下跌：Summit Therapeutics（SMMT -6.67%）、Revolution Medicines（RVMD -6.40%）

5. Why stocks moved: analyst ratings + headlines + earnings

In [5]:
ACTION = {"up": "升評", "down": "降評", "init": "首評", "main": "維持", "reit": "重申"}
DAY = AS_OF.date()
HOUR = {"bmo": "盤前", "amc": "盤後", "dmh": "盤中"}
COMMON_WORDS = {"first", "united", "american", "general", "national", "international", "new", "applied", "advanced", "texas",
                "southern", "western", "northern", "digital", "global", "live", "public", "universal", "international"}


def analyst(sym, days=3):
    """Yahoo 分析師評等變動（近 N 天）"""
    ud = yf.Ticker(sym).upgrades_downgrades
    if ud is None or ud.empty:
        return []
    idx = pd.to_datetime(ud.index)
    idx = idx.tz_localize(None) if idx.tz is not None else idx
    ud = ud[idx >= pd.Timestamp(AS_OF) - pd.Timedelta(days=days)]
    out = []
    for _, r in ud.iterrows():
        text = f"{r.get('Firm', '')} {ACTION.get(r.get('Action'), r.get('Action', ''))}"
        if r.get("FromGrade") and r.get("FromGrade") != r.get("ToGrade"):
            text += f"：{r['FromGrade']} → {r.get('ToGrade', '')}"
        elif r.get("ToGrade"):
            text += f"：{r['ToGrade']}"
        if pd.notna(r.get("currentPriceTarget")) and r.get("currentPriceTarget"):
            prior = r.get("priorPriceTarget")
            text += f"，目標價 {prior:g} → {r['currentPriceTarget']:g}" if pd.notna(prior) and prior else \
                f"，目標價 {r['currentPriceTarget']:g}"
        out.append(text)
    return out


def stock_news(sym, name, n=3):
    """有 Finnhub 金鑰用 Finnhub（含摘要），否則用 Yahoo。
    Finnhub 會把不相干的文章也掛在個股底下，所以只留標題有提到公司名或代號的；
    其中以公司名開頭、在講異動原因（財報、評等、大漲大跌）的排前面，再依時間新到舊"""
    if FINNHUB_API_KEY:
        parts = re.split(r"[\s,]+", re.sub(r"^The\s+", "", name))
        # 公司名第一個字太常見（First Solar、United Airlines、GE HealthCare）就用前兩個字比對
        two = len(parts) > 1 and (len(parts[0]) <= 3 or parts[0].lower() in COMMON_WORDS)
        word = r"\s+".join(map(re.escape, parts[:2] if two else parts[:1]))
        named = lambda h: 2 if re.match(rf"\W*{word}", h, re.I) else int(bool(
            re.search(rf"\b{word}", h, re.I) or (len(sym) >= 3 and re.search(rf"\b{re.escape(sym)}\b", h))))
        score = lambda d: named(d["headline"]) and named(d["headline"]) + bool(MOVE_RE.search(d["headline"]))
        data = [d for d in company_news(sym, DAY) if d.get("headline") and not blocked(d.get("url"), d.get("source"))
                and not junk(d["headline"], d.get("source")) and score(d)]       # 標題沒提到這家公司的不要
        data.sort(key=lambda d: (score(d), d.get("datetime", 0)), reverse=True)
        data = dedupe(data, title=lambda d: d["headline"])[:n]
        return [f"[{clean(d['headline'], 200)}]({d.get('url', '')}) — {d.get('source', '')}"
                + (f"：{clean(d['summary'], 160)}" if d.get("summary") else "") for d in data]
    out = []
    for it in (yf.Ticker(sym).news or [])[:n]:
        c = it.get("content", it)            # yfinance 新舊兩種格式
        url = (c.get("canonicalUrl") or {}).get("url") or c.get("link", "")
        src = (c.get("provider") or {}).get("displayName") or c.get("publisher", "")
        if c.get("title"):
            out.append(f"[{c['title']}]({url}) — {src}")
    return out


# ---- 財報（Finnhub）：寫「重點訊息」與「盤後」段落用 ----
def beat(actual, est):
    if actual is None or est is None:
        return ""
    return "（優於預期）" if actual > est else "（不如預期）" if actual < est else "（符合預期）"


def earnings_table(data):
    """前一交易日盤後（反應在報告日股價）到報告日的財報。
    範圍：S&P 500 成分股，加上第 4 格的非成分股異動與 AI 觀察名單（CoreWeave 這類非成分股才看得到）"""
    cal = sorted((d for d in data["earningsCalendar"] if d.get("symbol") in STOCKS
                  and (d.get("date") == f"{DAY:%Y-%m-%d}" or d.get("hour") == "amc")),
                 key=lambda d: -(d.get("revenueEstimate") or 0))
    if not cal:
        return "- 這段期間沒有 S&P 500 成分股、非成分股異動或觀察名單個股公布財報"
    rev = lambda v: f"{v / 1e8:,.1f}" if v else "—"
    eps = lambda v: f"{v:.2f}" if v is not None else "—"
    rows = ["| 公司 | 公布時間 | EPS 實際 / 預估 | 營收 實際 / 預估（億美元） | 報告日股價 |", "|:---|:---|---:|---:|---:|"]
    for d in cal[:25]:
        name, px = STOCKS[d["symbol"]]
        mark = "" if d["symbol"] in SP500 else "＊"
        rows.append(f"| {name}（{d['symbol']}）{mark} | {d['date'][5:].replace('-', '/')} {HOUR.get(d.get('hour'), d.get('hour') or '—')} | "
                    f"{eps(d.get('epsActual'))} / {eps(d.get('epsEstimate'))}{beat(d.get('epsActual'), d.get('epsEstimate'))} | "
                    f"{rev(d.get('revenueActual'))} / {rev(d.get('revenueEstimate'))} | {px:+.2f}% |")
    return "\n".join(rows) + (f"\n\n＊＝非 S&P 500 成分股。{PREV:%m/%d} 盤後公布的財報反應在報告日股價；"
                              f"{DAY:%m/%d} 盤後公布的，報告日股價尚未反映，實際數字也可能稍晚才更新。")


# 評等（Yahoo）、新聞與財報（Finnhub）分兩組同時抓。
# 新聞有上限：科技（非成分股、觀察名單、資訊科技）先抓，其餘依漲跌幅大小，超過 STOCK_NEWS_MAX 的只查評等
tech_first = [m for m in MOVERS if "科技" in m["group"]]
rest = sorted((m for m in MOVERS if "科技" not in m["group"]), key=lambda m: -abs(m["chg"]))
with_news = {m["symbol"] for m in (tech_first + rest)[:STOCK_NEWS_MAX]}
tasks = {"cal": ("finnhub", partial(finnhub, "calendar/earnings", **{"from": f"{PREV:%Y-%m-%d}", "to": f"{DAY:%Y-%m-%d}"}))}
for r in MOVERS:
    tasks["analyst", r["symbol"]] = ("yahoo", partial(analyst, r["symbol"]))
    if r["symbol"] in with_news:
        tasks["news", r["symbol"]] = ("finnhub", partial(stock_news, r["symbol"], r["name"]))
res = gather(tasks, {"yahoo": YAHOO_WORKERS,
                     "finnhub": FINNHUB_WORKERS if FINNHUB_API_KEY else 2})   # 沒金鑰時新聞也走 Yahoo，壓低同時請求數

lines, analyst_errs, group = [], [], None
for r in MOVERS:
    if r["group"] != group:
        group = r["group"]
        lines += [f"### {group}", ""]
    (ratings, ratings_err), (news, news_err) = res["analyst", r["symbol"]], res.get(("news", r["symbol"]), (None, None))
    lines.append(f"**{tag(r)}**")
    lines += [f"- 📊 {a}" for a in ratings or []]
    lines += [f"- 📰 {n}" for n in news or []]
    if news_err:
        lines.append(f"- 📰 （新聞無法取得：{news_err}）")
    if ratings_err:
        analyst_errs.append(ratings_err)
    if not ratings and not news and not news_err:
        lines.append("- （近期無評等變動或新聞）" if r["symbol"] in with_news else "- （近期無評等變動；新聞未抓）")
    lines.append("")
if analyst_errs:
    lines.append(f"⚠️ 分析師評等 {len(analyst_errs)}/{len(MOVERS)} 檔無法取得（Yahoo 限流時常見，稍後重跑本格即可）：{analyst_errs[0]}")
if len(MOVERS) > len(with_news):
    lines.append(f"ℹ️ {len(MOVERS) - len(with_news)} 檔漲跌幅較小的個股沒有抓新聞（每次最多 {STOCK_NEWS_MAX} 檔，Finnhub 免費額度）")
display(Markdown("\n".join(lines)))

cal, err = res["cal"]
body, err = (None, err) if err else attempt(earnings_table, cal)
display(Markdown(f"**財報｜{PREV:%m/%d} 盤後 ～ {DAY:%m/%d}（依營收規模排序）**\n\n{body or f'- 無法取得：{err}'}"))

### 科技與 AI（S&P 500 以外＋觀察名單）

**CoreWeave（CRWV +24.71%）**
- 📊 Citigroup 維持：Buy，目標價 128 → 149
- 📊 DA Davidson 維持：Neutral，目標價 100 → 45
- 📊 Evercore ISI Group 維持：Outperform，目標價 150 → 100
- 📊 Truist Securities 維持：Buy，目標價 165 → 115
- 📊 Needham 重申：Hold
- 📊 Stifel 維持：Hold，目標價 120 → 60
- 📊 Cantor Fitzgerald 維持：Overweight，目標價 176 → 100
- 📊 Rosenblatt 維持：Buy，目標價 250 → 160
- 📊 Wells Fargo 維持：Overweight，目標價 160 → 115
- 📰 [CoreWeave Reportedly Plans $4B Investment-Grade Loan After Strong Earnings Beat — But Shares Fall After-Hours On Nvidia CEO Drama](https://finnhub.io/api/news?id=431179156ef133e7a3180778cae8eb5f5deeb163a14ddd5883d2fb65f01b5256) — Yahoo：The AI cloud provider is preparing to raise the funds through an investment-grade loan, with Morgan Stanley and Mitsubishi UFJ Financial Group leading the poten…
- 📰 [CoreWeave (CRWV) Stock Rebounds As Q3 Results Ignite Margin Debate](https://finnhub.io/api/news?id=fa5881520482c5d6a8f42d9777afa50276328d57ba3d12d683846e430cf5ac30) — Yahoo：CoreWeave just dropped Q3 2026 numbers that lit a fire under the stock. Shares ripped 24.9% today to US$60.84, a sharp rebound after a weak 3 month stretch, as…
- 📰 [CoreWeave, Inc. 2026 Q3 - Results - Earnings Call Presentation](https://finnhub.io/api/news?id=50507aa37840d0b40c42dae738a761403f799ba08b3cbcff1c028fa6f9b23b56) — SeekingAlpha：2026-11-11. The following slide deck was published by CoreWeave, Inc.

**Nebius（NBIS +15.44%）**
- 📰 [Nebius Stock Jumps Ahead of Earnings. CoreWeave Gives It a Lift.](https://finnhub.io/api/news?id=36da3f5b96f08346a9f3e536944a67081fcb6f61480dcd90081eab99bb7daf3c) — Yahoo：CoreWeave’s financing outlook lifts neocloud stocks as Nebius prepares to report quarterly results Thursday.
- 📰 [Nebius Group Stocks Jump Nearly 17% Ahead of Thursday Results as CoreWeave Lifts AI Clouds](https://finnhub.io/api/news?id=2e4e804890854a3a16fc19c8888c79fc7f5d7422b27cc38fdd7cda7e15e0e326) — Yahoo：Nebius reports third-quarter results before Thursday's open, after CoreWeave's beat reset expectations for rented AI capacity.
- 📰 [Nebius earnings preview: what to watch when Q3 results hit Thursday](https://finnhub.io/api/news?id=e1d4fbb960ec354a3c2c45cef222754433f214e2034f8b374e79a096073f4f28) — Yahoo：Investing.com -- Nebius Group NV (NBIS) reports Q3 before the market opens Thursday, Nov 12, 2026. The stock is trading at $159.10 (+18.73%) as of Nov 11, 12:34…

**Arm（ARM +12.05%）**
- 📰 [Arm Sinks 10% in a Week on Phone Royalties and SoftBank: “Circular Financing Concerns”](https://finnhub.io/api/news?id=fba62278dd718ee39cd5c935ed1c334c27aceca0dcb088f28a17adde806cc01e) — Yahoo：Key TakeawaysArm stock fell 10% in a week in a chip selloff that hit Nvidia and Micron even harder. Pricier memory stalled Arm’s phone royalties, but its data c…
- 📰 [ARM Just Dropped 13% After Earnings. Is It Time to Sell, or Load Up?](https://finnhub.io/api/news?id=55f925ef194c4bffa5adc68e54d556ff2c30b95dfb130607706966cdb15f1a2e) — Yahoo：Key TakeawaysArm shares fell about 13% on November 5 even though revenue rose 27% to a record $1. 44 billion and adjusted EPS beat estimates.
- 📰 [Server CPU strength persists as AMD and Arm gain share: UBS](https://finnhub.io/api/news?id=af22fd5d29e97ae5cdf6aaf44bffddae022be152043ca78efe1f6a3acdec625e) — Yahoo：Investing.com -- UBS told clients in a note on Wednesday that demand for server processors stayed strong in the third quarter, with chips based on Arm (NASDAQ:…

**Entegris（ENTG +11.29%）**
- 📰 [Allegro MicroSystems, IPG Photonics, Lattice Semiconductor, Entegris, and Broadcom shares skyrocket, what you need to know](https://finnhub.io/api/news?id=390b8d86aca11af899d7b2c0c0da47e39e66fc8476e2b1af4d9cc64b12ccae87) — Yahoo：A number of stocks jumped in the morning session after dip-buyers moved into beaten-down semiconductor and hardware names following the sector's steepest one-da…
- 📰 [Entegris, Inc. (ENTG) Analyst/Investor Day Transcript](https://finnhub.io/api/news?id=9c2f7af8862882568d14df8f17e7bbbdeae0f599bbc96d65e3c039390bfb1452) — SeekingAlpha：Entegris, Inc. (ENTG) Analyst/Investor Day November 9, 2026 1:00 PM ESTCompany ParticipantsJeffrey Schnell - Vice President of Investor RelationsDavid...

**SK hynix（SKHY +9.71%）**
- 📰 [SK hynix Stocks Jump 6.69% as Seoul Chip Rally Lifts KOSPI 4.39%](https://finnhub.io/api/news?id=539ef00e8a8246b820bf6cb2de8e83d7573a8847931c82f5359fdb5cb828ade7) — Yahoo：The memory maker closed at 1,212,000 won as foreign and institutional buyers returned after a three-session KOSPI slide.
- 📰 [SK hynix (NasdaqGS:SKHY) Stock Looks Cheap Following A 20% Slide](https://finnhub.io/api/news?id=c10001d252c30fe885c38ca07ec25fc5331ff8c0e8f21ec8caa5a256e7c224f0) — Yahoo：SK hynix has slipped sharply over the past month, even as headlines focus on record earnings and large chip investment plans. This puts a fresh spotlight on whe…
- 📰 [SK hynix Rides the AI Data Center Boom and HBM4 Ramp-Up](https://finnhub.io/api/news?id=aee23cb228ffdf5605f349bfb1503e13ed662cc6a69ccc44acb1cdc76b5bbf23) — Yahoo：SK hynix is poised to gain from booming AI data center investments and rising memory demand as its HBM4 ramp-up strengthens growth prospects.

**Credo Technology（CRDO +8.94%）**
- 📰 [Credo Technology Stocks Climb 10.8% as AI Connectivity Names Rally](https://finnhub.io/api/news?id=6f34fd909ff0e4cd8d4b414888fd68c196470b8d2d9761ef787b20b2a963b054) — Yahoo：Credo guided October-quarter revenue to $525 million to $535 million after a 115% jump in its previous quarter.
- 📰 [Credo Schedules Second Quarter Fiscal Year 2027 Financial Results Conference Call](https://finnhub.io/api/news?id=31a4a84bc4b08200782555881c6bd99243b308b7897d2e5e519bea0f81f2753e) — Yahoo：SAN JOSE, Calif., November 11, 2026--Credo Technology Group Holding Ltd (Credo) (NASDAQ: CRDO), an innovator in providing connectivity at scale through fast, re…
- 📰 [Credo Technology Stock Is Surging Wednesday: What's Going On?](https://finnhub.io/api/news?id=2c697c9a5a2f7e9414cf0872b4368d2cb5c45c5fbf80513e29d17bfc29bd6142) — Benzinga：Credo Technology (CRDO) jumped nearly 9% premarket as AI infrastructure stocks rebounded. Explore growth catalysts and technical levels.

**Marvell（MRVL +9.33%）**
- 📊 B. Riley Securities 維持：Buy，目標價 275 → 250
- 📰 [Marvell Acquires Lumentum in Blockbuster $80 Billion Deal](https://finnhub.io/api/news?id=d4bd92908d146b4356e44b56a87682db36c2216bbc72c1a5c246cb4cb799f9a3) — Yahoo：There has been no shortage of news these last few weeks coming out of the memory and optics sectors in regards to AI investing. Eric Bleeker and Austin Smith ar…
- 📰 [Marvell and Intel Rally 7% as CoreWeave’s Results Reignite the AI Chip Trade; Taiwan Semiconductor Climbs 4%](https://finnhub.io/api/news?id=6931756178a346a9167dae39aad5a384d1597e5579ab7f009de5a5be50a685fb) — Yahoo：After a brutal month that wiped roughly 40% off some of the biggest names in AI silicon, one cloud provider's earnings and a quiet funding shift just sparked a…
- 📰 [Marvell (MRVL) Sees $222.6 Million in Net ETF Buying on Monday](https://finnhub.io/api/news?id=cb9901028e7ef5d5d4f2fea2d3312199c6d0a68ba221a8b0066c31bfead60ddb) — Yahoo：ETFs bought $222.6 million of Marvell (MRVL) shares on Monday, Nov. 9.

**Oracle（ORCL +8.82%）**
- 📰 [Oracle Stock Just Surged After CoreWeave Earnings. Here's Why One Analyst Says It's Built for the Data-Center Backlash](https://finnhub.io/api/news?id=271ebc43316bb908bbcd7656435a5cc9e5ca98fbc76fad5c1f195edbbc487b7d) — Yahoo：Oracle shares rallied with other AI stocks on Wednesday. One analyst thinks Oracle is better positioned than rivals as political pushback against data centers g…
- 📰 [Oracle Stocks Climb 7.2% as CoreWeave's Funding Reset Eases AI Debt Fears](https://finnhub.io/api/news?id=aa20ed272aa499a8d471a7ed6cba303a0f2432d814a0c334eb4d3409576a52f7) — Yahoo：Oracle now trades above the $116 price of its $10 billion share sale, while new notes carry coupons up to 8.65%.
- 📰 [Oracle CFO Promises “$30 Billion Plus” in Free Cash Flow by Fiscal 2029 After a 70% Plunge](https://finnhub.io/api/news?id=5ad62a653c1d985541e9fb6b541edaee8fb2f47d66ed118b2b39b4f7b47c2643) — Yahoo：Key TakeawaysOracle CFO Hilary Maxson told CNBC that Oracle will be cash-flow positive in the second half of fiscal 2028 and produce more than $30 billion of fr…

**Cerebras（CBRS +7.92%）**
- 📊 Morgan Stanley 維持：Overweight，目標價 279 → 178
- 📊 Wedbush 維持：Outperform，目標價 290 → 190
- 📰 [Cerebras Systems Q3 2026 earnings preview: key metrics and risks to watch](https://finnhub.io/api/news?id=c98ae21fbee300ad20a0ee8fb8f5e218dae2a01c22c4b9ed104ac2b6eb1e5317) — Yahoo：Investing.com -- Cerebras Systems Inc (CBRS) is scheduled to report fiscal Q3 2026 results on Nov 12, 2026, after the close (official date). The stock is a long…
- 📰 [Cerebras Q3 Earnings Preview: What To Expect From Upcoming Report](https://finnhub.io/api/news?id=bbb01a21788b0a71bdaca6bd776a502da23c4fa5766bb20aec6c78ed7531389b) — Yahoo：Cerebras Earnings on Tap as AI Chip Demand Faces Its Biggest Test Yet
- 📰 [Cathie Wood, Brad Gerstner and 8 Other Big Funds Bought Cerebras. Is It a Buy Before Earnings?](https://finnhub.io/api/news?id=68f60a3ede1ca172a9ff4199fcbc654822ad3aeef0402c46f30c85662e8fe5ed) — Yahoo：Key TakeawaysCathie Wood’s ARK Invest, Brad Gerstner’s Altimeter Capital and eight other big funds reported new Cerebras stakes as of June 30, and the stock is…

**Intel（INTC +7.52%）**
- 📰 [Intel Stock Faces 87 Million-Share Risk After SoftBank Deal](https://finnhub.io/api/news?id=9b792649ad8d56ef66b03b6d48fc0a017e8b5b4b5b14c1d88b5508df8061dd23) — Yahoo：SoftBank lost about $7 billion in its latest quarter as its Intel stake took losses on both the shares and the hedge contracts backing them.
- 📰 [Intel Stocks Climb 5.23% as SoftBank Reveals $7.4 Billion Raised Against Intel Shares](https://finnhub.io/api/news?id=1ce94ac6cec927d13ec0a2ce581fdbdebbb6dba4c6fd0eaecfb43244738aea71) — Yahoo：SoftBank's Intel stake jumped to $10.5 billion and was used to raise cash through prepaid forward contracts.
- 📰 [Intel: Why I'm Turning Bullish After The Crash](https://finnhub.io/api/news?id=67cf5ede92697638345e681255195c2a14064fa57589197d2b72c812efc82ff8) — SeekingAlpha：Intel shares plunged nearly 39% following disappointing Q4 guidance, creating a speculative recovery opportunity. Check out why INTC stock is a buy.

**Astera Labs（ALAB +6.81%）**
- 📰 [Astera Labs Announces Upcoming Financial Conference Participation](https://finnhub.io/api/news?id=3e36c3c2753224a98ce83430300c1447d06341397380dbeb13b972068f90fd1b) — Yahoo：SAN JOSE, Calif., Nov. 10, 2026 (GLOBE NEWSWIRE) -- Astera Labs, Inc. (Nasdaq: ALAB), a leader in semiconductor-based connectivity solutions for rack-scale AI i…
- 📰 [ETFs Are Net Sellers of Astera Labs (ALAB) on Nov. 6](https://finnhub.io/api/news?id=5601bfd4d5e41085856eb1b3f330a800d2a450285838e89e943cd02f61ac5aac) — Yahoo：ETFs sold a net $80.2 million of Astera Labs in Friday's session.

**Broadcom（AVGO +3.99%）**
- 📰 [Broadcom Stocks Climb 5.18% as Goldman Calls Chips Oversold and Anthropic Stake Lands](https://finnhub.io/api/news?id=22db75f1d54fb564eb26cb36504f4bf02ef537275ef2d772912ab036426fbab9) — Yahoo：Goldman Sachs says Broadcom trades at 13 times a no-growth 2028 baseline, while Broadcom commits $1 billion to Anthropic's IPO.
- 📰 [Broadcom Stocks Rise 2.69% as It Joins $26 Billion Anthropic Placement](https://finnhub.io/api/news?id=339d202067180d2bf69fe8a153cff2cc96fd6f0c06c9d31d12c9f921adc97308) — Yahoo：The custom-chip leader adds to Monday's 8.78% jump after Anthropic named it among seven suppliers buying shares at the IPO price.
- 📰 [Nvidia Stock Sinks as Jensen Huang Ouster Report Sends Investors Toward AMD, Broadcom](https://finnhub.io/api/news?id=407992f75d0a56c8bd01297bee7864730d3600f1425ea884dc0d36f12a2bf59c) — Yahoo：Broadcom Inc. (NASDAQ:AVGO) and Advanced Micro Devices Inc. (NASDAQ:AMD) shares each rose more than 3% in extended trading Wednesday as fresh turmoil surroundin…

### 其他 S&P 500 以外的大型股

**Lattice Semiconductor（LSCC +10.36%）**
- 📰 [Lattice and NXP Deliver Flexible, Secure Computing Solution for the Industrial Edge](https://finnhub.io/api/news?id=81968757beede03a1a1f4a8f97fbad94fb20264645dbb7e25bd693c26f4082e5) — Yahoo：HILLSBORO, Ore., November 10, 2026--Lattice Semiconductor (NASDAQ: LSCC), the low power programmable and platform firmware leader, today announced the Lattice N…
- 📰 [Allegro MicroSystems, IPG Photonics, Lattice Semiconductor, Entegris, and Broadcom shares skyrocket, what you need to know](https://finnhub.io/api/news?id=390b8d86aca11af899d7b2c0c0da47e39e66fc8476e2b1af4d9cc64b12ccae87) — Yahoo：A number of stocks jumped in the morning session after dip-buyers moved into beaten-down semiconductor and hardware names following the sector's steepest one-da…

**Semtech（SMTC +10.34%）**
- 📊 B. Riley Securities 維持：Buy，目標價 210 → 190
- 📰 [Will Semtech's SGP.32 eSIM Launch Expand Its Cellular IoT Business?](https://finnhub.io/api/news?id=742044f777f574cc62333b5bfa2cd27b33e806b996e76c863fc66ebfa0e240aa) — Yahoo：SMTC's new SGP.32 eSIM lets businesses manage multiple carrier profiles remotely, simplifying global IoT connectivity and reducing reliance on single networks.

**MACOM Technology Solutions（MTSI +10.32%）**
- （近期無評等變動或新聞）

### 資訊科技

**Sandisk（SNDK +13.69%）**
- 📰 [Sandisk Stocks Jump 16% as Storage Rally Returns Ahead of 8-for-1 Split](https://finnhub.io/api/news?id=dba3f029fd38261593a66e7302a85e5d3464a2e3cf967bbe64051eb2798b9f79) — Yahoo：Sandisk rebounds after a roughly 41% monthly drop, backed by guidance of up to $12.6 billion in quarterly revenue.
- 📰 [Sandisk Leads Memory Stock Rally. The AI Recovery Has Far to Go.](https://finnhub.io/api/news?id=ec1dacc8d5881f4038d90a2d78938793e6306154f9fa0a40d5d9198bd89f4165) — Yahoo：The memory-card supplier surges as part of a broad AI revival, bolstering hopes it could be about to end a dismal recent run.
- 📰 [Sandisk Sinks 24% in a Week on “Industry-Wide Memory Dynamics”: Time to Sell, or Load Up?](https://finnhub.io/api/news?id=9fc3ec7badbafa8dc7155c94503ecd308285ba50c4a0e50521d49daa8e97ab24) — Yahoo：Key TakeawaysSandisk fell 24% in a week after the midterms and a Qualcomm warning on memory costs, and Micron and SK hynix fell too. Sandisk’s own quarter held…

**Micron Technology（MU +11.89%）**
- 📰 [Micron, Sandisk Soar as CoreWeave CEO Says AI Demand Outpaces World's Capacity](https://finnhub.io/api/news?id=0ba9109ad174864a24b6a9e2eb5ba9e86ec2f39134b3ce93f7525fc6dc2dd04f) — Yahoo：Memory and storage stocks surged Wednesday after CoreWeave (NASDAQ:CRWV) reported blockbuster earnings, with CEO Michael Intrator dismissing fears that the arti…
- 📰 [Micron Stocks Jump 11.2% as Memory Rebound Meets $1 Billion Anthropic Placement](https://finnhub.io/api/news?id=f0988e3842df38ab5b1601c349f75df20f53dcebbaeffec9fc24079c113f43ca) — Yahoo：Micron joins Anthropic's $26 billion private placement as memory names recover from a steep one-month slide.
- 📰 [Micron Stock Surges After CoreWeave's AI Earnings Surprise](https://finnhub.io/api/news?id=91fd804092f8833f316cb837576f85346ddc626e2b7fedf0108dfdea926d102e) — Yahoo：Wall Street's biggest technology concern faces an unexpected reality check.

**Dell Technologies（DELL +11.75%）**
- 📰 [Dell Stocks Jump 11.42% as CoreWeave's Bigger Spending Plan Revives AI Server Bets](https://finnhub.io/api/news?id=2db68ca97702926477d0dad0b672bd63502e5490fe862991186e80448529a23e) — Yahoo：CoreWeave lifted the low end of its 2026 capex range to $37 billion, a read-through for Dell's record $95 billion AI backlog.
- 📰 [Dell Jumps 11% as AI Server Stocks Rebound From a Rough Month; Super Micro Surges 8%, Hewlett Packard Enterprise Advances 6%](https://finnhub.io/api/news?id=92107fc66e5f294958cdc77a03846c3d5ff4a6898bf25d45861768edb0567609) — Yahoo：AI server makers are clawing back ground after a hard month, and Dell Technologies (NYSE:DELL) is leading the rebound. Dell stock is at $370.25, up 11% in early…
- 📰 [Dell Sinks 12% in a Week on Texas Data Center Fears: “A Moratorium, a Ban”](https://finnhub.io/api/news?id=e4516d6d97c84f2acf787d63f00b52149497bacfdef4fd3cd885e9cd47edf894) — Yahoo：Key TakeawaysDell fell 12% in the week to Nov. 9 as AI hardware stocks, Nvidia and Super Micro among them, sold off after the Nov.

**Lumentum（LITE +10.67%）**
- 📊 UBS 維持：Neutral，目標價 960 → 686
- 📊 Bernstein 維持：Outperform，目標價 635 → 700
- 📰 [Lumentum Stocks Jump 14.85% as AI Optics Rebound Narrows Marvell Deal Gap](https://finnhub.io/api/news?id=ae8d076253ad558a2e210f2af1361c1a1b69bf3314ad2903172a8ecac7308fd8) — Yahoo：Each Lumentum share converts into 4.5 Marvell shares, and Wednesday's rally reshapes the arbitrage spread.
- 📰 [Lumentum Slips on Deal Reaction; Fossil Group Rallies on Outlook | Stock Movers](https://finnhub.io/api/news?id=2dbcdb3767be8c0926c2211474a712defa73f2a04c2dae976c8e552fe2e979d7) — Yahoo：On this episode of Stock Movers: - Shares of CoreWeave (CRWV) rallied in premarket trading after the compay eased fears about its debt-laden business during a h…
- 📰 [Lumentum Sinks 11% in a Week as Marvell’s All-Stock Offer Shrinks: “Our Demand Couldn’t Be Stronger”](https://finnhub.io/api/news?id=bacd426c428d0421b82dad2f1637561f04161e7320d7fe0927cba4f01534015c) — Yahoo：Key TakeawaysLumentum fell 11% in the past week as the AI hardware selloff hit Marvell, whose all-stock offer drives its price, and it lagged Coherent’s rebound…

**Seagate Technology（STX +9.99%）**
- 📰 [Seagate Technology Holdings (STX) Gains Fresh Backing As AI Storage Demand Tests Its Valuation](https://finnhub.io/api/news?id=91db80d01d27dbd5275f5be3578ada5464da544a2d93b5c3c05c484d35333ea8) — Yahoo：Seagate Technology Holdings (STX) is back in focus after a sharp move tied to stronger fiscal guidance, fresh AI driven storage demand, and the launch of its Al…
- 📰 [Seagate Stocks Jump 11.36% as CoreWeave Results Revive Nearline Storage Demand](https://finnhub.io/api/news?id=9498276adfc2ea984f9ffdd4b93081e085040d499c309c430e776f0596148b70) — Yahoo：Seagate guided to $4.8 billion in quarterly revenue, with long-term deals allocating most nearline supply through 2028.
- 📰 [Seagate Soars 10% as CoreWeave’s “Breakout Quarter” Eases AI Spending Fears](https://finnhub.io/api/news?id=51af9102aa2b3398a1204474a879e29b0a6b811248d2109142f2403af4a66656) — Yahoo：Key TakeawaysSeagate shares rose 10% today after CoreWeave beat estimates and raised its 2026 guidance. The selloff in Seagate and Western Digital went well bey…

**Microsoft（MSFT -2.26%）**
- 📰 [Microsoft stock sinks as new AI rival targets $27 billion business](https://finnhub.io/api/news?id=080915c6efb08161eb276baa522c981637f16ad5b48e26c5141364925f6b11b9) — Yahoo：The development adds to concerns about how AI software could reshape established productivity tools.
- 📰 [Microsoft Stock Falls as Anthropic Takes Aim at Office Work With New Claude Tools](https://finnhub.io/api/news?id=d727b1fbe84197cd271f4c94e6ef167b42c9a32ccd8bbc4f0abd88e2f198e75e) — Yahoo：Anthropic Builds Out Claude for the Office and Microsoft Gets a Warning Shot
- 📰 [Microsoft Corporation (MSFT) Surged 38% as Azure Accelerated, but One Investor Remains Skeptical: Here’s Why](https://finnhub.io/api/news?id=2966112214f479526e44525c094fead46a2a60b00c56c4bf245efd18b8f58eb8) — Yahoo：Liontrust Investment Partners recently released its “GF Global Dividend Fund” Q3 2026 investor letter. A copy of the letter can be downloaded here. The Fund ret…

**Accenture（ACN -1.88%）**
- 📰 [Accenture vs. Adobe: What Does Fluctuating vs. Steady Revenue Tell Investors?](https://finnhub.io/api/news?id=1a725dcb9014c0cdc3e3f539992545ee914c20cd51f36f5f3b292804e267715c) — Yahoo：Adobe has posted eight straight quarters of growth, while Accenture's revenue swings mask a deeper story about business stability.
- 📰 [Accenture vs. Intuit: Does Corporate Consulting or Financial Software Make For a Better Stock Buy in 2026?](https://finnhub.io/api/news?id=8863df3b5a2473e5bd6b0184b49d4ed6c17bdcc83505c2b608646b3a1dbd9b51) — Yahoo：Accenture's 11.3% net margins and $11.6B free cash flow contrast sharply with Intuit's 21.3% margins but higher valuation multiples, a key tension for value-con…
- 📰 [Life Science Analytics - Global Strategic Business Report 2026: Capitalize on a US$6.6 Billion Revenue Surge to 2032 as IBM, Oracle, Accenture, SAS, and Cognizant Disrupt Data-Driven Healthcare](https://finnhub.io/api/news?id=a1e0d86b51b97692adfef08cc5daf44a01bca453f0fbf8b828649ca97271f572) — Yahoo：Boost ROI with AI, machine learning and cloud analytics to accelerate drug development, personalize care, optimize trials and uncover new revenue opportunities.…

**First Solar（FSLR -1.59%）**
- 📊 UBS 維持：Buy，目標價 330 → 295
- 📊 Piper Sandler 維持：Overweight，目標價 251 → 222
- 📰 [First Solar Stock: Is Wall Street Bullish or Bearish?](https://finnhub.io/api/news?id=a8df67bdf9de8bbed65ba3fd8b11b4c1983e4efd2b8bccf3792434ce2a635d7e) — Yahoo：Despite underperforming the broader market over the past year, First Solar continues to receive a cautiously optimistic outlook from Wall Street analysts.
- 📰 [UBS Maintains Buy on First Solar, Lowers Price Target to $295](https://finnhub.io/api/news?id=033b9a7d6cf3b0d11b24dc5bc35d7494b97d9824c12e0b0bc727f3c19ad6b2ef) — Benzinga：UBS analyst Jon Windham maintains First Solar (NASDAQ:FSLR) with a Buy and lowers the price target from $330 to $295.
- 📰 [Stock of the Day: Will First Solar Rally From Here?](https://finnhub.io/api/news?id=cf45d98bab66efc467f4b940c6d5e2917beb28daa675494052f250a7f23d80f5) — Benzinga：First Solar falls to key $188 support, where remorseful sellers may buy back, potentially sparking a reversal and move higher.

**Cognizant（CTSH -0.88%）**
- 📰 [Cognizant Technology Solutions (CTSH) Could Be 7% Undervalued On Claude AI Expansion](https://finnhub.io/api/news?id=21ba8c2f8d4881d2ffb0c5cf5de922226d91933727b4f72448c07acec930c595) — Yahoo：Cognizant Technology Solutions (CTSH) has moved to deepen its push into enterprise AI by dedicating more than 25,000 professionals to a new Claude Frontier Capa…
- 📰 [Cognizant Establishes 25,000-Person Claude AI Group and Introduces Enterprise Managed Services](https://finnhub.io/api/news?id=e4e2bd096d2a6642ef30e84c9044d1e3f4f014c52a05c1cecc9dc42e5797da70) — Yahoo：Cognizant (NASDAQ:CTSH) announced the creation of its Claude Frontier Capability Group, a dedicated organisation comprising more than 25,000 professionals focus…
- 📰 [Cognizant Announces Formation Of Claude Frontier Capability Group, Bringing More Than 25,000 Professionals For Global Claude Delivery](https://finnhub.io/api/news?id=fc5905eb87aab97de5f2d5747536ecd2cb41f25d033681ae570f1eacb529d100) — Benzinga：Cognizant's new Claude Frontier Capability Group brings more than 25,000 professionals together in a dedicated global Claude delivery organization.Cognizant's n…

**Amphenol（APH -0.80%）**
- 📰 [Amphenol (APH) Delivers Record Sales, Plans to Expand Through $1.6 Billion in Deals](https://finnhub.io/api/news?id=9ba4040eb2af69a4b4b67bd97e2f9f56e4332b6721cfe7c43378e308ffe96077) — Yahoo：We just published our list of the 10 Best US Stocks to Invest In for Long Term and Amphenol Corporation (NYSE:APH) ranked 9th on this list. Recent economic and…
- 📰 [Amphenol Stock: Analyst Estimates & Ratings](https://finnhub.io/api/news?id=34966aa5585dbd1ea9eefd889fac0b02c0c65e075f81a949dcce66e415bf4e63) — Yahoo：As Amphenol has outperformed the SPX over the past year, Wall Street analysts maintain a bullish outlook on the stock’s prospects.
- 📰 [Vertiv's Data Center Growth Fuels Revenue: Can It Beat APH and SMCI?](https://finnhub.io/api/news?id=8c125132ce9eacc8663e589288578ffae548f65a00db4cf59fb2c793cb84c0f0) — Yahoo：VRT's AI-driven data center demand fuels 61% revenue growth in the third quarter of 2026, strengthening its position against Amphenol and Super Micro Computer.

### 能源

**Targa Resources（TRGP +3.72%）**
- 📊 Morgan Stanley 維持：Overweight，目標價 359 → 362
- 📊 Scotiabank 維持：Sector Outperform，目標價 319 → 325
- 📊 Wells Fargo 維持：Overweight，目標價 324 → 330
- 📰 [TRGP Q3 Earnings Beat on Permian Growth and Higher Fees](https://finnhub.io/api/news?id=b3c098f665be93468a3d60ffd9478748bf2b4db862566c78416fefb033ab0fb5) — Yahoo：Targa Resources' Q3 2026 earnings beat estimates as record Permian volumes and higher fees fueled growth, prompting a raised full-year EBITDA outlook.
- 📰 [Morgan Stanley Maintains Overweight on Targa Resources, Raises Price Target to $362](https://finnhub.io/api/news?id=0d218d8841ac28ebd64e2f042dd59ad2105cfff71b9c0d87c2a9196ea16ba06b) — Benzinga：Morgan Stanley analyst Robert Kad maintains Targa Resources (NYSE:TRGP) with a Overweight and raises the price target from $359 to $362.

**Valero Energy（VLO +3.71%）**
- 📰 [3 Reasons Why Growth Investors Shouldn't Overlook Valero Energy (VLO)](https://finnhub.io/api/news?id=1f012c9d5fd532de3ea1c66671e98876d20951b22218b2f9f995deb368742ea0) — Yahoo：Valero Energy (VLO) is well positioned to outperform the market, as it exhibits above-average growth in financials.
- 📰 [The Zacks Analyst Blog Highlights Quanta, BrightSpring Health, Phillips 66, Valero Energy and HF Sinclair](https://finnhub.io/api/news?id=164717a201bfdded07967843012f965a9031ac3614c379caaaa411a397839ba2) — Yahoo：PWR, BTSG, PSX, VLO and DINO offer growth potential, backed by infrastructure demand, refining strength and expanding earnings prospects.
- 📰 [Cramer Recommends Selling Valero Energy Stock](https://finnhub.io/api/news?id=a1ebbd5150442c60e2636dcb1d54e9f045520519e90bb05e152522f153fcde9b) — Benzinga：CNBC’s Jim Cramer says Valero’s refining-cycle rally is done and recommends selling VLO before margins normalize further.

**APA Corporation（APA +3.59%）**
- 📊 Truist Securities 維持：Hold，目標價 45 → 50
- 📰 [APA Raises 2026 Free Cash Flow Forecast to $2.9B After Q3 Results](https://finnhub.io/api/news?id=5e954384fa04455b4c472dd22d96a9768de9ba5d95a46b2d839468e18bcf04e9) — Yahoo：APA raises its 2026 free cash flow outlook to $2.9 billion as stronger U.S. oil production, higher prices and cost controls boost shareholder returns and debt r…
- 📰 [APA Corporation: Firing On All Cylinders As It Approaches FCF Inflection In Suriname](https://finnhub.io/api/news?id=0199500d3d1402349bbebc2cece59c5d126c8885295381df14d2ee14093be7c6) — SeekingAlpha：APA Corp. is rated Buy: GranMorgu could lift production and free cash flow from 2028; shares trade at 3.5x mid-cycle EBITDA. See more on APA stock here.
- 📰 [Should Value Investors Buy APA (APA) Stock?](https://finnhub.io/api/news?id=5066903b07124c963755e2d97b283735f5b1829632ae685b9c718c11e1767eb8) — Yahoo：Here at Zacks, our focus is on the proven Zacks Rank system, which emphasizes earnings estimates and estimate revisions to find great stocks. Nevertheless, we a…

### 工業

**Bloom Energy（BE +12.97%）**
- 📊 RBC Capital 重申：Outperform，目標價 335 → 335
- 📰 [Bloom Energy Just Jumped 13% After CoreWeave Calmed AI Data Center Fears. History Says What Usually Comes Next for a Stock That Halved](https://finnhub.io/api/news?id=f4dac49b27f7101d0e06ce8e0768668960bf3eaf7cf1617700092d08a4b6eb01) — Yahoo：Bloom Energy jumped 13% on Wednesday after CoreWeave's results eased fears about AI data center spending, adding about $7 billion of market value. The stock is…
- 📰 [Bloom Energy stock surges as CoreWeave earnings spur rally in power stocks](https://finnhub.io/api/news?id=1ace3e4d8437b7e5bbc1f69e501e678088b7122bd2a60c899354c3fb103f30b5) — Yahoo：Bloom Energy rode the wave of positive sentiment following CoreWeave's stellar earnings report.
- 📰 [Bloom Energy (BE) Lands a Major AI Data Center Deal as Revenue Surges](https://finnhub.io/api/news?id=5ffed28d0bfc691e01d37d5382cabb7ac529ce2cffa84e6258bc8f37ef9cf019) — Yahoo：We just published our list of the 10 Best US Stocks to Invest In for Long Term and Bloom Energy Corporation (NYSE:BE) ranked 7th on this list. Recent economic a…

**Generac（GNRC +6.72%）**
- 📊 Piper Sandler 維持：Overweight，目標價 279 → 266
- 📰 [Generac: 3 Key Takeaways for Long-Term Investors](https://finnhub.io/api/news?id=c20957cbd6c05c19d7b1dec3cb6e6a66fce60f92090cfbfafa7ae863a92fd2d5) — Yahoo：Manufacturing is working to triple production with minimal capital spending, a structural advantage that reshapes return-on-investment expectations.
- 📰 [Generac Encourages Homeowners to Prepare for an Unpredictable and Potentially Historic El Niño Winter](https://finnhub.io/api/news?id=4a4fc3b4fe91d744b8cedf34694e6ca80eee0ade9b1224208e9eaed6c45d92c0) — Yahoo：Generac Holdings Inc. (NYSE: GNRC), a leading global designer, manufacturer and provider of energy technology solutions and other power products, is encouraging…

**Vertiv（VRT +6.33%）**
- 📰 [Vertiv Stocks Climb 6.9% as AI Data Center Rebound Lifts Power and Cooling Supplier](https://finnhub.io/api/news?id=d163d80f942a2e3c443e201e6fa69df73d727b0c1043a422392ed7f47bf9d287) — Yahoo：Vertiv's organic orders more than doubled last quarter, and full-year sales guidance rose by $650 million.
- 📰 [Vertiv's Data Center Growth Fuels Revenue: Can It Beat APH and SMCI?](https://finnhub.io/api/news?id=8c125132ce9eacc8663e589288578ffae548f65a00db4cf59fb2c793cb84c0f0) — Yahoo：VRT's AI-driven data center demand fuels 61% revenue growth in the third quarter of 2026, strengthening its position against Amphenol and Super Micro Computer.
- 📰 [Vertiv Beat Big, Then Got Crushed by the Midterms: Why Bernstein Sees 82% Upside](https://finnhub.io/api/news?id=d6f2abadc51790f004c7506c4f8ac6b392a78c5e8dba22952cd8d6b056b73a87) — Yahoo：Key TakeawaysVertiv’s stock is 46% below its May high, even after third-quarter sales rose 61% and organic orders more than doubled. Bernstein kept its Outperfo…

**Leidos（LDOS -2.44%）**
- 📊 Citigroup 維持：Buy，目標價 141 → 163
- 📰 [Citigroup Maintains Buy on Leidos Holdings, Raises Price Target to $163](https://finnhub.io/api/news?id=6f9497e642ba14ec4f5b650e5d18a56575a68aeec1a91450afd0ccf9da1659b6) — Benzinga：Citigroup analyst John Godyn maintains Leidos Holdings (NYSE:LDOS) with a Buy and raises the price target from $141 to $163.
- 📰 [Leidos Launches Agentic AI Platform To Speed Response to Cyber Threats](https://finnhub.io/api/news?id=013c9dad55ee6439e860f017173dce70a2839c8acd7e4b41a342872d2f778355) — Yahoo：UpHold Effect™ Agentic SOC Automation helps federal security teams move from alert overload to clear guidance and automated response while keeping human analyst…

**United Airlines Holdings（UAL -1.82%）**
- （近期無評等變動；新聞未抓）

**Xylem Inc.（XYL -1.79%）**
- （近期無評等變動；新聞未抓）

### 非必需消費

**Tesla, Inc.（TSLA +2.91%）**
- 📰 [Tesla (TSLA) Forms Special Committee To Review Possible All Stock Merger](https://finnhub.io/api/news?id=6eebeb6631107359c4dce0293036797b322c327a2b816ca8059eeee8fbb6977d) — Yahoo：Tesla (NasdaqGS:TSLA) and SpaceX have each formed special committees to evaluate a potential all-stock merger, according to a new report. The review coincides w…
- 📰 [Tesla Rises 4% as Finland Clears Its Driver-Assistance System Ahead of EU Vote; Lucid Slides 2% After Citi Cut, Rivian Dips](https://finnhub.io/api/news?id=d4ad259d2d49d25c770ccaacf7ce043439854f53beca3bf11001396152f6dadc) — Yahoo：Finland just handed Tesla a green light for its driver-assistance software, but the fine print on speed settings and a pending EU committee vote could shape wha…
- 📰 [Nasdaq 100 Futures Drop As Reports Relating To Jensen Huang’s Ousting Outweigh CoreWeave’s Strong Outlook — NVDA, TSLA, SNDK, CAPR, CRWV In Focus](https://finnhub.io/api/news?id=d7e64c7d1c923e54529848fa5f0a9c68b37f6ea780e8006c16005e87309242b6) — Yahoo：Nvidia’s board is preparing to meet as soon as today to determine Jensen Huang’s future as the chipmaker’s chief executive, WSJ reported.

**Expedia Group（EXPE +2.37%）**
- 📊 Argus Research 維持：Buy，目標價 360 → 300
- 📊 Jefferies 維持：Buy，目標價 350 → 325
- 📰 [Expedia and Booking shares rise after United Airlines blocks AI booking](https://finnhub.io/api/news?id=e3af5f930d0d110dcef644cd0555699a063ee777ddb1ce5ee332b9072cbc5ca1) — Yahoo：Investing.com -- Expedia Group Inc and Booking Holdings Inc shares gained Wednesday after United Airlines announced it will block personal artificial-intelligen…
- 📰 [Expedia, Booking and Airbnb fall as OpenAI reportedly buys AI travel platform](https://finnhub.io/api/news?id=273ae4c2ad93cd31ccf3302130189da330f52603399b1d4afda58986e9d091a1) — Yahoo：Investing.com -- Online travel stocks fell to session lows Tuesday after The Information reported that OpenAI had acquired Odessia, an AI-powered travel plannin…
- 📰 [Expedia: B2B Momentum And Higher Margins Reinforce Our Buy Rating](https://finnhub.io/api/news?id=2acfd46eab0e3b74f73d467da282eabb2f98444c8c2b5bc9c89d6f52a5231382) — SeekingAlpha：Expedia delivered 12% revenue growth in Q3, with adjusted EBITDA up 17% and margins expanding by 135 basis points. Read why EXPE stock is a buy.

**Chipotle Mexican Grill（CMG +2.00%）**
- 📰 [Chipotle Mexican Grill (CMG) Could Be 29% Overvalued After Earnings Profit Slipped](https://finnhub.io/api/news?id=2939bc7d9637f3396115ba0b25de24322f67156284b841f730cd562e85e8aa51) — Yahoo：Chipotle Mexican Grill (CMG) just posted third quarter results that paired higher revenue with lower profitability, as net income and earnings per share decline…
- 📰 [Can Chipotle's 1,700 Upgraded Kitchens Revive Traffic in 2027?](https://finnhub.io/api/news?id=2d381c5d67d0d51874c943ac0132ed4877ee7f489f7ef7ba392f3a5c38faed6e) — Yahoo：CMG's 1,700-plus upgraded kitchens are lifting sales and speeding service. Can the broader rollout revive traffic in 2027 despite weak consumer demand?
- 📰 [Chipotle (CMG) Stock Could Be 19% Overvalued Following Q3 Expansion News](https://finnhub.io/api/news?id=b5f8d347db5603bb06674d065bb3fb319085ffb788fd3bcc417684e573d7ddfb) — Yahoo：Chipotle Mexican Grill has delivered a 20.5% gain over the past year, even as the year-to-date return is slightly negative, which puts a spotlight on whether th…

**Airbnb（ABNB -3.09%）**
- 📊 Jefferies 維持：Buy，目標價 215 → 230
- 📊 Rothschild & Co 升評：Sell → Neutral，目標價 135 → 190
- 📰 [Airbnb and Travel Stocks Fall as OpenAI Acquisition Raises New AI Threat](https://finnhub.io/api/news?id=4f4c845b9491b2289ad70a16bd45fe70d55eb54d5467c12fc4de12b37dd822bd) — Yahoo：OpenAI’s acquisition of Odessia raises fresh concerns about artificial-intelligence disruption.
- 📰 [Airbnb Is Building Its Own AI Travel Agent, Co-Founder Says](https://finnhub.io/api/news?id=84d48ffb730602747adad2af9b275ed74e33cac72019473c0c537e4871cc5f14) — Yahoo：Airbnb Co-founder and Chief Strategy Officer Nathan Blecharczyk says the company is developing its own specialized AI travel agent that will eventually be able…
- 📰 [Airbnb reduces customer support cost per booking 17%](https://finnhub.io/api/news?id=701e935ebbf7dfb38268be6bcde205ab35748f8ebe48792c7e5ade7d29c5a55f) — Yahoo：CEO Brian Chesky said the year-over-year cost savings were driven by improvements to its AI assistant.

**Lululemon Athletica（LULU -2.65%）**
- 📊 Cantor Fitzgerald 首評：Neutral，目標價 121
- 📰 [Lululemon Brings Enzymatically Recycled Polyester to Running Shorts](https://finnhub.io/api/news?id=f1b7b23553cb6b8e91c7efd154f69703a90bd58e9b7b6ec20c0d7feaadb02802) — Yahoo：The limited Australia and New Zealand launch builds on the brand’s 10-year sourcing plan with Samsara Eco.
- 📰 [Lululemon Rebuild Continues With New Marketing, Strategy Officers](https://finnhub.io/api/news?id=9ed88d3306b6b8e729905786fc3d0ba58bb65df2a13df26f2675645ac9193d77) — Yahoo：Execs from Four Seasons Hotels and Boston Consulting Group join the retailer's new team.
- 📰 [Lululemon Athletica (LULU) Names Ben Hallam And Emily Sunderland To Top Roles](https://finnhub.io/api/news?id=5fd8697d6c02910f5defea974b45d95ba6f93ad0bcd4e57e1ecfd57afc67057a) — Yahoo：Lululemon Athletica (NasdaqGS:LULU) appointed Ben Hallam as Chief Brand Officer, bringing experience from large global consumer companies. Emily Sunderland join…

**McDonald's（MCD -2.61%）**
- 📰 [McDonald's (MCD) Could Be 21% Above Fair Value Following Its Uber Expansion](https://finnhub.io/api/news?id=135822986d82a32fd214a5bb1d78374c29bfa3819d6c4453c7752653a991293b) — Yahoo：McDonald's (MCD) has expanded its alliance with Uber across 14 countries, making Uber the preferred delivery provider for orders in the McDonald's app and linki…
- 📰 [McDonald's Expands Uber Partnership: Can It Drive More Growth?](https://finnhub.io/api/news?id=099d9c7940f80555492d34c6e35449091c3affa2569abb2b1aeaaabba63aa880) — Yahoo：MCD expands its Uber partnership across 14 global markets to boost delivery, digital ordering and loyalty, while improving convenience and driving incremental s…
- 📰 [McDonald's Stock Is Up 24% in the Last Month - Is MCD Near a Peak?](https://finnhub.io/api/news?id=67471599679cd8b2b5b33c8c337a91340a20e33a170d0a72eb3589d003b95fbf) — Yahoo：McDonald's Corp. stock hit a low of $230.88 about a month ago (Oct. 7), but has risen 24% since then. Is it at a peak? It just released the Q3 10-Q and its FCF…

### 不動產

**Equinix（EQIX +1.72%）**
- （近期無評等變動；新聞未抓）

**Iron Mountain（IRM +1.49%）**
- （近期無評等變動；新聞未抓）

**Alexandria Real Estate Equities（ARE +0.85%）**
- （近期無評等變動；新聞未抓）

**Vivmark Residential（VMRK -1.70%）**
- 📊 UBS 維持：Buy，目標價 71 → 74
- 📊 Cantor Fitzgerald 維持：Neutral，目標價 70 → 68

**Camden Property Trust（CPT -0.80%）**
- 📊 Morgan Stanley 維持：Equal-Weight，目標價 118 → 107
- 📊 UBS 維持：Neutral，目標價 101 → 107
- 📊 Cantor Fitzgerald 維持：Neutral，目標價 111 → 101

**Invitation Homes（INVH -0.78%）**
- （近期無評等變動；新聞未抓）

### 醫療保健

**Regeneron Pharmaceuticals（REGN +4.73%）**
- 📰 [Regeneron Pharmaceuticals, Inc. (REGN) Discusses Genetic Center and CIDEB siRNA Development for MASH - Slideshow](https://finnhub.io/api/news?id=0817781b89426e0f7ab31c5c131e5fe023db8d56cdee298f09bd858b0fe7cd3b) — SeekingAlpha：2026-11-10. The following slide deck was published by Regeneron Pharmaceuticals, Inc.
- 📰 [Regeneron Pharmaceuticals Advances Liver Drug After Early Trial Cuts Liver Fat by Up to 80%](https://finnhub.io/api/news?id=ac7c833edc2f63c517444d0ba137a28c386ce3ed089ee0715b3a78baab4cc73a) — Yahoo：Regeneron Pharmaceuticals (NASDAQ:REGN) is advancing its experimental CIDEB therapy into later-stage studies for a form of liver disease after an early trial sh…
- 📰 [Alnylam Collaborator Regeneron Advances CIDEBsiRNA with Promising First-in-Human Results in Patients with MASLD](https://finnhub.io/api/news?id=2bdf68199e7f85275b77562ced30cb39bcbf04ed625e6e1f5e3dbf05532885e0) — Yahoo：CAMBRIDGE, Mass., November 10, 2026--Alnylam Pharmaceuticals, Inc. (Nasdaq: ALNY), the leading RNAi therapeutics company, today announced that its collaborator,…

**DaVita（DVA +2.35%）**
- 📰 [DaVita Inc. (DVA) Presents at 8th Annual Wolfe Research Healthcare Conference Transcript](https://finnhub.io/api/news?id=01dc74d765b0cb32a8a7d0571a6ced767af338375c47adf0409a422f1f89e80f) — SeekingAlpha：DaVita Inc. (DVA) 8th Annual Wolfe Research Healthcare Conference November 10, 2026 8:40 AM ESTCompany ParticipantsJoel Ackerman - CFO &...
- 📰 [DaVita Sees Growth and Drug Savings Offsetting 2027 Reimbursement Pressure](https://finnhub.io/api/news?id=34eb0ab95399ea87a2271ccb9f98b11f3ddcb5734db36b9d3428f026005f5e44) — Yahoo：DaVita (NYSE:DVA) expects stronger dialysis treatment growth and pharmaceutical cost savings to help offset reimbursement pressures in 2027, though Chief Financ…

**ResMed（RMD +1.89%）**
- （近期無評等變動；新聞未抓）

**Stryker Corporation（SYK -1.70%）**
- 📊 Piper Sandler 維持：Overweight，目標價 325 → 350

**Solventum（SOLV -1.62%）**
- （近期無評等變動；新聞未抓）

**GE HealthCare（GEHC -1.26%）**
- 📊 Argus Research 維持：Buy，目標價 85 → 110

ℹ️ 12 檔漲跌幅較小的個股沒有抓新聞（每次最多 40 檔，Finnhub 免費額度）

**財報｜11/10 盤後 ～ 11/11（依營收規模排序）**

| 公司 | 公布時間 | EPS 實際 / 預估 | 營收 實際 / 預估（億美元） | 報告日股價 |
|:---|:---|---:|---:|---:|
| CoreWeave（CRWV）＊ | 11/10 盤後 | -1.00 / -1.18（優於預期） | 36.6 / 36.2 | +24.71% |

＊＝非 S&P 500 成分股。11/10 盤後公布的財報反應在報告日股價；11/11 盤後公布的，報告日股價尚未反映，實際數字也可能稍晚才更新。

6. Treasury Yield Curve

In [6]:
def treasury_curve():
    frames = []
    for yr in (AS_OF.year, AS_OF.year - 1):          # 年初資料不足兩天時補前一年
        url = ("https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/"
               f"{yr}/all?type=daily_treasury_yield_curve&field_tdr_date_value={yr}&page&_format=csv")
        r = HTTP.get(url, timeout=TIMEOUT)
        r.raise_for_status()
        frames.append(pd.read_csv(io.StringIO(r.text)))
        if sum(len(f) for f in frames) >= 2:
            break
    df = pd.concat(frames, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    return df.sort_values("Date", ascending=False).reset_index(drop=True)


def curve_shape(d2, d10):
    """依 2 年、10 年殖利率變動（bp）判斷曲線型態"""
    if abs(d10 - d2) < 1:
        return "大致平行移動（2 年與 10 年變動相近）"
    if d2 <= 0 and d10 <= 0:
        return "牛平（長端降更多）" if d10 < d2 else "牛陡（短端降更多）"
    if d2 >= 0 and d10 >= 0:
        return "熊陡（長端升更多）" if d10 > d2 else "熊平（短端升更多）"
    return "扭轉變平（短升長降）" if d2 > 0 else "扭轉變陡（短降長升）"


def fred_curve():
    """備援：FRED 的公債殖利率（資料同樣來自財政部），5 個天期同時抓"""
    if not FRED_API_KEY:
        raise ValueError("未設定 FRED_API_KEY")
    cols = {"3 Mo": "DGS3MO", "2 Yr": "DGS2", "5 Yr": "DGS5", "10 Yr": "DGS10", "30 Yr": "DGS30"}

    def series(sid):
        obs = fred("series/observations", series_id=sid, observation_start=f"{AS_OF - timedelta(days=30):%Y-%m-%d}")
        return pd.to_numeric(pd.Series({o["date"]: o["value"] for o in obs["observations"]}), errors="coerce")

    with ThreadPoolExecutor(len(cols)) as ex:
        df = pd.DataFrame(dict(zip(cols, ex.map(series, cols.values())))).dropna()
    df.index = pd.to_datetime(df.index)
    return df.rename_axis("Date").reset_index().sort_values("Date", ascending=False).reset_index(drop=True)


try:
    tc, source = treasury_curve(), "美國財政部"
except Exception as e:
    print(f"財政部資料無法取得（{safe_err(e)}），改用 FRED")
    tc, source = fred_curve(), "FRED"
cur, prev = tc.iloc[0], tc.iloc[1]
rows = ["| 天期 | 殖利率 | 變動 |", "|:---|---:|---:|"]
for t in ["3 Mo", "2 Yr", "5 Yr", "10 Yr", "30 Yr"]:
    rows.append(f"| {t} | {cur[t]:.2f}% | {(cur[t] - prev[t]) * 100:+.0f} bps |")
d2, d10 = (cur["2 Yr"] - prev["2 Yr"]) * 100, (cur["10 Yr"] - prev["10 Yr"]) * 100
spread, spread_prev = (cur["10 Yr"] - cur["2 Yr"]) * 100, (prev["10 Yr"] - prev["2 Yr"]) * 100
stale = "" if cur["Date"].date() == AS_OF.date() else (
    f"\n\n⚠️ 這是 {cur['Date']:%m/%d} 的曲線，不是報告日 {AS_OF:%m/%d}：財政部約在美東 15:30–18:00 公布當日數字，"
    "債市休市日（如退伍軍人節、哥倫布日）不更新。稍後重跑本格；當日 10 年期變動先看第 3 格的美債。")
display(Markdown(f"**美債殖利率｜{cur['Date']:%Y-%m-%d}（vs {prev['Date']:%m/%d}，{source}）**\n\n" + "\n".join(rows) +
                 f"\n\n- 2s10s 利差：{spread:+.0f} bps（前一日 {spread_prev:+.0f} bps）\n- 曲線型態：{curve_shape(d2, d10)}" + stale))

**美債殖利率｜2026-11-10（vs 11/09，美國財政部）**

| 天期 | 殖利率 | 變動 |
|:---|---:|---:|
| 3 Mo | 4.30% | +6 bps |
| 2 Yr | 4.73% | +17 bps |
| 5 Yr | 4.81% | +14 bps |
| 10 Yr | 4.98% | +10 bps |
| 30 Yr | 5.35% | +9 bps |

- 2s10s 利差：+25 bps（前一日 +32 bps）
- 曲線型態：熊平（短端升更多）

⚠️ 這是 11/10 的曲線，不是報告日 11/11：財政部約在美東 15:30–18:00 公布當日數字，債市休市日（如退伍軍人節、哥倫布日）不更新。稍後重跑本格；當日 10 年期變動先看第 3 格的美債。

7. Economic data releases + free news

In [7]:
CNBC = "https://www.cnbc.com/id/{}/device/rss/rss.html"
TOP, FINANCE, GENERAL = CNBC.format(100003114), CNBC.format(10000664), "finnhub:general"
FIRST = ["聯準會官員談話", "總經與聯準會", "科技與 AI"]   # 先挑這幾組；同一則新聞只出現一次，不會在後面的組別重複
MARKET_RE = re.compile(r"\b(stocks?|shares|markets?|wall street|dow|s&p|nasdaq|earnings|rall(y|ies)|sell-?off|investors?|"
                       r"futures)\b", re.I)
OIL_RE = re.compile(r"\b(oil|crude|brent|wti|opec|barrels?|gasoline|diesel|refiner\w*|hormuz|iran\w*|eia|inventor(y|ies)|"
                    r"energy prices?|tankers?|sanctions?)\b", re.I)
FX_RE = re.compile(r"\b(dollar|greenback|dxy|euro|yen|sterling|pound|treasur(y|ies)|yields?|bonds?|auction)\b", re.I)

GROUPS = {   # 組別: (最多幾則, [(來源名稱, 網址, 篩選)])。篩選＝標題要符合的關鍵字，None＝全收；同一網址只抓一次
    "盤勢與個股": (10, [("CNBC", TOP, MARKET_RE), ("CNBC Earnings", CNBC.format(15839135), None),
                        ("CNBC Investing", CNBC.format(15839069), None),
                        ("Google News", gnews('"stock market today"'), None), ("Finnhub", GENERAL, MARKET_RE)]),
    # Fed 官員談話多半登在 CNBC Finance 與各家媒體（CNBC Economy 沒有，Fed 官網只有理事的演講稿），所以用 Google News 加關鍵字
    "聯準會官員談話": (10, [("CNBC Finance", FINANCE, SPEAK_RE), ("Google News", GN_FED, SPEAK_RE), ("CNBC", TOP, SPEAK_RE),
                            ("Finnhub", GENERAL, SPEAK_RE),
                            ("Fed 理事演講", "https://www.federalreserve.gov/feeds/speeches.xml", None)]),
    "總經與聯準會": (12, [("CNBC Economy", CNBC.format(20910258), None), ("CNBC Finance", FINANCE, MACRO_RE),
                          ("Google News", GN_POLICY, MACRO_RE), ("Google News", GN_DATA, MACRO_RE),
                          ("CNBC", TOP, MACRO_RE), ("Finnhub", GENERAL, MACRO_RE),
                          ("Fed 聲明與會議紀錄", "https://www.federalreserve.gov/feeds/press_monetary.xml", None)]),
    "科技與 AI": (12, [("CNBC Tech", CNBC.format(19854910), None), ("Google News", GN_TECH, TECH_RE),
                       ("CNBC Finance", FINANCE, TECH_RE), ("Finnhub", GENERAL, TECH_RE)]),
    "美債與美元": (8, [("Google News", gnews('"Treasury yields"'), FX_RE),
                       ("Google News", gnews('"dollar index" OR "US dollar" euro OR yen OR sterling'), FX_RE)]),
    "黃金": (6, [("Google News", gnews('"gold prices"'), None)]),
    "原油": (8, [("CNBC Energy", CNBC.format(19836768), OIL_RE), ("OilPrice", "https://oilprice.com/rss/main", OIL_RE),
                 ("Google News", gnews('"oil prices"'), OIL_RE)]),
    "中文（鉅亨網美股）": (8, [("鉅亨網", "cnyes:us_stock", None)]),
}

# 近兩個交易日公布的美國經濟數據。FRED 代號: (名稱, 算法, 單位)；pct＝月增與年增率，chg＝較前期增減，lvl＝數值
ECON = {
    "CPIAUCSL": ("CPI", "pct", ""), "CPILFESL": ("核心 CPI", "pct", ""),
    "PPIFIS": ("PPI（最終需求）", "pct", ""), "PPIFES": ("核心 PPI（不含食品能源）", "pct", ""),
    "PCEPI": ("PCE 物價指數", "pct", ""), "PCEPILFE": ("核心 PCE", "pct", ""), "PCE": ("個人消費支出", "pct", ""),
    "PAYEMS": ("非農就業人數", "chg", " 千人"), "UNRATE": ("失業率", "lvl", "%"), "CES0500000003": ("平均時薪", "pct", ""),
    "ICSA": ("初領失業金人數", "lvl", " 人"), "JTSJOL": ("JOLTS 職缺", "lvl", " 千"),
    "RSAFS": ("零售銷售", "pct", ""), "RSFSXMV": ("零售銷售（不含汽車）", "pct", ""),
    "INDPRO": ("工業生產", "pct", ""), "DGORDER": ("耐久財訂單", "pct", ""), "NEWORDER": ("核心資本財訂單", "pct", ""),
    "HOUST": ("新屋開工", "lvl", " 千戶（年率）"), "PERMIT": ("建築許可", "lvl", " 千戶（年率）"),
    "HSN1F": ("新屋銷售", "lvl", " 千戶（年率）"), "A191RL1Q225SBEA": ("實質 GDP 季增年率", "lvl", "%"),
}


def econ_row(sid):
    """這項數據在前一交易日之後有公布 → 表格的一列；沒有 → None"""
    name, kind, unit = ECON[sid]
    meta = fred("series", series_id=sid)["seriess"][0]
    released = pd.Timestamp(meta["last_updated"][:10])
    if released < PREV:
        return None
    obs = [(pd.Timestamp(o["date"]), float(o["value"])) for o in
           fred("series/observations", series_id=sid, sort_order="desc", limit=15)["observations"] if o["value"] != "."]
    (d0, v0), (d1, v1), (d2, v2) = obs[:3]
    freq = meta.get("frequency_short")
    period = f"{d0.year} Q{d0.quarter}" if freq == "Q" else f"{d0:%m/%d} 當週" if freq == "W" else f"{d0.month} 月"
    num = lambda v: f"{v:,.1f}" if abs(v) < 100 else f"{v:,.0f}"
    if kind == "pct":
        # 中間缺月份（如政府停擺）時不算，免得把兩個月的變動當成月增
        mom = lambda a, da, b, db: f"月增 {(a / b - 1) * 100:+.1f}%" if da - pd.DateOffset(months=1) == db else "月增 —"
        yoy = next((v for d, v in obs if d == d0 - pd.DateOffset(years=1)), None)
        cur = mom(v0, d0, v1, d1) + (f"、年增 {(v0 / yoy - 1) * 100:+.1f}%" if yoy else "")
        before = mom(v1, d1, v2, d2)
    elif kind == "chg":
        cur, before = f"{v0 - v1:+,.0f}{unit}", f"{v1 - v2:+,.0f}{unit}"
    else:
        cur, before = f"{num(v0)}{unit}", f"{num(v1)}{unit}"
    return released, f"| {name} | {period} | {released:%m/%d} | {cur} | {before} |"


def fetch_items(name, url):
    if url == GENERAL:
        if not FINNHUB_API_KEY:
            return []
        return [(datetime.fromtimestamp(d["datetime"], timezone.utc), clean(d.get("headline"), 300), d.get("url", ""),
                 f"Finnhub / {d.get('source', '')}", clean(d.get("summary")))
                for d in finnhub("news", category="general")
                if d.get("datetime") and not blocked(d.get("url"), d.get("source"))]
    if url.startswith("cnyes:"):
        r = HTTP.get(f"https://api.cnyes.com/media/api/v1/newslist/category/{url[6:]}",
                     params={"limit": 50}, timeout=TIMEOUT)
        r.raise_for_status()
        return [(datetime.fromtimestamp(d["publishAt"], timezone.utc), clean(d.get("title")),
                 f"https://news.cnyes.com/news/id/{d['newsId']}", name, clean(d.get("summary")))
                for d in r.json()["items"]["data"]]
    return rss(name, url)


def pick(sources, limit, fetched, shown):
    """一組新聞：只留最近 NEWS_HOURS 小時、符合篩選、不是雜訊、前面的組別沒出現過的；各來源輪流取（主要財經媒體優先），
    去掉重複轉載後取 limit 則，依時間新到舊顯示。挑中的標題記到 shown"""
    cutoff = datetime.now(timezone.utc) - timedelta(hours=NEWS_HOURS)
    key = lambda it: re.sub(r"\W+", "", it[1].lower())
    pool, errs = [], []
    for i, (name, url, flt) in enumerate(sources):
        its, err = fetched[url]
        if err:
            errs.append(f"{name}：{err}")
        its = [it for it in its or [] if it[0] and it[0] >= cutoff and it[1] and not junk(it[1], it[3])
               and (flt is None or flt.search(it[1])) and key(it) not in shown]
        if "news.google.com" in url:
            its.sort(key=lambda it: not preferred(it[3]))      # Google News：主要財經媒體在前，其餘維持原本的相關度順序
        else:
            its.sort(key=lambda it: it[0], reverse=True)
        pool += [((not preferred(it[3]), pos, i), it) for pos, it in enumerate(its)]
    items = dedupe([it for _, it in sorted(pool, key=lambda x: x[0])])[:limit]
    shown.update(map(key, items))
    return sorted(items, key=lambda it: it[0], reverse=True), errs


# 所有新聞來源與 FRED 數據一次同時抓，再依組別挑選
tasks = {url: ("news", partial(fetch_items, name, url)) for _, sources in GROUPS.values() for name, url, _ in sources}
n_news = len(tasks)
if FRED_API_KEY:
    tasks.update({("econ", sid): ("fred", partial(econ_row, sid)) for sid in ECON})
res = gather(tasks, {"news": n_news, "fred": 6})

rows = sorted((res["econ", sid][0] for sid in ECON if res.get(("econ", sid), (None,))[0]), key=lambda r: r[0], reverse=True)
econ_errs = [f"{ECON[k[1]][0]}：{err}" for k, (v, err) in res.items() if isinstance(k, tuple) and err]
lines = [f"### 經濟數據｜{PREV:%m/%d} 以來公布（FRED）"]
if rows:
    lines += ["| 數據 | 資料期 | 公布日 | 數值 | 前期 |", "|:---|:---|:---|---:|---:|"] + [r for _, r in rows]
    lines += ["", "月增、年增用季調後指數計算，四捨五入後可能和新聞稿差 0.1 個百分點；FRED 沒有市場預期值，"
              "預期與細項看下方新聞與原始新聞稿。ISM、消費者信心、密大信心不在 FRED，看新聞。"]
else:
    lines.append("- 清單內的數據這兩個交易日都沒有公布" if FRED_API_KEY else "- 未設定 FRED_API_KEY")
if econ_errs:
    lines.append(f"- ⚠️ 無法取得：{'、'.join(econ_errs[:3])}{f' 等 {len(econ_errs)} 項' if len(econ_errs) > 3 else ''}")
lines.append("")
shown, picks = set(), {}
for g in FIRST + [g for g in GROUPS if g not in FIRST]:
    picks[g] = pick(GROUPS[g][1], GROUPS[g][0], res, shown)
for g, (items, errs) in picks.items():
    lines += [f"### {g}"]
    for when, title, link, src, summary in items:
        title = title.replace("[", "(").replace("]", ")")
        lines.append(f"- `{when.astimezone(TW):%m/%d %H:%M}` [{title}]({link}) — *{src}*" + (f"：{summary}" if summary else ""))
    if not items:
        lines.append("- （無）")
    if errs:
        lines.append(f"- ⚠️ 無法取得：{'、'.join(errs)}")
    lines.append("")
display(Markdown("\n".join(lines)))

### 經濟數據｜11/10 以來公布（FRED）
| 數據 | 資料期 | 公布日 | 數值 | 前期 |
|:---|:---|:---|---:|---:|
| CPI | 10 月 | 11/10 | 月增 +0.4% | 月增 +0.6% |
| 核心 CPI | 10 月 | 11/10 | 月增 +0.3% | 月增 +0.3% |

月增、年增用季調後指數計算，四捨五入後可能和新聞稿差 0.1 個百分點；FRED 沒有市場預期值，預期與細項看下方新聞與原始新聞稿。ISM、消費者信心、密大信心不在 FRED，看新聞。

### 聯準會官員談話
- `11/12 06:35` [Chicago Fed's Goolsbee Calls October CPI 'Borderline Disturbing' as December Rate Decision Looms](https://news.google.com/rss/articles/CBMidkFVX3lxTFBoTWUweE1lc1VBVE8wenRuWlR5VEJIbmwtdHRKOXZnSXpsYmxRSmZJQm02cTZaRG54UFFseXJ3akxveV9iSEUtZmRlSmNEMWJVTHFSYl9zYWhXa3FUWG5WSHI3alFUWk9VOFBMVlNjYjRVUDRCekE?oc=5) — *Google News / BigGo Finance*
- `11/12 05:18` [Fed’s Bowman Says AI Now Drives About a Quarter of America’s Excess Inflation](https://news.google.com/rss/articles/CBMiuwFBVV95cUxOMmZCc1gtdGh3V2t4TjljczE2T1NXbEhHQVdkeGUzZUpMbER6bl9leFQ1SDNtQ0Z0dWhVSE5pMlduRXhMTGVkMnNGTjk3WmRjVThaV05VSFdIcWNNczJXYXYtS1lCQWN5N3NmWXdkOG0tT3FJZjFBa1NGWXJBb0Zqb2JOXzg3ZDMxcEs4RFNmSVFHT3pNX01scV9FZ1BiZUFGQWR4eUI1VnQ2bldac29jUkw0aTladVF6WHAw?oc=5) — *Google News / 24/7 Wall St.*
- `11/12 04:08` [Fed’s Schmid Warns Against Viewing Oil Shock as Transitory](https://news.google.com/rss/articles/CBMikgFBVV95cUxOb0dkQlJ3Nm5XZEI5OHRFVEUyQ2xrWFJUR3lYSERGSTgzMXd4ejhxVjVJQmFDSThkQWliQXlSOTZ6QVpaVEFZUUtYU3hLaDB0VklwdG1FQjdvZ3gxTVJ6Z0Z5aU1BNlFfd1dEd2lRMmFSYVBIdUdGQndyOEVOZ0ZNdnhPTFk2ZnU5N1ZfazZWWnI0dw?oc=5) — *Google News / EnergyNow.com*
- `11/12 02:43` [Philadelphia Fed's Anna Paulson backs gradual interest rate hikes](https://news.google.com/rss/articles/CBMigAFBVV95cUxPb1JlbERBeVM5R2FRNTNzQzQ3Q2ljRWdOV2tWeG4xNzMtRWNYSUFuaWFEbWdrNzBXVlZDcjhMSGl1dWZ2RS0zdWxLSmtYZnh5UUEzbHM5QXRWTVBWeHpqM1ZXVEY3TG54TERwbFc3WWh1VC0yV3JQd1hCTG82bVczTg?oc=5) — *Google News / Quartz*
- `11/12 02:31` [Fed debates if New York Fed's John Williams must retire in 2027](https://news.google.com/rss/articles/CBMie0FVX3lxTE5RdGE5ekJHT3dFR0FoOGRKNjNXSkNjOUZOUllKUnh5cFdYZ3ItWUpTeHJ5dkZoYjIyY244cTlBX2dyUGNQVjBQSWMzeG8xeDB2VnlUX0QyV0w0bVF5TGZPeWVDRHY5OWxPM00xWDNrOWZ3Yl9LcnJxMHo1cw?oc=5) — *Google News / Quartz*
- `11/12 02:30` [Fed’s Goolsbee Warns AI Data Center Boom Is Starting to Look Like ‘Demand Overheating,’ Says Central Bankers Must Be ‘Aggressive Up Front’](https://news.google.com/rss/articles/CBMilgFBVV95cUxPMWJnUTlDbXJyck9HVDdLVVowVWpyZ0VkUk45bG51V2FzeldVU1N0TFdwektyTjZuMjQtMVVXNmZYSEkwQ3ZzUzd1LTJDNjFOR2p5SlU2VExBZ3dzcXZOSEtWUGltY2cySmVDaXJzanVvbEZGZ0FEWW50aHJLc3pxYjJEYVFWNXdSWFVjQ3FCWEVaeGE3OFE?oc=5) — *Google News / Yahoo Finance*
- `11/12 00:49` [Fed's Goolsbee calls inflation report 'borderline disturbing' as central bank weighs more hikes](https://www.cnbc.com/2026/11/11/goolsbee-october-inflation-fed-rates-disturbing-distressing.html) — *CNBC Finance*：The consumer price index rose more than expected in October, raising alarm among Federal Reserve officials.
- `11/12 00:19` [NY Fed's Williams retirement date is debatable - report](https://news.google.com/rss/articles/CBMilwFBVV95cUxPQlBOSWxUSWZJU3lIYWFaMjl6aWtTaFdCbGR2Q3RmYTVVY0R1Rm5NeHNQQ3p4VER0QU5FUmxhYmdMRV8teGpVdEdQRTJ4LVhuNEFYNTBQOGhNQXNLajB0WDFhSHRQNUNDWG9la0tuc294N2NocG11OHFOWHFselFuNmdLMkpHVG5EZjNyQ1dlXzVmRkthdFFN?oc=5) — *Google News / Seeking Alpha*
- `11/11 22:39` [Fed’s Bowman Warns of a ‘Large Correction’ in Stocks if AI Bets Fall Short, but Says the Inflation Fight Isn’t Over](https://news.google.com/rss/articles/CBMi4gFBVV95cUxObFJMWTdRQkxpSkx1Z3g2ZC1vNHhxeXRfU3BsbnlUV21meGJSaFdsdmM3WG1yWmppampWMEZDUG5fekZxbWxCbG9WSmJ1OS14MnhRMDcwSFVZRlpmMGM2dGw0bXo3ZEpMUnA1X09GamF5WW1kWWw0dy1sSGJuWFVlRzFHVEQtaloyRm1PZDM0MHY0UE1SMC0yb01QbkJUVHlsemM3NWpiaWVVSEdvZ1BaczNzMXM4dEFDdlpDby1EZFBlX3ZyVmZ4VmFkMVRLRTkyTmxUTDlFenBkX2FBS3Y0aVlR?oc=5) — *Google News / InsuranceNewsNet*
- `11/11 16:32` [Gold dips after Fed's Waller drops the 50 bps hike bomb and US CPI comes above expectations](https://news.google.com/rss/articles/CBMiygFBVV95cUxQOGF2aV8wWnB3cGZGMHRKd3lqUzZ1WFNBU1ZaZ3NISDdBT0tVVlRGTnQ5NU02Rnh3cEhMNUtPaVFZTjZpZlRhVjI2V0NQRFJybXRzSzhwUFZRQzM1ZnNodDZWcE1DbnRDUGlEdmdCT3VEWkJLc3V0SHozWVFkSEpjRjBHVEd5MFhxSFVaSmpzeUptX20wb2pjcVpxYTF2bl9VTjFVN3JfWXVqU2h6Z2ZZUURic185enZUMGhqeUx0SmdsX0NaTk14ZmF3?oc=5) — *Google News / investingLive*

### 總經與聯準會
- `11/12 09:31` [Gold clings below $4,000 as inflation fears fuel bets on bigger Fed hikes](https://news.google.com/rss/articles/CBMixgFBVV95cUxPZWJfbkhuZVk5eEFDdFB0NUV5Skl4djhRZkF6QkpwQkxMZ0s1VzlYMjlPNGRSak9EYnVocENJQWhYMFVSTWV0a2p3cm1reHd6V0hXSnVVWC1DZUhhTGhuSFpGbzBRTkx4T0VEM3dVS1BZQ3BYRDNfQkRrTXhEX0tXQWhzYzljVEp0ek5yNDhGYUdqTElZbXFqQm9MaTNQRXpCQjZQNGNzTGhMVHFwdU9HRmR3ZVV5QlI4R1hpdlZlM042NmNfN0E?oc=5) — *Google News / Investing.com*
- `11/12 06:59` [Analysis: Is Nvidia too big to fail? Its AI financing web raises the stakes](https://www.cnbc.com/2026/11/11/nvidia-jensen-huang-ai-financial-system-risk.html) — *CNBC Economy*：The tech giant's web of ties to other companies is raising the possibility that it could do far-reaching economic and financial damage.
- `11/12 06:30` [Canada’s ambassador warns Trump tariffs are driving up costs, threatening U.S. investment](https://www.cnbc.com/2026/11/11/canada-ambassador-trump-tariffs-investments.html) — *CNBC*：U.S.-Canada trade tensions persist as Trump's tariffs drive up costs, threaten investment and complicate negotiations between the two countries.
- `11/12 06:00` [Gold falls below $4,000 as dollar surges, Fed hike bets rise - Kitco PM Report](https://news.google.com/rss/articles/CBMitwFBVV95cUxPQkRUZVFwS0pNb3NBQjg1aUNMcjQ5Y0pVU0NLUjU4M3pJLXh5WFYtd3ZFMXhzM2I2Y3RDd25WeXhvaWY4ZHh1TzhrRVJIRzRFT2JJRWE3Tkc2MkVTOFNSZnUzQTVTMFltZXl1cVZRSXdmR0h4ekJUdEhHYjNGTmpnbU5ZTnhyZzlJVGxlc3F2NUhCa3pxd1UtRnd1Yk1UblE2Y05TdmNOR0xMbjhyQkhLZ0hCUFJwYUE?oc=5) — *Google News / Kitco*
- `11/12 03:47` [Economists say Fed could go overboard on rate hikes after hot October inflation report](https://news.google.com/rss/articles/CBMirgFBVV95cUxOWDRqb2M0ckpxTENJNWhNaDNHRUpYaE9pVlI3Ri1zUDJrdkxiMGFMTG5MY0wtUFAzY3JFZjNuX1Rpb2VoQWJhamJ6ejdwbW9TaGVBM2FwTGdoajNvTGwzUFNmWWxWWGJlRzA1czJQc0VkcUkxaEs2dVYyQTRia2I3NHZleVVleFhEVDRnVEQ4d2NWUHMyWTMyQ2gwaWlKQTNYdk8wQTdVbEZNTG5abkE?oc=5) — *Google News / Business Insider*
- `11/12 01:20` [Wall Street thinks the Fed made a 'mistake' — and now more hikes are coming](https://news.google.com/rss/articles/CBMiyAFBVV95cUxNdGY2RnBVRFBmcnVBQWw3TG1jSV94WHcyTTFtV3ZPZE1PYU13QlhhbFlDZHpweDhKTGc5TzcyVDZJcEJYbmNFMGdLYjdVQlhxRUxiQTUwX3p2RVZXMUs0ZjkwQU8yQVdsQ2ZuNEtOWHpfSWdFREVNRXFZeEE0bTJtZUQ5SVJUX1k5ZDA0TUFqSzJVRks3N3AwbTVMVHNoUThKUk91MjRFZHFqUDNnb1pseHB3SUxuc1FzMFpLZjNfMElrc2xtWTBXUw?oc=5) — *Google News / Yahoo Finance*
- `11/11 20:40` [Goldman now sees Fed interest rate hike at December, March meetings](https://news.google.com/rss/articles/CBMirgFBVV95cUxPZFIzUlJQMWd1NnFDdGRWYUVWNnIxSHhTX0NFUWwtbC1zMkIwOE80R3VSVVRRNnVkSTVxSmdwZGQtTlpuOTBMQ2VkaURQdlU2U25aTG90cTM0ZUtMY0U1OG1XVlNBY0MtdHFSaFhUbE44U0FULTljWmJkZlp3enBLckdzWTlnaWJ0Y0lfVDdrWERwN2tHY0NibzZ0MmVEakxnc2x4LW1WaDI3TlBfQUE?oc=5) — *Google News / CNBC*
- `11/11 20:38` [Mortgage rates jump sharply higher, after easing for four straight weeks. Here's why](https://www.cnbc.com/2026/11/11/mortgage-rates-jump-sharply-higher-after-easing-for-4-straight-weeks.html) — *Finnhub / CNBC*：After falling for a month, mortgage rates jumped sharply higher on a hotter-than-expected inflation report.
- `11/11 20:27` [Consumers are showing signs of strain even as they keep spending, CNBC/NRF Retail Monitor shows](https://www.cnbc.com/2026/11/11/october-retail-sales-cnbc-nrf-2026.html) — *CNBC Economy*：Retail sales rose at their slowest pace in nearly a year in October as American consumers showed signs of strain from high prices.
- `11/11 20:19` [Morning Call Sheet: Fed rates, AI debt concerns and market rotation](https://news.google.com/rss/articles/CBMirAFBVV95cUxNanVweEZ4S2RDelJMU0F4cGRXc0VoUTEzRC1YNHd6Z0dWS0ZyZnBqWDNzN18wRjlpWmVPUkxfQXRSMURQM1FiSEc3Skd2ZlJIbUFKSW9acHdkTDhPMXg1UXR0OU5uNy1kZ2RvQU9keHhBWHJxclVseWw1ZnVRWU41UW9pSlRDMnFYbzA4N0FraVAxTExGd0VnWFVkci1Xbmd2eC1wOVZ1eHdpLS1V?oc=5) — *Google News / CNBC*
- `11/11 20:14` [Goldman adds another rate hike to Fed outlook after hot inflation data](https://news.google.com/rss/articles/CBMimwFBVV95cUxONExVbW1kNl90NWNsM3VkVjlqVUh2cE9JZzY5YzhTTzhyUEVvcS1lNDhadGlNYlJ6OHVoaG9yUFdBTE9MX0l0eEVxaWlZWjBZRjZzRk4zcjBQNnBuN0tTaWJnZzluQTZid1AxSUVYZXAteGVHRjRha3pVUS1OX1lTV3Rtcjh2TVRzMHBTaDBSUmFGWlV6Q2o0UUN3dw?oc=5) — *Google News / Yahoo Finance*
- `11/11 19:00` [Energy prices matter way more for inflation than they used to: Fed report](https://news.google.com/rss/articles/CBMiyAFBVV95cUxQLUlfbV8zR0ZSSnJNTXZoM1JKZmdEYm9MVzlNa2Vybncyb3J6RWhxZUhZQVpnWF9RZ05FSEZGRkdvekU3N3otT3o3Z1NVVnkxRlpZUlFaZXBadWY5a1RQNmRiX29SalpqTF9LZmg0cTJOLXB5V1F4QTJ4RHRrNkllU2NyUzNOUXo3RVJqQlQtZDAtdHdNRWEwb3lZTlVLQ0ZBWUU5YnY3Ul81TGVuX19tWXBncFVqLUpTX2pCbzkzb2Q1Vm1od3hKOQ?oc=5) — *Google News / Yahoo Finance*

### 科技與 AI
- `11/12 09:14` [Cramer's stance on Nvidia's stock amid reports that CEO Jensen Huang may be ousted](https://www.cnbc.com/investingclub/2026/11/11/jim-cramer-jensen-huang-nvidia-stock-advice.html) — *Finnhub / CNBC*：Jim Cramer said Wednesday night that he was saddened by a news report that indicated Nvidia's board of directors is preparing to fire Huang.
- `11/12 08:46` [Meta head of infrastructure will leave and be succeeded by former AWS cloud exec](https://www.cnbc.com/2026/11/11/meta-infrastructure-departure-aws-cloud.html) — *CNBC Tech*：Meta's head of infrastructure and co-head of engineering Santosh Janardhan will leave in 2027, with former AWS senior vice president David Brown taking over.
- `11/12 07:41` [Jim Cramer says investors need to pay attention to CoreWeave. Here's why](https://www.cnbc.com/2026/11/11/jim-cramer-coreweave-earnings-ceo-interview.html) — *CNBC Tech*：Cramer said he's becoming less skeptical of CoreWeave after strong earnings and reassuring comments from its CEO.
- `11/12 07:30` [Jim Cramer: This is the bet you're making if you buy the Anthropic IPO](https://www.cnbc.com/2026/11/11/anthropic-ipo-profit-computing-commitments.html) — *CNBC Tech*：CNBC's Jim Cramer said that Anthropic must sustain extraordinary growth to cover its mounting infrastructure costs.
- `11/12 06:46` [Jensen Huang affair in D.C.: Resignation calls and moves to tighten Nvidia China exports](https://www.cnbc.com/2026/11/11/nvidia-jensen-huang-congress-resign-ai-chips.html) — *CNBC Tech*：Huang was notably opposed to strict export controls on AI chips to China, which ruffled feathers with congressional China hawks.
- `11/12 05:43` [Hedge Funds Just Turned Against AI Stocks for the First Time Since 2023. Should You Follow Them?](https://news.google.com/rss/articles/CBMinAFBVV95cUxOTTgtR1FyeDI3MFUzRGYtdFJyUmZFcHpCVDdURF9Uak8wZGw1Z0RYX1p4V3J5aFpDS051YjFGVzI2ejlGeGNWQUh1ZnlIdnhabjBLQm5xZDJUTXRWeENDODlkTEg4Y2x4UFNfRDlWNkJFZzUyMDZOd3ZQVmlRaVJ6UGxzOFliWmtrdUotUG1zN0Q4RzV1MXV3QkQwYWg?oc=5) — *Google News / Yahoo Finance*
- `11/12 04:04` [We're adding a chipmaker to our Bullpen watchlist and downgrading another name](https://www.cnbc.com/investingclub/2026/11/11/were-adding-a-chipmaker-to-our-bullpen-watchlist-and-downgrading-another-name.html) — *Finnhub / CNBC*：Every weekday, the Investing Club releases the Homestretch; an actionable afternoon update just in time for the last hour of trading.
- `11/12 04:03` [Anthropic IPO could give AI stocks a needed spark](https://news.google.com/rss/articles/CBMidEFVX3lxTE9WcnpXY25VdUZRMy1MdVlJczZxWVVsMXRTaE1JYU9xMkVPZ3VuV1E5QWt5Y0x4Yi1yWnZxOHRFRFdzTkNTNWRkam1RVTRmX0ZoYWxaenhwRWQtNE40U2FMZEljbzRxbUZReWF4OXdnYW1FNmts?oc=5) — *Google News / Axios*
- `11/12 03:24` [3 Possible Paths Forward for AI Stocks (Bull Case, Bear Case, Base Case) -- and How to Invest for Each](https://news.google.com/rss/articles/CBMilwFBVV95cUxNaXdXcmxBbnV5YzNBdmVuUDNXZ2VsWHpzTF9ZNk1kR1ZqNEZTbFNOQ0tuV3J1ZkxVQWdHQmJkeDJvbFdMWEpOY2h4a0U1SmcydW9ZbUg3bnlXWmltdDhaRG1fUkJCbWtFa053anNSa1F2amNweU5YcF9ldW12M2tzV1lHOFRaX2xMODA5b3phRGU3dV9TbHgw?oc=5) — *Google News / Yahoo Finance*
- `11/12 02:53` [Apple's record run ignites options flurry — Watch Options Action live](https://www.cnbc.com/2026/11/11/apples-record-run-ignites-options-flurry-watch-options-action-live.html) — *Finnhub / CNBC*：Apple's resilience continues to catch the attention of options traders looking for a safe harbor in tech.
- `11/11 23:44` [CoreWeave Leads Tech Stocks In Rally—Up 21%—After Earnings Eased AI Debt Fears](https://news.google.com/rss/articles/CBMiyAFBVV95cUxNSFpTRlVaOEwwRG44Z25UNlhYWFpIRVFUS2hvcnpKRHVaejFTc09JaHduZWhVbVVQOHBfMHp5dUplMWtlUDQ5ODBRUlE3Zlo0MUFuRW4tX3BUWElIZFRoLWFfMC1Yd1BzZlJnZkNuR1JxVHI0N1ktSjJCdUNtYnh4S1dKcUJQU2xBTnhkb1hhU1RZQmVQekV6aHA5OW1uN2ZPWHp5Ny1YeUpReFNDZnNFZk9kaHpoaDRMRkN1b2NZNHJYbWRhdHh5Rw?oc=5) — *Google News / Forbes*
- `11/11 15:47` [Alibaba announces plans to list T-Head, its Nvidia chip rival](https://www.cnbc.com/2026/11/11/alibaba-pr.html) — *CNBC Finance*：Alibaba noted it had already filed confidentially on Sept. 11 to list T-Head on the Hong Kong Stock Exchange.

### 盤勢與個股
- `11/12 08:21` [Nvidia prepares to oust Jensen Huang, WSJ reports — stock sinks](https://www.cnbc.com/2026/11/11/nvidia-jensen-huang-affair-employee.html) — *CNBC*：The report comes days after Huang admitted to an affair with a woman who is believed by U.S. intelligence of being a Chinese spy.
- `11/12 06:52` [Stock Market Today, Nov. 11: Advanced Micro Devices Jumps -- and Not Only Because of CoreWeave's Results](https://news.google.com/rss/articles/CBMilAFBVV95cUxOMlVrNjhxTHlicm8yN1V4LXluY3VWdUhXeUpxNkp3ZGtqUVB2SGItYl9QVFZ6M2JkekFWWFdtY0pQcnRsbWUxaVBMSG9RRFZ1cnc1VndsVWl3d09KSE5MVUt0SUJxLTJSSHdoNFVqS3BwejNKLXk2ZE9sbW80UnBueE93LUpXZVpDSXdPUkZib1NrVDho?oc=5) — *Google News / Yahoo Finance*
- `11/12 06:45` [Stock Market Today, Nov. 11: Stock Market Today, Nov. 11: Nasdaq Surges as CoreWeave's 25% Rally Reignites the AI Trade](https://news.google.com/rss/articles/CBMilAFBVV95cUxNVFFBbktKOWpIakhjSm1YeFU0WTRwQnVYRUtrS2E2TVhwcHNMcjdVdWUwb1gyN1gzbjdLYU1Tb21hMVRESWV5WkVXQTRiTldLUmtJT1V2ZXB3X3R5dmN5bk5hNFhWRFpDQUhQUy05OWpOUVpNVG1icjNWNlRhX1Z3NWU4SFNPOUUxNkdLQ19WUHFMaW90?oc=5) — *Google News / Yahoo Finance*
- `11/12 06:23` [Stock Market Today, Nov. 11: SpaceX Shares Slip Amid Upcoming Starship Test Flight Jitters](https://news.google.com/rss/articles/CBMilAFBVV95cUxQbmpVa1pTRXRtaGlTaG94WS0zM2tTWWg3QVlXOHdOalF4Z2ZNUk1BbDZPeG1tVEdpaGpuaW9YRk5JV09mWnp0TTVHX3ZEWUJaN3B5VkxyWTNnLVp4SnVOckVoampfOVZOVzhObHlnOXc2TWRRX253UVY4ZmJ5MG1HWlR1V0U0WGMxSkNVazVIY3R4YXZs?oc=5) — *Google News / Yahoo Finance*
- `11/12 05:13` [CoreWeave beats on revenue and lifts guidance, easing AI slowdown concerns](https://www.cnbc.com/2026/11/10/coreweave-crwv-q3-earnings-report-2026.html) — *CNBC Earnings*：CoreWeave's stock has been hammered heading into the neocloud's Q3 earnings report.
- `11/12 05:11` [CoreWeave soars more than 24% for best day since April 2025 after strong Q3 report](https://www.cnbc.com/2026/11/11/coreweave-crwv-outlook-earnings-ai.html) — *CNBC Earnings*：Neoclouds such as CoreWeave, Nebius and IREN have come under scrutiny this month as investors assessed the sustainability of the AI trade.
- `11/12 03:16` [Prediction markets tied to clinical trials and drug approvals draw lawmakers' ire](https://www.cnbc.com/2026/11/11/kalsi-polymarket-prediction-market-senators-hhs-drug-approval.html) — *CNBC*：Sens. Angela Alsobrooks and Ron Wyden are asking what steps the Department of Health and Human Services has taken to mitigate risks from the markets.
- `11/12 02:24` [Bouncing Wynn Shares might be primed for the 'wheel strategy'](https://www.cnbc.com/2026/11/11/bouncing-wynn-shares-might-be-primed-for-the-wheel-strategy.html) — *Finnhub / CNBC*：From a valuation standpoint, WYNN is trading very close to its five-year historical averages across key metrics. So how should traders play this setup?
- `11/11 23:53` [CoreWeave CEO Intrator tells CNBC he has canceled his planned share sale program](https://www.cnbc.com/2026/11/11/coreweave-ceo-election-data-center-buildout.html) — *CNBC Earnings*：After CoreWeave beat on revenue and upped its forecast, CEO Mike Intrator indicated the company will be fine despite last week's Democratic midterm sweep.
- `11/11 22:13` [Jim Cramer's top 10 things to watch in the stock market Wednesday](https://www.cnbc.com/investingclub/2026/11/11/jim-cramers-top-10-things-to-watch-in-the-stock-market-wednesday.html) — *Finnhub / CNBC*：CoreWeave lifts the AI trade, and my advice to Nvidia.

### 美債與美元
- `11/12 04:04` [iShares TLT Premium Income+: Recent Treasury Auctions Point To Rangebound Yields](https://news.google.com/rss/articles/CBMingFBVV95cUxPSTdaTmFpNjNVYnNOUUxnVlg1ODZ2SUlfbnRDZ0RxXzdiUE5YMGdCSGZHZ0s0UGZfTG8xMXIzN2tJNDJVWFVUZ3puNFVYU0RvM1hrZkNSTnlXc0x2Ykp5U0phdlNsU3BsSzVJUzc1bjlUTTNFY2Z3V1J1Y0kteUxiVzRmWF91YTMxOUFsMF9FQ2Z2cjRGQ0UxX05TQVVBQQ?oc=5) — *Google News / Seeking Alpha*
- `11/12 02:38` [Gold drifts below $4,000 as high US yields fuel December Fed hike odds](https://news.google.com/rss/articles/CBMisgFBVV95cUxORktvSEZxNXNxamljb2I3R0xVcllESEloZGZuZnZNTVZlWl9Nc0EzWmZhQWhQYV9hYVE2Z0h5czJQLWx0OW12QXZwVkx0a1BocFpaMks3SWs3WU9QbmtYbzl0QUxSSndhRWc2eEZXTUNnZmdVc1FfbVRuZTNUNUNmNWR5dkFEUUVfYVp5ZkRQOEtCY3lQcFFrcWlEa3NNWGZmYnRBeXBIdUVqclVOTVF5MVVR?oc=5) — *Google News / FXStreet*
- `11/12 02:33` [Inflation Fears Keep Treasury Yields Elevated, but Is the Market Overreacting?](https://news.google.com/rss/articles/CBMivgFBVV95cUxON2Y5aVdudzNYeUtNUnhlYTQ1WG05MHFmSjc1aWNJT2lWZ2Jvd3I2U3lrbDhwUUIzc0xORDB6bW8wUF9ubENJM0RPMXAwQVRGcXV5dHBJVHNhcDNtd3o4RWhwVmRIMjREczZhVVRWcThDU1ZiTXZNVTQybUs5Q3BrMGlhTk1EZWRJalZobXRVdjdyVzVzMEhoeFVFY1gtbEx6WFFqZThPb255OGZVMEVybFNLaGVkSlJRWjlXV2V3?oc=5) — *Google News / Investing.com*
- `11/12 01:07` [Why the US Dollar's rally isn't as strong as it looks](https://news.google.com/rss/articles/CBMinwFBVV95cUxQOFJJS1lPb05uRWxTYVpQcUszUUk1X0t3Y2ZSVmJEZGRraUhqOUlTc3BLa0s4dl9ueHpUQ0d5V1Rpd2N6ak5Da05ONDVLR2pMM1lseng3ZWlreUU3cGFPOFQ4SU5NLVQ1WmY4ZGxuazRub0tRRkRZb2JJcWpVQVBKa0FjUFNBNVJnbl95cERLX19ra0c4RjJNRExGLXNYdVU?oc=5) — *Google News / FXStreet*
- `11/12 00:09` [Silver defies rising yields, holds above $52 as inflation stays hot](https://news.google.com/rss/articles/CBMitwFBVV95cUxOME4xTXBrbjE2d1ZPNFhVMkdsUWhzUk5iZzlsaUpKYVZqRkotX2ZTa2R6d1Y5TGRtakZSaEMwOXBpNVRocFMzWG8yam9OSEtZMGtMYlJuUl82XzVjQUJjWG5Oa09vdXAwSEJQTE9hSTBrVUpzeEFyVm9pejFzNUo2NS1HemtmZmNVbllQUmcwbTFiS0gyenpyU1BJQVZIZEZtQ0E2T3pzSjRjbl84X0RuR1VmOE9xRlU?oc=5) — *Google News / FXStreet*
- `11/11 22:29` [US inflation surprise steadies pound sterling to US dollar exchange rate as buyers defend support](https://news.google.com/rss/articles/CBMijgFBVV95cUxQdFgteHZiTWJwNzdram5MZXljZGh2bWtMQkJtaXlnVW56SnNNSTNhTFBQNWhtNkVhZ1gwb0F2MFlJRGItemxxOTMzQ2RQbjNaV1J4WTlqYWFJYWRSVlVqenpVU3RLR0xZaVRVZ0xoQW1qd056NFUyczFLSmdsQmxXQnlJX0dqalpJak1Dc1Zn?oc=5) — *Google News / Traders Union*
- `11/11 20:19` [Japanese Yen weakens toward 160 as US inflation bolsters Fed hike bets](https://news.google.com/rss/articles/CBMiswFBVV95cUxQUlNIY2JSZ1htZzNjb1IxbUxreTljZFQ5RXM4OHAyM1JBZVFnT24zbDN0UWR6T3kyNUVSTjlrT21xbkt2SUdSNlB5OW41ck5zMzFib0VQMzFSRnp4N0l1S0tHZFZKc1JxMzNoRTkzZk1UUzBQYVNUQTZOWF9uNUl0N3N6S3pibHdEX0FPbnluVGIwaWJta2RMWjlkWXVOb1ZNdy15cUhiakxIVnBXekJlTnAxaw?oc=5) — *Google News / FXStreet*
- `11/11 16:04` [U.S. dollar weakens sharply against the Japanese yen after market interventions](https://news.google.com/rss/articles/CBMiXkFVX3lxTE5LaVZOS1NHWXlXUkpDVTdYWFMzZ09xd2xvbm85M05NLTFJNEJleXhBSVZmQlVMdUJQVkFNUXZxeUx2Q0d3akdzOFFvSXp1ZGZfdXJJSnpDR3ZHLXdZV3c?oc=5) — *Google News / 朝日新聞*

### 黃金
- `11/12 04:46` [Gold prices fall below $4,000 as analysts warn of further losses toward $3,900](https://news.google.com/rss/articles/CBMitAFBVV95cUxQdTg3THhtUnZVa1FYX29ZaFN4RlBkbHFWbVk1S2JwWGl3N2tlRFg3TjFPV3ktU29fMGZvRzZ1bkZsZzYyOU0zT3FXTTFLeEJGNDlGN0hYdDBNMENPaDNYY0ZDOWk3dUtJaGJkX3NkN3FBdU94Q1Vhb05BUElqMTF0TDB6cEFYaG1KWnNBOW1GTGdFRzg2dThhR0JTODNUancwNDdKdk5Telc1TzV3dkh1OVIzaVg?oc=5) — *Google News / Kitco*
- `11/12 03:46` [Gold miners enjoy strong Q2 despite lower gold price, higher expenses and operational disruptions – WGC’s Blagden](https://news.google.com/rss/articles/CBMiugFBVV95cUxNLXhFSnNuR3lYOWJfZVZNZVlodWJlZmdzcU1fYVdNY3FDMHFFVmx1NE00eHVCNmVzbVQ4X0dIV1dtM1lkaVdSamJjZG8xeXdDV1VBVHZMTkZwX0tmTFVNa091c2xsSVkzTEg5VkFGVFJHM2pOaWhOQURYYkxyeWJyN0pSXzhiLXB4b3UwUko1eGNkaUd6MEhUZFpEQi1xd1JBSkhmWV9DSGFqVk9nQlpiTUYtUG8zeE9iNEE?oc=5) — *Google News / Kitco*
- `11/11 22:59` [Gold prices find solid support at $4,000 as Standard Chartered sees rally to $4,650 by year-end](https://news.google.com/rss/articles/CBMivgFBVV95cUxQUllaZWRQc1poVDQzVEh2N3BDSmhHd0ZDMnE0WU8zZi1YUEI3TTg0Yk1LTWJ1UlhzNnlIY0ZRYWdzZkhicGZvRWxBTk9uazE3UERhdE15MUQxR25oOGdkVS1UTEc4aW8yY2U1WDNvdTFBMzNjNmJoeldUOW1UOEVKNzU0UUhZaVprR0dFcHRWV19SR05fV0czM3FkdWdhbFQ0eXBtay1SRi1laFMzRE9RT25Bcy1GMlRHSlhGWU5n?oc=5) — *Google News / Kitco*
- `11/11 20:58` [Gold gains after steep drop, but hawkish central bank signals weigh](https://news.google.com/rss/articles/CBMiswFBVV95cUxPVENmb2FBaGQxQWZacy1HMmpGNk52N1JUV1VBaW54UFFHTG9wOTVSeVo1MFF6MVVXMklhWGkyaHRWbk5JLTRwYXNBRFJKRG5JVGMxYjRjVHQ3aDlURlFFdWxldHZEdEFNUzdnb3ZDakhPY2dfMGlOdnIzR01mdk43M0hSRENMaGNLY2lkWGlqSndvTVNxZkI0Ql82ZFdxVnItUTgyWk82cUhiQXpiczM3NGJXOA?oc=5) — *Google News / Kitco*
- `11/11 20:11` [Gold price today, Wednesday, November 11, 2026: Gold prices fall back as inflation concerns broaden](https://news.google.com/rss/articles/CBMigAJBVV95cUxPa2ZWdnE1NlJaSFUzNWdvMnF5NXpwQzlKUFZ5MUczdFU1ZTNGZkRlRzNoUTFweTZ1aEVRVzR1cHEwWjJ4VS10R2dQYWJiaXk4a2F5UDZFLU1PX2JINGh0dnNBQ0Z1a1I4WXRiTTV5dFhPRWZaR1YxYWxJYzdoVTNCS0dWQ3pjZmJfMmlyX0ZFMk1rNXZEeGxRdWVXWlZmelR0RDA1WGtrbjZRU1FJMmMzaTI3cnJYaXBDVjF5cHJwNjZtTTY2dkpoLWprT2dOb1RENGhld3RBSjE4TU95YWNReGczOWZmYmxYLUc5cWlNQVVmOE9DRXJHRWJIbnpUa3Et?oc=5) — *Google News / Yahoo Finance*
- `11/11 18:13` [Gold Prices Recover Above $4,040 as Investors Assess Federal Reserve Rate Hike Expectations](https://news.google.com/rss/articles/CBMingFBVV95cUxPbE9pY2xWMlFnbjZTNnNOU04tbE1fZzUzQ2xfS3JaUUZ3ZnM4bFpGLUM1QXJ1UEVSOVlzc3ctUUtkYVowR01lQTNEUExDMzlNTU53bnVYN0ZLTTB5ZXpzVE9rcDM3ME9ncTNYM3hoa21UY0trOG9IWHlOVzhHOFNONGtYOHF3WjZ2RXF4S0k5eDZpUEZWVU1IX1V5dGNzQQ?oc=5) — *Google News / Yahoo Finance*

### 原油
- `11/12 09:00` [Who Will Buy Sable’s Oil if California Keeps Closing Refineries?](https://oilprice.com/Energy/Crude-Oil/Who-Will-Buy-Sables-Oil-if-California-Keeps-Closing-Refineries.html) — *OilPrice*：Situated off California’s Santa Barbara coast, the Santa Ynez project operated by Sable Offshore has become a litmus test of President Trump’s determination to revive oil production in a state pursuin…
- `11/12 03:30` [Murphy Oil Draws Boycott Call Over Israeli Gas Licenses](https://oilprice.com/Latest-Energy-News/World-News/Murphy-Oil-Draws-Boycott-Call-Over-Israeli-Gas-Licenses.html) — *OilPrice*：Pro-Palestinian activists who pressured Eni and Dana Petroleum to quit an Israeli offshore gas venture are now going after Murphy Oil, the Houston-based company that replaced them on Monday. The Boyco…
- `11/12 02:30` [Vitol’s Venezuela Oil Windfall Traced to Trader’s Ties to JD Vance](https://oilprice.com/Latest-Energy-News/World-News/Vitols-Venezuela-Oil-Windfall-Traced-to-Traders-Ties-to-JD-Vance.html) — *OilPrice*：U.S. Energy Secretary Chris Wright called Vitol gasoline trader John Addison, an early financial backer of Vice President JD Vance, on January 5, two days after the U.S. operation that removed Nicolás…
- `11/12 02:00` [Crude Is Back at 80% in the Gulf, but Refined Products Are Stuck at 35%](https://oilprice.com/Energy/Energy-General/Crude-Is-Back-at-80-in-the-Gulf-but-Refined-Products-Are-Stuck-at-35.html) — *OilPrice*：The story in the Strait of Hormuz has been that Middle Eastern crude exports have recovered to about 80% to 85% of pre-war levels, including pipeline shipments, while refined-product exports remain mu…
- `11/12 00:59` [Bessent moved to military base amid 'growing threats' from Iran](https://www.cnbc.com/2026/11/11/us-iran-war-trump-hormuz-bessent-threats.html) — *CNBC Energy*：Bessent's role leading Operation Economic Outcast has made him one of the Trump administration's top targets of Iranian threats, The New York Times reported.
- `11/11 22:02` [Current price of oil as of November 11, 2026](https://news.google.com/rss/articles/CBMiYkFVX3lxTFB5azdwdTFySFNNNk8zcWRlWUgybnZGX0NsN2hadVppWlFlSVpzQjZtaXpEaXhCaXM4NFZzUGY4VnF2dFJYTmVrUmx2VVo1cHFXUTNDYnVFVzk5TFdrTnFFNTRn?oc=5) — *Google News / Fortune*
- `11/11 19:24` [Oil prices touched their highest level in 4 years. What does it mean for shoppers?](https://news.google.com/rss/articles/CBMingFBVV95cUxPYmQwUV9tR2hlRnZLSjQ5dlJqNWQzQVFiaWM3MmpsbzluTi1idEg4VWhEcjVLRVFFNXpldG1jd2xwU3VuTDQ1ZTFaSkNYeGc1RUx2QVczZEpjcjBsQkhGUGdnSDlnQVlkM1BQZ0VYSUhUcmlfc3lRZmxlRFlxU1ZuUUV0Ykg5cFdRWTJBeFBSbDNHV3FRbXl6RmFSNmN2UdIBowFBVV95cUxPNUx3b3VfM1Q5blJiS0ZWR0o4TkczSmEyczhCX2NRSWZvZll0Vlp5T0t3NEZCV2JRdTI4UTlZQVNQMUJPMUxRc0hEZlljWUJYeXRxbmUxdGVJQ1M3aFk2MzFBY05Zd0NjVVdvVkhTcWN2T1BqODh6bW9tRjV0Y2p2ZWFkb0RsNVAtRkxDbDJzMjI0b1Z0ZnA3OF85QzBZLUxCTkw0?oc=5) — *Google News / ABC News - Breaking News, Latest News and Videos*
- `11/11 18:00` [Trump’s peaceful mood doesn’t fully explain why energy prices are plunging](https://news.google.com/rss/articles/CBMia0FVX3lxTE5GODFZUEVzUkNEYmh3WjQ3VVFiNGZ4TWlFY2JKVzMtUzBaTy0waDJVUU9sLV82MnlMVjBTWnhzZW1VM3FoR2JJTVFVVU1QQzZVYnB4dmpJdmxZUlhzYnJlcmc4T0RXb2xfTjRR?oc=5) — *Google News / CNN*

### 中文（鉅亨網美股）
- `11/12 10:39` [黃仁勳再爆與副總裁有染！隨扈扣留目擊者 要求簽保密協議](https://news.cnyes.com/news/id/6660069) — *鉅亨網*：《紐約時報》引述多名曾在輝達任職、但未具名人士報導，黃仁勳曾與輝達醫療事業副總裁Kimberly Powell發展關係。 今年3月，在輝達一場大型會議期間，黃仁勳被拍到與Powell在加州聖荷西一家飯店內親密互動的畫面。 事發後，黃仁勳的隨扈扣留了目擊者，並要求對方簽署保密協議。
- `11/12 10:30` [Anthropic上市前永續合約單日漲8% 交易者估值推升至1.84兆美元](https://news.cnyes.com/news/id/6660091) — *鉅亨網*：Anthropic上市前永續合約周三漲8%，隱含估值約1.84兆美元。 公司擬以每股70至78美元發行8.25億股，若以價格上限計，募資約644億美元、估值約1.4兆美元。 彭博稱認購需求達數倍，承銷銀行考慮將估值提高至1.5兆美元以上；分析師對定價與獲利前景看法分歧。
- `11/12 10:22` [網友拱延攬黃仁勳 馬斯克：那是我的榮幸](https://news.cnyes.com/news/id/6660097) — *鉅亨網*
- `11/12 10:22` [沒買輝達也受影響！標普500權重近8% 一份財報牽動全美退休金](https://news.cnyes.com/news/id/6660168) — *鉅亨網*：輝達11月17日公布2027會計年度第三季財報，市場聚焦AI需求與資料中心資本支出。 輝達占SPY投資組合7.81%、標普500權重近8%，股價每波動10%約影響指數0.78個百分點。 董事會已就黃仁勳遭外國情報鎖定報導啟動獨立調查，稱目前未見國安或商業機密外洩。
- `11/12 10:21` [私生活風暴燒向國安議題！美眾院外交、情報委員會出手 準備傳喚黃仁勳](https://news.cnyes.com/news/id/6660137) — *鉅亨網*：共和黨眾議員擬向黃仁勳發傳票，要求就婚外情醜聞與輝達安全規程出席閉門說明會。 Moolenaar、Mast、Crawford上週已聯名要求黃仁勳就晶片製造商安全規程作證，週三前須回應。 醜聞衝擊輝達股價兩日跌約10%，也加速國會將晶片出口管制納入《國防授權法》。
- `11/12 10:20` [WSJ：黃仁勳婚外情風波不斷 衝擊在台形象](https://news.cnyes.com/news/id/6659957) — *鉅亨網*：黃仁勳婚外情風波延燒台灣，黃樂被指涉中國間諜；內政部長劉世芳稱將禁止她再入境，國安機關調查中。 事件從名人私生活轉為國安議題，因台灣近年查辦中國間諜案、半導體又是「矽盾」核心，輿論格外敏感。 蔣萬安稱市府與輝達溝通順暢，台灣總部案將按原計畫推進；輝達與台積電等台灣供應鏈合作仍受關注。
- `11/12 09:56` [黃仁勳小三人在哪？彭博：黃樂目前人在北京](https://news.cnyes.com/news/id/6660081) — *鉅亨網*：黃樂目前在安全官員知情的情況下居住於北京。 知情人士表示，目前不清楚她何時以及為何前往北京，也不清楚她可能在從事什麼活動。 自從黃仁勳承認婚外情以來，外界一直在關注黃樂的行蹤。
- `11/12 09:40` [算力訂單能否變現成關鍵！Sixth Street執行長示警：約2000億美元晶片閒置 AI基建融資風險升高](https://news.cnyes.com/news/id/6659934) — *鉅亨網*：Sixth Street執行長沃克斯曼稱，約2000億美元晶片因資料中心未完工而閒置，AI基礎設施投資規模前所未見。 他警告AI信貸擴張過快、投機氾濫，若資料中心延誤與新晶片折舊壓低租金，專案現金流恐出現缺口。 市場已見ING擬售約5億美元資料中心貸款，高盛稱AI投資正進入檢驗現金流與償債能力階段。


8. Pre-market and early session（盤前與早盤）

In [ ]:
# 只需要先跑第 1 格。
# ・台北傍晚～深夜執行：即時看當天美股的盤前與早盤。
# ・隔天清晨寫報告時執行：自動回看剛收盤那個交易日的盤前與早盤（寫「盤前 → 早盤 → 終場」用）。
SESSION_DATE = None                 # None＝自動；要回看某一天就填美東日期，例如 "2026-11-11"
PRE_WINDOW = ("04:00", "09:30")     # 盤前（美東）＝台北 17:00–22:30（冬令）／16:00–21:30（夏令）
EARLY_WINDOW = ("09:30", "12:00")   # 早盤（美東）＝台北 22:30–01:00（冬令）／21:30–00:00（夏令）
FUT = {"ES=F": "標普500期貨", "NQ=F": "那斯達克100期貨", "YM=F": "道瓊期貨"}
EARLY_IDX = {s: n for s, n in INDICES.items() if s != "^VIX"}
PRE_PATH = ["04:00", "06:00", "08:00", "08:30", "08:45", "09:30"]     # 08:30 是多數美國經濟數據公布時間
EARLY_PATH = ["09:35", "10:00", "10:30", "11:00", "11:30", "12:00"]   # 10:00 也常有數據（ISM、消費者信心）
# Finnhub 個股新聞（每檔 1 次請求）：{代號: 標題關鍵字}，標題沒提到公司的略過（Finnhub 會把不相干的新聞稿也掛在大型股底下）
STOCK_NEWS = {"NVDA": ("nvidia",), "AAPL": ("apple",), "MSFT": ("microsoft",), "AMZN": ("amazon", "aws"),
              "GOOGL": ("google", "alphabet"), "META": ("meta",), "TSLA": ("tesla",), "TSM": ("tsmc", "taiwan semi"),
              "AMD": ("amd", "advanced micro"), "AVGO": ("broadcom",), "CRWV": ("coreweave",)}
STOCK_NEWS_PER_WINDOW = 1           # 每檔每個時段最多幾則
NEWS_PER_TOPIC = {"聯準會官員談話": 5, "總經與聯準會": 5, "科技與 AI": 10, "其他": 3}   # 綜合新聞每個時段、每個主題最多幾
CNBC_FEEDS = {"CNBC": 100003114, "CNBC Economy": 20910258, "CNBC Earnings": 15839135, "CNBC Finance": 10000664,
              "CNBC Tech": 19854910, "CNBC US": 15837362, "CNBC World": 100727362, "CNBC Energy": 19836768}
NEWS_FEEDS = {**{n: f"https://www.cnbc.com/id/{fid}/device/rss/rss.html" for n, fid in CNBC_FEEDS.items()},
              # Fed 官員談話、利率政策、科技股：CNBC 沒有專屬的 RSS，用 Google News 補（只有最近 24 小時）
              "Google News（Fed 官員）": GN_FED, "Google News（利率政策）": GN_POLICY, "Google News（科技股）": GN_TECH}
FEED_TOPIC = {"CNBC Economy": "總經與聯準會", "Google News（Fed 官員）": "總經與聯準會", "Google News（利率政策）": "總經與聯準會",
              "CNBC Tech": "科技與 AI", "Google News（科技股）": "科技與 AI"}   # 這幾個來源直接歸到主題，其餘看標題關鍵字


def bars(sym):
    """近 5 天的 5 分 K（美東時間）。期貨幾乎 24 小時都有，指數只有 09:30–16:00"""
    h = yf.Ticker(sym).history(period="5d", interval="5m", prepost=False)
    if h is None or h.empty:
        raise ValueError("Yahoo 沒有回傳 5 分 K")
    return h[["Open", "High", "Low", "Close"]].dropna().tz_convert(NY)


def finnhub_items(label, data, words=()):
    return [(datetime.fromtimestamp(d["datetime"], timezone.utc), clean(d.get("headline"), 300), d.get("url", ""),
             f"{label} / {d.get('source', '')}", clean(d.get("summary")))
            for d in data if d.get("datetime") and d.get("headline") and not blocked(d.get("url"), d.get("source"))
            and (not words or re.search(r"\b(" + "|".join(words) + r")", d["headline"], re.I))]


def official_closes(symbols):
    """指數的官方日收盤（算「前收」用）；一次請求"""
    df = yf.download(symbols, period="10d", interval="1d", auto_adjust=False, progress=False)["Close"]
    return df.to_frame(symbols[0]) if isinstance(df, pd.Series) else df


now = pd.Timestamp.now(tz=NY)
if SESSION_DATE:
    S = pd.Timestamp(SESSION_DATE).date()
else:
    S = now.date()
    while S.weekday() >= 5:            # 週末：回看週五
        S -= timedelta(days=1)

# 行情（Yahoo）與新聞（CNBC、Finnhub）一次同時抓
tasks = {("bar", s): ("yahoo", partial(bars, s)) for s in [*FUT, *EARLY_IDX]}
tasks.update({("news", n): ("news", partial(rss, n.split("（")[0], url)) for n, url in NEWS_FEEDS.items()})
if FINNHUB_API_KEY:
    tasks["news", "Finnhub"] = ("finnhub", lambda: finnhub_items("Finnhub", finnhub("news", category="general")))
    for sym, words in STOCK_NEWS.items():
        tasks["stock", sym] = ("finnhub", partial(
            lambda sym, words: finnhub_items(sym, company_news(sym, S), words), sym, words))
tasks["daily", "指數日收盤"] = ("yahoo", partial(official_closes, list(EARLY_IDX)))
res = gather(tasks, {"yahoo": YAHOO_WORKERS, "news": len(NEWS_FEEDS), "finnhub": FINNHUB_WORKERS})
B = {k[1]: v[v.index <= now] for k, (v, err) in res.items() if k[0] == "bar" and v is not None}
DAILY = res["daily", "指數日收盤"][0]
errs = [f"{k[1]}：{err}" for k, (v, err) in res.items() if err]

cash_days = sorted({d for s in EARLY_IDX if s in B for d in B[s].index.date if d < S})
P = cash_days[-1] if cash_days else S - timedelta(days=3 if S.weekday() == 0 else 1)    # 前一個美股交易日
T = lambda d, hm: pd.Timestamp(f"{d} {hm}", tz=NY)
pre0, open_t, early1, ref_t = T(S, PRE_WINDOW[0]), T(S, PRE_WINDOW[1]), T(S, EARLY_WINDOW[1]), T(P, "16:00")
tw = lambda t: f"{t.tz_convert(TW):%H:%M}"
num = lambda v: f"{v:,.2f}"


def pct_at(df, ref, lo, t):
    """t 那一刻的價格（t 之前最後一根 5 分 K 的收盤）相對 ref 的漲跌；時間還沒到就留空"""
    v = df[(df.index >= lo) & (df.index < t)]["Close"]
    return "—" if t > now or v.empty else f"{(v.iloc[-1] / ref - 1) * 100:+.2f}%"


def path_table(names, rows, times):
    head = "| 台北時間 | " + " | ".join(f"{tw(T(S, t))}" for t in times) + " |"
    return "\n".join([head, "|:---|" + "---:|" * len(times)] + [f"| {names[s]} | " + " | ".join(r) + " |" for s, r in rows])


stamp = f"執行時間：台北 {now.tz_convert(TW):%m/%d %H:%M}＝美東 {now:%m/%d %H:%M}"
lines = [f"## 美股 {S:%m/%d}（{'一二三四五六日'[S.weekday()]}）盤前與早盤", f"{stamp}｜前一交易日：{P:%m/%d}", ""]

# ---- 盤前：指數期貨 ----
state = "尚未開始" if now < pre0 else f"進行中，數字截至執行時間" if now < open_t else "已結束，以下是開盤前最後價"
lines += [f"### 盤前｜指數期貨（台北 {tw(pre0)}–{tw(open_t)}，{state}）",
          f"漲跌基準：前一交易日美股收盤時（美東 {P:%m/%d} 16:00）的期貨價格，和現貨指數的漲跌可以直接比較。", "",
          "| 期貨 | 基準 | 開盤前 / 最新 | 漲跌 | 盤前高（台北時間） | 盤前低（台北時間） |", "|:---|---:|---:|---:|---:|---:|"]
rows = []
for s, name in FUT.items():
    f = B.get(s)
    base = f[f.index < ref_t]["Close"] if f is not None else []
    seg = f[(f.index >= ref_t) & (f.index < open_t)] if f is not None else []
    if not len(base) or not len(seg):
        lines.append(f"| {name} | — | — | — | — | — |")
        continue
    ref, last, w = base.iloc[-1], seg["Close"].iloc[-1], seg[seg.index >= pre0]
    hi = f"{num(w['High'].max())}（{tw(w['High'].idxmax())}）" if len(w) else "—"
    lo = f"{num(w['Low'].min())}（{tw(w['Low'].idxmin())}）" if len(w) else "—"
    lines.append(f"| {name}（{s}） | {num(ref)} | {num(last)} | {last - ref:+,.2f}（{(last / ref - 1) * 100:+.2f}%） | {hi} | {lo} |")
    rows.append((s, [pct_at(f, ref, ref_t, T(S, t)) for t in PRE_PATH]))
if any(v != "—" for _, r in rows for v in r):
    lines += ["", "盤前走勢（各時點相對基準的漲跌；台北時間）：", "", path_table(FUT, rows, PRE_PATH)]

# ---- 早盤：主要指數 5 分 K ----
state = f"尚未開盤（台北 {tw(open_t)} 開盤）" if now < open_t else "進行中，數字截至執行時間" if now < early1 else "已結束"
lines += ["", f"### 早盤｜主要指數（台北 {tw(open_t)}–{tw(early1)}，{state}）", ""]
rows, approx = [], []
if now >= open_t:
    lines += ["| 指數 | 前收 | 開盤（跳空） | 早盤高（台北時間） | 早盤低（台北時間） | 早盤結束 / 最新 |", "|:---|---:|---:|---:|---:|---:|"]
    for s, name in EARLY_IDX.items():
        d = B.get(s)
        prev = d[d.index.date == P]["Close"] if d is not None else []
        day = d[(d.index >= open_t) & (d.index < early1)] if d is not None else []
        if not len(prev) or not len(day):
            lines.append(f"| {name} | — | — | — | — | — |")
            continue
        off = DAILY[s].dropna() if DAILY is not None and s in DAILY else pd.Series(dtype=float)
        off = off[off.index.date == P]
        if off.empty:
            approx.append(name)                    # 抓不到官方收盤：退而用前一交易日最後一根 5 分 K
        ref, o, last = (off.iloc[-1] if len(off) else prev.iloc[-1]), day["Open"].iloc[0], day["Close"].iloc[-1]
        lines.append(f"| {name} | {num(ref)} | {num(o)}（{(o / ref - 1) * 100:+.2f}%） | "
                     f"{num(day['High'].max())}（{tw(day['High'].idxmax())}，{(day['High'].max() / ref - 1) * 100:+.2f}%） | "
                     f"{num(day['Low'].min())}（{tw(day['Low'].idxmin())}，{(day['Low'].min() / ref - 1) * 100:+.2f}%） | "
                     f"{num(last)}（{(last / ref - 1) * 100:+.2f}%） |")
        rows.append((s, [pct_at(d, ref, open_t, T(S, t)) for t in EARLY_PATH]))
    if rows:
        lines += ["", "早盤走勢（各時點相對前收的漲跌；台北時間）：", "", path_table(EARLY_IDX, rows, EARLY_PATH),
                  "", "前收是官方收盤價；高低點時間是該根 5 分 K 的起始時間。"
                  + (f"⚠️ {'、'.join(approx)} 抓不到官方收盤，前收改用前一交易日最後一根 5 分 K（有極小誤差）。" if approx else "")]

# ---- 新聞：依時間分到盤前、早盤 ----
seen, general, stock, forced = set(), [], {}, {}
for k, (items, err) in res.items():
    if k[0] in ("news", "stock"):
        for it in items or []:
            key = re.sub(r"\W+", "", it[1].lower())
            if k[1] in FEED_TOPIC:
                forced.setdefault(key, FEED_TOPIC[k[1]])
            if it[0] and key and key not in seen and not blocked(it[2]) and not junk(it[1], it[3]):
                seen.add(key)
                (general if k[0] == "news" else stock.setdefault(k[1], [])).append(it)


def news_lines(items):
    return [f"- `{t.astimezone(TW):%m/%d %H:%M}` [{title.replace('[', '(').replace(']', ')')}]({link}) — *{src}*"
            + (f"：{summary}" if summary else "") for t, title, link, src, summary in items]


for label, lo, hi in (("盤前", pre0, open_t), ("早盤", open_t, early1)):
    lines += ["", f"### {label}新聞（台北 {lo.tz_convert(TW):%m/%d %H:%M}–{hi.tz_convert(TW):%H:%M}）"]
    if now < lo:
        lines.append("- （時段尚未開始）")
        continue
    # 這個時段的綜合新聞分成四個主題；每個主題先取 CNBC，再取其他主要財經媒體，去掉重複轉載後依時間排列
    rank = lambda it: (0 if "/" not in it[3] else 1 if preferred(it[3]) else 2, -it[0].timestamp())
    g = dedupe(sorted((it for it in general if lo <= it[0] < hi), key=rank))
    topic = lambda it: "聯準會官員談話" if SPEAK_RE.search(it[1]) else forced.get(re.sub(r"\W+", "", it[1].lower())) or (
        "總經與聯準會" if MACRO_RE.search(it[1]) else "科技與 AI" if TECH_RE.search(it[1]) else "其他")
    for name, n in NEWS_PER_TOPIC.items():
        sel = sorted([it for it in g if topic(it) == name][:n])
        lines += ["", f"**{name}**"] + (news_lines(sel) or ["- （無）"])
    picks = [it for sym in STOCK_NEWS for it in sorted(i for i in stock.get(sym, []) if lo <= i[0] < hi)[-STOCK_NEWS_PER_WINDOW:]]
    if picks:
        lines += ["", f"**{label}個股新聞（Finnhub）**"] + news_lines(picks)
if not FINNHUB_API_KEY:
    lines += ["", "（未設定 FINNHUB_API_KEY：只有 CNBC 新聞）"]
if errs:
    lines += ["", "⚠️ 無法取得：" + "、".join(errs)]
display(Markdown("\n".join(lines)))

## 美股 11/11（三）盤前與早盤
執行時間：台北 11/12 10:40＝美東 11/11 21:40｜前一交易日：11/10

### 盤前｜指數期貨（台北 17:00–22:30，已結束，以下是開盤前最後價）
漲跌基準：前一交易日美股收盤時（美東 11/10 16:00）的期貨價格，和現貨指數的漲跌可以直接比較。

| 期貨 | 基準 | 開盤前 / 最新 | 漲跌 | 盤前高（台北時間） | 盤前低（台北時間） |
|:---|---:|---:|---:|---:|---:|
| 標普500期貨（ES=F） | 7,544.75 | 7,612.75 | +68.00（+0.90%） | 7,620.00（21:15） | 7,587.25（17:15） |
| 那斯達克100期貨（NQ=F） | 27,835.25 | 28,352.00 | +516.75（+1.86%） | 28,406.25（21:25） | 28,224.50（17:15） |
| 道瓊期貨（YM=F） | 52,317.00 | 52,424.00 | +107.00（+0.20%） | 52,501.00（21:55） | 52,259.00（18:10） |

盤前走勢（各時點相對基準的漲跌；台北時間）：

| 台北時間 | 17:00 | 19:00 | 21:00 | 21:30 | 21:45 | 22:30 |
|:---|---:|---:|---:|---:|---:|---:|
| 標普500期貨 | +0.71% | +0.77% | +0.85% | +0.96% | +0.92% | +0.90% |
| 那斯達克100期貨 | +1.62% | +1.76% | +1.87% | +1.96% | +1.87% | +1.86% |
| 道瓊期貨 | +0.23% | +0.03% | +0.17% | +0.28% | +0.27% | +0.20% |

### 早盤｜主要指數（台北 22:30–01:00，已結束）

| 指數 | 前收 | 開盤（跳空） | 早盤高（台北時間） | 早盤低（台北時間） | 早盤結束 / 最新 |
|:---|---:|---:|---:|---:|---:|
| 道瓊 | 52,231.89 | 52,196.35（-0.07%） | 52,338.10（22:30，+0.20%） | 52,055.14（23:20，-0.34%） | 52,306.98（+0.14%） |
| 標普 | 7,520.93 | 7,586.65（+0.87%） | 7,624.70（00:55，+1.38%） | 7,571.88（22:45，+0.68%） | 7,623.70（+1.37%） |
| 納指 | 25,065.87 | 25,452.79（+1.54%） | 25,586.82（00:55，+2.08%） | 25,327.23（22:45，+1.04%） | 25,582.40（+2.06%） |
| 費半 | 9,998.33 | 10,469.04（+4.71%） | 10,745.82（00:55，+7.48%） | 10,287.84（22:45，+2.90%） | 10,739.54（+7.41%） |

早盤走勢（各時點相對前收的漲跌；台北時間）：

| 台北時間 | 22:35 | 23:00 | 23:30 | 00:00 | 00:30 | 01:00 |
|:---|---:|---:|---:|---:|---:|---:|
| 道瓊 | -0.10% | -0.04% | -0.26% | -0.12% | +0.04% | +0.14% |
| 標普 | +0.90% | +0.87% | +0.91% | +1.18% | +1.26% | +1.37% |
| 納指 | +1.56% | +1.30% | +1.36% | +1.89% | +1.95% | +2.06% |
| 費半 | +4.19% | +4.13% | +5.21% | +6.65% | +6.84% | +7.41% |

前收是官方收盤價；高低點時間是該根 5 分 K 的起始時間。

### 盤前新聞（台北 11/11 17:00–22:30）

**聯準會官員談話**
- `11/11 20:23` [Fed’s Paulson backs somewhat restrictive policy, keeps gradual rate rises on table amid sticky inflation](https://news.google.com/rss/articles/CBMi4gFBVV95cUxNT2pvSGVHR3dEbi1tOTN4RWxrYVFlTmNZS1Z3ZGQ5SjVVbUxZQzRvR2tMODFLNG1rcUJaMlgxajlTVnZoYW1acGZ1UmVpRXhOdkltWTUxMWZTMW9DTmc5MmdHc1pxaER5ZG5fT29SdHh2VlVlVnhmTzBHalJvUmdxTEF5T29tT1YtSXR3TTZhc251MkxLeUs1VlZHWFJwR09ZWERNUjVYWmphLWJlQzhqeHlHcnpqMm9Ub1BtUWc3ZVViTExvZmRoNGRxS2lNU211MHN6Q1hWQmRrVVEya0ZkdE5R?oc=5) — *Google News / VT Markets*
- `11/11 21:00` [Lenders in Europe and Latin America Report Results as a Fed Governor Leans Toward a Hike](https://news.google.com/rss/articles/CBMi3AFBVV95cUxPYTZyRGw5dklDbVlPV2hOcWdkYTd6VjZQWDFlSjlDVS1nTGtpeGlXT0ZoZkVySzZIOF96bU4zX0FLY3NmMF84Z3ItYVRqRjMwakQtMUhfeldXYUVpR1MtMHp2enJjMnlFZTdLdnJueHhVbVhNTG8zR1BHSDFLeHNPcTNLc2Nkay04QXU1YnoyZHozaTd6QnJKU1hTRkZGOTkxMFpTWUZPRHlMYW9weThEZVVTTVIxNGhMbjl1bi1jeXJoVDIyWGVQTW9Nc3AxNFMyVnVFWGt4eDhKUVll?oc=5) — *Google News / PR Newswire*
- `11/11 21:02` [As higher prices seep into broader economy, Fed officials consider faster hikes](https://news.google.com/rss/articles/CBMinAFBVV95cUxOS3ZpVWRoY3RSb3kwZlFESnhfbnluX0stOC01UWxwckhXRTZVdkxXS3c0WnZFUW03QjFmVlRzUkc3YUpZeU15d29vYjQxdFpjLVpiR0FKMDY0em1paFlMaWxFaG81MFBpVnFjd2ZhdGs4NUpVbF9rTG52bVlKc09BT0ZqamxubGExMERXcHRqNko0VENJSTZkb3JWVG0?oc=5) — *Google News / AP News*
- `11/11 21:51` [Chicago Fed chief calls October inflation 'borderline disturbing'](https://news.google.com/rss/articles/CBMiigFBVV95cUxQMmxFNFRTLWJPdk5GMEhGSkU3V0JQOXRCdlczbl90cmU1aGVjVm9sMzcydGM1RHR3Q0wyOU1IaC16dzNwTDNpWWxmWTcteGQ0clVxM3NQT2UwMlphRUNXcl9Ick1BeVBtaTFSUVdjTnZMTlJNZnQ5WVVTcjJ4ODAwQl9TVlpNY1pHalE?oc=5) — *Google News / Quartz*
- `11/11 22:07` [Fed Chair Kevin Warsh Reports Up to $900,000 in Pet-Care Investments](https://news.google.com/rss/articles/CBMiqAFBVV95cUxPR2pYNFVkWHdXbzVpREpTZUN2Q0x3MktneUNyMnNHUVRuc0NvakJsNUZXeUdGTnc0akxfaTVFQnNYTnNyX2VhcUlNXzh2M2ZyUlh2RnJPTjVtLVFRR0g3VGNlSHBjYk0wZnJVZ0ZVelgzZWlieHlDREZ5WHU5c0FjLTZGdVlwLUltYVQ4dVF6emRuUkh3WGVIaE1pS3pGUTE2dVhyclFyUzA?oc=5) — *Google News / Quiver Quantitative*

**總經與聯準會**
- `11/11 19:57` [What is fuelling the rally in the US Dollar?](https://news.google.com/rss/articles/CBMixAFBVV95cUxPTjA3SkU0M2h0TlNxclYtNmNVSEg3SExBN2doLTlyNUctV1UyNkE0RkhBUy1MRF9kbFR0UXFqTVcyX1FjMG9Hd0JZa2xjZlZPRHdqbUdaZWlhSW1KbWh0Sjg5VzFBcG1wRzU0SEw3V2pmendFSGR0Z1M2TGJ2LXFXdzNKSXE1WEU0aVNrRTNoTXYxMnNDNHdYUm96SEJLLWZHUXY3OEp0ZEdrWUU2aUZmcnEyQ01rZnNkOHFxOElld3pQOE9S?oc=5) — *Google News / FXStreet*
- `11/11 20:14` [Goldman adds another rate hike to Fed outlook after hot inflation data](https://news.google.com/rss/articles/CBMimwFBVV95cUxONExVbW1kNl90NWNsM3VkVjlqVUh2cE9JZzY5YzhTTzhyUEVvcS1lNDhadGlNYlJ6OHVoaG9yUFdBTE9MX0l0eEVxaWlZWjBZRjZzRk4zcjBQNnBuN0tTaWJnZzluQTZid1AxSUVYZXAteGVHRjRha3pVUS1OX1lTV3Rtcjh2TVRzMHBTaDBSUmFGWlV6Q2o0UUN3dw?oc=5) — *Google News / Yahoo Finance*
- `11/11 20:19` [Japanese Yen weakens toward 160 as US inflation bolsters Fed hike bets](https://news.google.com/rss/articles/CBMiswFBVV95cUxQUlNIY2JSZ1htZzNjb1IxbUxreTljZFQ5RXM4OHAyM1JBZVFnT24zbDN0UWR6T3kyNUVSTjlrT21xbkt2SUdSNlB5OW41ck5zMzFib0VQMzFSRnp4N0l1S0tHZFZKc1JxMzNoRTkzZk1UUzBQYVNUQTZOWF9uNUl0N3N6S3pibHdEX0FPbnluVGIwaWJta2RMWjlkWXVOb1ZNdy15cUhiakxIVnBXekJlTnAxaw?oc=5) — *Google News / FXStreet*
- `11/11 20:19` [Morning Call Sheet: Fed rates, AI debt concerns and market rotation](https://news.google.com/rss/articles/CBMirAFBVV95cUxNanVweEZ4S2RDelJMU0F4cGRXc0VoUTEzRC1YNHd6Z0dWS0ZyZnBqWDNzN18wRjlpWmVPUkxfQXRSMURQM1FiSEc3Skd2ZlJIbUFKSW9acHdkTDhPMXg1UXR0OU5uNy1kZ2RvQU9keHhBWHJxclVseWw1ZnVRWU41UW9pSlRDMnFYbzA4N0FraVAxTExGd0VnWFVkci1Xbmd2eC1wOVZ1eHdpLS1V?oc=5) — *Google News / CNBC*
- `11/11 20:27` [Consumers are showing signs of strain even as they keep spending, CNBC/NRF Retail Monitor shows](https://www.cnbc.com/2026/11/11/october-retail-sales-cnbc-nrf-2026.html) — *CNBC Economy*：Retail sales rose at their slowest pace in nearly a year in October as American consumers showed signs of strain from high prices.
- `11/11 20:38` [Mortgage rates jump sharply higher, after easing for four straight weeks. Here's why](https://www.cnbc.com/2026/11/11/mortgage-rates-jump-sharply-higher-after-easing-for-4-straight-weeks.html) — *CNBC US*：After falling for a month, mortgage rates jumped sharply higher on a hotter-than-expected inflation report.
- `11/11 20:40` [Goldman now sees Fed interest rate hike at December, March meetings](https://news.google.com/rss/articles/CBMirgFBVV95cUxPZFIzUlJQMWd1NnFDdGRWYUVWNnIxSHhTX0NFUWwtbC1zMkIwOE80R3VSVVRRNnVkSTVxSmdwZGQtTlpuOTBMQ2VkaURQdlU2U25aTG90cTM0ZUtMY0U1OG1XVlNBY0MtdHFSaFhUbE44U0FULTljWmJkZlp3enBLckdzWTlnaWJ0Y0lfVDdrWERwN2tHY0NibzZ0MmVEakxnc2x4LW1WaDI3TlBfQUE?oc=5) — *Google News / CNBC*
- `11/11 21:48` [Ed Yardeni Backs 50-Point Fed Hike in December - iShares 1-3 Year Treasury Bond ETF (NASDAQ:SHY)](https://news.google.com/rss/articles/CBMiwgFBVV95cUxQY2NCTGRDUWRWeEJ2RnJDN2ZBRm1LMld4djAxTUs2STg5MW1IT0VsVzV0am5DQ2dzZS11WE51S0tsUlFXRWNkQ0FwREkwcUE1bTRSRlV0cnhLVkpBZzIzT2VmTTFDZEplbENIb1pxVjFleXhhU20tUU9OVDBYQkpQR2dfQTJkT2VoVzk5a29wSW0ybTFlY2hHUGl2LXJIb0RZbGthcmROV3lEOU9NUkMwc0lyb3otdzdxbndRMGlFS091QQ?oc=5) — *Google News / Benzinga*
- `11/11 22:12` [Goldman adds March hike to Fed outlook as January remains in play (TLT:NASDAQ)](https://news.google.com/rss/articles/CBMipgFBVV95cUxNSEhUSWdQNkRGdkRSVjMxNVJFNG1ic1M5eEVUZF84T2dSUXJhM0JhYjZYdVNXWTd3NHdrMkw2SmgteTQ0Q2JxZnlkcGJFR09EbklmY0FhQWhvTWZNdkJVNS0zLThoM0FDNGppMFhxV181WHFHekMwQU1NODVJcnlNNVpxRWE3T3B3aTJ4ZGRCZzdNd3MtOEhFNXR4aXZpeERWN2Z1eVpB?oc=5) — *Google News / Seeking Alpha*
- `11/11 22:22` [The Fed will hike, and the odds are rising](https://news.google.com/rss/articles/CBMikgFBVV95cUxOeUVTeDBKaVJ4Sk9rMkk4V09BblROUjFGTm94akhNSEl4VHVrOWVTcGVaVmRCV1lfVXBReVpPb2wzelAyQTlhWjRNX0VGbEZ6X0Qyd3dwQUdMWWQxT0Fua2M0aEg0clBnTEFHMGtpVFRmTFZEd0NReHM1clVoTkpoQm82ekc4LUJEUEd6cGx3bmpJdw?oc=5) — *Google News / FXStreet*

**科技與 AI**
- `11/11 19:56` [CoreWeave provides silver lining for neoclouds as Nvidia storm swirls](https://www.cnbc.com/2026/11/11/coreweave-provides-silver-lining-for-neoclouds-as-nvidia-storm-swirls.html) — *CNBC Tech*：Huang's close ties to the White House are in the firing line
- `11/11 20:00` [Nvidia shares rebound over 5% as chip stocks rise ahead of earnings](https://news.google.com/rss/articles/CBMiqwFBVV95cUxPQW9CcFdkaXZIZV9fcTFzX293ZldnYzdUenljSkl4cUo4SFVyZzBHUWJrUXZuS3NVV3BFTENqc0xiNXNVSkpIcEZnUHlzSWJWNHJtQURmVXpxa1BIWEtXSy15ZmlKamY3clJVcGlSUFF1YU1kOEsxOVRJNTliQjZUYlhGVjlPTFhkT2tjMWJBa3FwM1piQWRkZDVKdGZBTGt0X21EYmJjVk9DeFE?oc=5) — *Google News / Anadolu Ajansı*
- `11/11 20:02` [How AI Stocks Are Creeping Into Dividend ETFs](https://news.google.com/rss/articles/CBMihgFBVV95cUxQejBxc0lHeTAzR2ZXUTRtUEhFT0htOC1yUWRKRDZaM2psSDdOYjFCN1A0NEhvV3Nrems3RlZBS3pxclBrdm9BSWl6MXVoMHlza3lEUjFGYURYTWtIWnZPZ3hRdkV1ZEZRb0wxWTc3SjU3RnREaFMyaE9yYlU4U01ub1h0SnIyQQ?oc=5) — *Google News / Morningstar*
- `11/11 20:37` [Can the rebound in AI stocks reverse the downtrend in U.S. equities? Adjustment risks for the S&P 500 remain unresolved, with the 200-day moving average serving as a critical support level.](https://news.google.com/rss/articles/CBMingFBVV95cUxPeTN1bVFZdWFDZVJTYjZZOC0tYVFaNENadDB2STU5ZlNlaHlZd0F6M2lSWDZrRm5Wdnc1Z3NPeDhPQ2p3cklIbWtJSlMxOEdkWWhQM05fOUI1RTVma0M1M0NtOVpmQVFjZS1BS0l1eWcyZDRYVlZBRlphWGNPcThQY3lwUVh4UWIyR09sUW15NWVXV0RjNUhxaEVlendpQQ?oc=5) — *Google News / Moomoo*
- `11/11 20:57` [Tech stocks poised to rebound after Tuesday's losses, helped by CoreWeave](https://news.google.com/rss/articles/CBMidEFVX3lxTE8tQm1ISmJOay02YktGSkdOclZyV0I4R2xDSlkzcTc2MTJ2TjZxNG5FTDExT1NDNi1BdldGY2xvU3lKRk5DeHA0UmVqU0dhVFlYVEZsLWhDSThyVDRBbG1ITy1SYmxKVjdESGx2Y0s0QVctTFhJ?oc=5) — *Google News / Quartz*
- `11/11 21:09` [Nvidia Selloff Puts China Exposed US Chip Stocks Back In Focus](https://news.google.com/rss/articles/CBMinwFBVV95cUxOMnVkcGNhMVlYRlphSlp0dEhnWmc2NHVXdWVhVDFBNGtxSW9uV19sc0s4ZFI5Y09ma2xic1hrdkVxUEN2WWhYbWd1VTRXYmZxUmVOLWh0RXF3Vzk1WUt6UllaRjNBNWNGVVg5RUFMWTBUVTRqRTgxQkdwYmp0NlBOYW9iaVNLV3pjcVp1UFh5X2xsbHozN0hpU3JaelZFdWc?oc=5) — *Google News / Yahoo Finance*
- `11/11 21:46` [Nvidia's Huang tries to project 'business as usual' in return to office after affair revelation](https://www.cnbc.com/2026/11/10/nvidia-jensen-huang-affair-office.html) — *CNBC Tech*：Nvidia employees described the mood as "sad" as CEO Jensen Huang returned to the office after disclosing an affair.
- `11/11 22:01` [U.S. Market Move | Nasdaq Futures Surge 1.91%; Most High-Profile Tech Stocks Rise in Pre-Market Trading](https://news.google.com/rss/articles/CBMimgFBVV95cUxNbjBHanNHVVhmcW9vVnV1d205bDdZblZzRmF5TmswNUl1cElHZnh2b1NxRGIwQmlRSnpqMm9uTjZLd1dzeWFTQ0h0S0xpSF9sV1F2YVpTOXVaOTVFaFRNNFVCeWJ1RTA0QTA5dm5jcnlRWU5kLXc5U0JJb3lXa3cweTI3N2NTTHRtQzNIYzgzaFk4TnJQWTBnVTBB?oc=5) — *Google News / Moomoo*
- `11/11 22:20` [Chip Stocks Have Lost Nearly $5 Trillion. Nvidia Hasn't Been This Oversold Since 2019.](https://news.google.com/rss/articles/CBMisgFBVV95cUxOWXZxSkhzM0pFS1lYbzFPX2RnTThMTEJtcnFvV2xXaGNvaFpfVzJtamlYX2JiWl80b1p0ZndkUURlZ1IyMVczc3R6M2VVeHlOc2ppMHJNNU83Tm9YNDVNeWpKN0VLcjBhMWVjaUczcE1iMUZPMjlRbldUMjhLQ281WHlnYnYxZkNQV0FfdEltZzF6LXRESE91U1VhNk11OVljLWlwaHhaam5JcG01X19ZajB3?oc=5) — *Google News / The Wealth Advisor*
- `11/11 22:22` [CoreWeave raises forecasts to $13bn, sparking rally in AI stocks](https://news.google.com/rss/articles/CBMizAFBVV95cUxPZ25JQVNTR0JLOGMyN0FZdjk4eDVob1ZjcFRCZHd6ZGF6a18zQ181UUZlR3Rtbl9uZ1IxSHZLZUZXN3RoWlVKdU9oMFJZTzAxNEVGWHd3UklsekNtTkxrckdUejFCWlA5VGtRU2dBd3RpVkRHVVBfMUJqOXRaYXlld3R1Xy00cWJrVGhJVGx5akNhR3BtQzhjTWxIUHo1ZGR5OTRrbUxpTEJrQl9jR2VLSk5lSm1Ja2VaSkZzVjl0eTY5VGM0SmdmM3pFcXg?oc=5) — *Google News / The Times*
- `11/11 22:23` [How To Trade SPY, QQQ And Top Tech Stocks, Including Nvidia](https://news.google.com/rss/articles/CBMiqwFBVV95cUxNcHRVdzhZOEcwaHBiQXY3T1JuVHpsTWtRdEZuU3JST0lsR1J6R05kMDJDbTR6R1AtNDFnUDlacXp2MS1XVFh6cGJKZnlzbzc3eHlEbGczWmxjdTIzSWNIMl9iQzRWdExqb0N0VlJwaHFUcXV6OHBBU2Z2RGIxQ3VlMG1NYjhMRlRMN3JvWFpJaE4zNDllZW52UFh3S3NKUFdFczBVeVpUVFNWbXM?oc=5) — *Google News / Benzinga*
- `11/11 22:24` [Would California home prices tumble if tech stocks crashed?](https://news.google.com/rss/articles/CBMipAFBVV95cUxOdXZZVmtWM1g0VE1Ybi1jeFlLX0JxbVNiN1ZaSEJrYktGV2lTUk1uaUljSElOc3pwM191RUthUWdvUTNkdlpHSTUxQnBSREdXRWp5cTFibUUtejBnanl3MHRrOHRPbmp1alVEUkRISlBXaEs5azVWRmVhaU11b3BwUFFlMWlTSUR0V01HckpsejJtRGdSUHlxbVNQci1sR19nMW5GMA?oc=5) — *Google News / Orange County Register*

**其他**
- `11/11 18:06` [American soldier arrested on suspicion of spying for Russia](https://www.cnbc.com/2026/11/11/us-army-soldier-russia-espionage-arrest.html) — *CNBC*：The 41-year-old man is being detained in London under the U.K.'s National Security Act.
- `11/11 20:42` [Rep. Jim Jordan, Sen. John Thune elected GOP leaders in Congress](https://www.cnbc.com/2026/11/10/jim-jordan-congress-republicans-midterm.html) — *CNBC*：House Speaker Mike Johnson, R-La., announced the day after Election Day he would not run for minority leader in the 120th Congress.
- `11/11 20:57` [Hundreds of schools in Italy, UK evacuated over 'hoax' bomb threats](https://www.cnbc.com/2026/11/11/hoax-bomb-threats-schools-italy-uk.html) — *CNBC*：Hundreds of schools across Europe were evacuated on Wednesday after receiving bomb threats, police forces say, amid growing worries of hybrid threats.
- `11/11 21:18` [Nissan cuts sales forecast again on China woes as CEO outlines plan to fight back](https://www.cnbc.com/2026/11/11/nissan-sales-forecast-earnings-china.html) — *CNBC*：Intensifying competition in the world's largest auto market forced Nissan to cut its annual sales forecast for a second time in months.
- `11/11 22:13` [Jim Cramer's top 10 things to watch in the stock market Wednesday](https://www.cnbc.com/investingclub/2026/11/11/jim-cramers-top-10-things-to-watch-in-the-stock-market-wednesday.html) — *Finnhub / CNBC*：CoreWeave lifts the AI trade, and my advice to Nvidia.
- `11/11 22:13` [Zelenskyy warns Ukraine is 'not real estate' as experts say a Russia peace deal remains elusive](https://www.cnbc.com/2026/11/11/ukraine-russia-peace-talks-zelenskyy-putin.html) — *CNBC*：President Zelenskyy warned Tuesday that his country "was not real estate," as a second and final day of U.S.-brokered peace talks got underway.

**盤前個股新聞（Finnhub）**
- `11/11 21:46` [Stock Market Today: Tech Futures Jump As CoreWeave Surges On Results; Nvidia Rebounds (Live Coverage)](https://finnhub.io/api/news?id=ca2a49486304830684c74600d49a55f4bd0eae1f274d9c9d00502c4ce1206dbe) — *NVDA / Yahoo*：Tech futures jump Wednesday as CoreWeave surges on earnings. Nvidia stock rebounds premarket.
- `11/11 21:51` [Update: US Equity Futures Rise Pre-Bell With Nvidia, CoreWeave Gaining Amid Tech Sector Rebound](https://finnhub.io/api/news?id=86fda81b0d9d6a970df1acd9e01fb8ff7a3a1fb77259b5fcd7451013c44fb9c7) — *NVDA / Yahoo*：(Updates with economic data, recent oil price movement, world markets' overview and corporate stock
- `11/11 19:04` [What State Street Tech Select SPDR (XLK) Sold: Apple Leads on Nov. 9](https://finnhub.io/api/news?id=5694dd3f6ccbc633d5fe2f387ec1e09bbeea3489686d30ce985ca7e715528398) — *AAPL / Yahoo*：The fund sold $136.7 million of stock, cutting all 76 positions and adding nothing new.
- `11/11 20:57` [Buy, Sell, or Hold? Wednesday’s Top Upgrades and Downgrades: Akamai Technologies, Apple, BILL Holdings, Corning, Form Factor, Guidewire Software, NVIDIA, Riot Platforms, Toast, and More](https://finnhub.io/api/news?id=7cdf40837326d672aab7d0eada0eb35580bdf7937b88127cfefe5e202c35ae1b) — *AAPL / Yahoo*：Hot inflation data rattled Tuesday's close, and now one chip giant faces a Baird downgrade tied to concerns about its CEO. Which analyst calls, from Apple's upgrade to a steep Akamai cut, deserve your…
- `11/11 20:00` [Northern Trust leans on Microsoft FDEs to boost agentic buildout](https://finnhub.io/api/news?id=2b630d3e2de4da3ede3af91eaad05a3e28695f283c189ac7defaad7189b837dc) — *MSFT / Yahoo*：The financial services firm cut analyst’s data preparation time by 83% with the use of an onboarding agent, the companies said.
- `11/11 20:13` [Microsoft Stock Fell After Anthropic Launches Claude Sheets to Rival Excel](https://finnhub.io/api/news?id=b9e4b92fee3f1ace7785f0e1d29b26ee3b38423d6cd77fe6578dd4a3df9abcc3) — *MSFT / Yahoo*：Microsoft shares fell about 2% after Anthropic launched AI tools for spreadsheets, whiteboards and data dashboards, intensifying competition with Microsoft 365 and its Copilot assistant.
- `11/11 17:14` [Amazon EU may face Italian trial over alleged unpaid import VAT – report](https://finnhub.io/api/news?id=89cf0bffa98d58ba81288d451cdb3e5047859f5e9c8593e2a6b405c35328b25d) — *AMZN / Yahoo*：Italian prosecutors have concluded their investigation and served a notice on Amazon EU, under Italy's corporate liability law, and on the two senior managers.
- `11/11 20:15` [Why Alphabet and Amazon Will Always Need Nvidia](https://finnhub.io/api/news?id=25d5650df933cc5a8580a39961454bc061417fbe4fc8ea0a988cc94b1b202479) — *AMZN / Yahoo*：Google sells its own TPUs and Amazon pushes Trainium, yet both keep writing NVIDIA enormous checks. One supply bottleneck and one software moat explain why the custom-chip threat may be harder to exec…
- `11/11 20:35` [Alphabet or CoreWeave: Which AI Stock Is Better to Buy in a Market Crash?](https://finnhub.io/api/news?id=9952f9b7ba197523596b4b1448966a52b67a59d5cc018a3c908070dc8084bba9) — *GOOGL / Yahoo*：Alphabet and CoreWeave are both positioned to benefit from AI, but one has a much stronger financial foundation.
- `11/11 21:17` [Google Now Has Two of the Top AI Models for Accounting Work, DualEntry Benchmark Finds](https://finnhub.io/api/news?id=0a40df70a73df502d3daa38bcf929f3c16e3c361df2d3545044b5beff7ad9481) — *GOOGL / Yahoo*：NEW YORK, November 11, 2026--Google's newly released Gemini 4 Carbon scored 88.1% on the DualEntry Accounting AI Benchmark, tying for second among 24 leading AI models tested. Google's Gemini 4 Argon…
- `11/11 19:04` [State Street Comm Services SPDR (XLC) Sells $29.9 Million of Meta on Monday](https://finnhub.io/api/news?id=8bc6a80a3287fc7684867f88cd30cd338ec7eacfbd39119fd67309cc2627cc2b) — *META / Yahoo*：The $22.12 billion fund sold $139.5 million of stock and bought nothing, cutting all 22 of its positions.
- `11/11 21:24` [Meta Fined by Brazil Electoral Court for Allegedly Failing to Provide Equal Content During Presidential Campaign](https://finnhub.io/api/news?id=4c51f312537b484e6acd09cf2a51c496b86a78f396a53c1808d9d20a6e979276) — *META / Yahoo*：Meta Platforms' (META) Facebook was fined 7.5 million Brazilian reais ($1.34 million) by Brazil's Su
- `11/11 20:12` [Tesla wins major European approval with a surprising catch](https://finnhub.io/api/news?id=510ad94bb96127a51083a7f99800bb485b7e82f557d5c0eacd4f20f53fa32014) — *TSLA / Yahoo*：The decision allows Tesla to offer its supervised driving technology to Finnish customers while broader European Union approval remains pending.
- `11/11 22:20` [How Much Is Tesla’s Robotaxi Actually Worth?](https://finnhub.io/api/news?id=b07bda4fe1d6414a7c54c85a567ec87bc7186a87bcdd54d59725d11968cf56b9) — *TSLA / Yahoo*：Tesla's Austin geofence just grew, unsupervised miles are surging, and a $1.5 trillion valuation leans heavily on the story. But Robotaxi has no reported revenue yet, so one future earnings report may…
- `11/11 21:28` [TSMC plans up to $1bn investment in Anthropic, OpenAI](https://finnhub.io/api/news?id=0598aa17c435cc9df026fff821e7d70ac548cd8e76be3bff6ff8ce616b0e3c5b) — *TSM / Yahoo*：TSMC's board approved an investment of $500m to acquire Anthropic common shares at the company’s IPO price.
- `11/11 22:05` [Supermicro Now Shipping Next-Generation Servers Powered by 6th Gen AMD EPYC™ 9006 Series CPUs for Cloud, HPC, and Agentic AI Workloads](https://finnhub.io/api/news?id=cf3c422b38f76ee52ceea917ceede4ec54fadee329ce4d378d206838a4766e84) — *AMD / Yahoo*：Super Micro Computer, Inc. (NASDAQ: SMCI), an AI, Enterprise, Storage, and 5G/Edge Total IT Solution Provider featuring Data Center Building Block Solutions® (DCBBS), today announced that its H15 gene…
- `11/11 22:05` [NVIDIA Rises 4% Despite Baird Downgrade and Huang Uncertainty; AMD Climbs 5%, Broadcom Gains 4%](https://finnhub.io/api/news?id=56f6337331c38c400fb24dfbc317ebfff5be2066750c699cd4bfecd7d96470ff) — *AMD / Yahoo*：Baird slashed its NVIDIA price target by more than half and questioned whether Jensen Huang will still be CEO in 2027, yet buyers rushed in anyway. What spooked one analyst, and what is Goldman seeing…
- `11/11 19:04` [US High Relative Profitability (DUHP) Adds Broadcom and Micron on Nov. 9](https://finnhub.io/api/news?id=08e739707fd6c3c576d9b2d5410f7b08569074fc95ae4dc35a9f0eff475f105b) — *AVGO / Yahoo*：The fund bought $125.9 million more than it sold, lifting 75 positions and trimming 4.
- `11/11 21:41` [Goldman Sees Anthropic Revenue at “$685B by 2030.” Broadcom Is Tied to $161 Billion in Lease Deals](https://finnhub.io/api/news?id=7ff06e4f736e2a7050362911171299179e013b7c5d5593879bbd9d384dcdebe6) — *AVGO / Yahoo*：Key TakeawaysGoldman Sachs analysts see Anthropic’s revenue reaching about $685 billion by 2030. Goldman is co-leading Anthropic’s IPO.
- `11/11 21:52` [Little-Known CoreWeave CEO Line Helps Fuel 15% CRWV Stock Rally](https://finnhub.io/api/news?id=dc107c2ae21f4f6e1d555ee396ae89bad98f342c4e4066cfd4f43b79711c0d16) — *CRWV / Yahoo*：CoreWeave stock jumped 15.6% premarket after its CEO ruled out unsecured debt at current yields, citing $9B in prepayments.
- `11/11 22:23` [CoreWeave: Things Aren't The Same (Rating Downgrade)](https://finnhub.io/api/news?id=d3f8720578303219306b61e938cdecf811bf43d221c0bb09223657865178fd70) — *CRWV / SeekingAlpha*：CoreWeave is downgraded at current levels because October and November set new lows and lower highs for the stock. Discover more on CRWV in this update.

### 早盤新聞（台北 11/11 22:30–01:00）

**聯準會官員談話**
- `11/11 22:39` [Fed’s Bowman Warns of a ‘Large Correction’ in Stocks if AI Bets Fall Short, but Says the Inflation Fight Isn’t Over](https://news.google.com/rss/articles/CBMi4gFBVV95cUxObFJMWTdRQkxpSkx1Z3g2ZC1vNHhxeXRfU3BsbnlUV21meGJSaFdsdmM3WG1yWmppampWMEZDUG5fekZxbWxCbG9WSmJ1OS14MnhRMDcwSFVZRlpmMGM2dGw0bXo3ZEpMUnA1X09GamF5WW1kWWw0dy1sSGJuWFVlRzFHVEQtaloyRm1PZDM0MHY0UE1SMC0yb01QbkJUVHlsemM3NWpiaWVVSEdvZ1BaczNzMXM4dEFDdlpDby1EZFBlX3ZyVmZ4VmFkMVRLRTkyTmxUTDlFenBkX2FBS3Y0aVlR?oc=5) — *Google News / InsuranceNewsNet*
- `11/11 23:19` [A Fed Governor Called the Latest Inflation Data 'Borderline Disturbing.' A Half-Point Hike is Now Firmly on the Table.](https://news.google.com/rss/articles/CBMigAFBVV95cUxQS0NGczVPbXByWWtEcnlrUnJ1d25nRTQ2cm5xanRaY1I4el9nLXYyRXNZMkxEbXRpN2dBTnhTM0daUEhtQXJjZG5OYlRVQ1BXWFVTUEtDQjFNbEhPSmJiV2RIa1UydXNFamxwRVZCbVR6T2dPejJ1Vk5idmUxbnNvUw?oc=5) — *Google News / International Business Times*
- `11/11 23:22` [Fed’s Barkin Admits Inflation Fight Isn’t Over, Warns of “Patch and Pray” Economy](https://news.google.com/rss/articles/CBMitwFBVV95cUxQRlc4MVlQUW9iY1JfQTVHRmlFcTZNYjA5bWJ5VzBaNm1TYUhvVFVsZFdCTHVOcEFUYkNVTUdKUWlyd1RsTGZIZEM1em9Xcnh5LUpTWkx0SDZhN0Z3M2V1Y3dzWExEVnA0Y1I2N09QTmxMT1ZtbXBDOWF5N0JIejNURmQ5ZkRRRzJCR0J1cHk2NllkYXNyQXh6WW1zZGtkcHhvRDhBcGlLU0J3bnhvcElVSmZZelN4TGM?oc=5) — *Google News / SchiffGold.com*
- `11/12 00:10` [Democrats Slam Fed Chair Kevin Warsh Over Secret Donald Trump Calls and Missing Calendar Records](https://news.google.com/rss/articles/CBMif0FVX3lxTE0yWGJ3bXc5YTEyazh0WExiRGlYWFhXQm55SkdzT3BsTjVjNTc1OFRDekI0QTRIdC1UUWpMb1BfX1hGN0FieGlEMXVDX0lDcks3ckJRZFFTUll5eEpoR3FHV01wd3ZOQzR1MWlPbXl6Yi1mUjJmNm14ZnBUS2JCbEE?oc=5) — *Google News / AOL.com*
- `11/12 00:17` [Dallas Fed’s Logan flags two more rate rises, sees scope for faster winter tightening](https://news.google.com/rss/articles/CBMiyAFBVV95cUxNMmU5OVNSMEFmQ1JNQ1phQ1VkNVptYTJxVmhEcUE2Q05tWWkzTVNEdlRBV0p1aFhIdW5IOHREWlBFdWxhVTFiTi10QkItTmNVMWFwYnFZRkRBUlVZSmJYYmUyX2lodUV3NkV2UEdqNDBHb0JWcWYwcmR2c01PMWF0N2NqeWllbWlaTnI1ZE81S09HNjJuM0lmTzBwX19UbUd0eEtTU1dTaVg3UGYyazJBd2UxM3Z0SE1pN3dPdVZVdk83cWlrYzZoXw?oc=5) — *Google News / VT Markets*
- `11/12 00:19` [NY Fed's Williams retirement date is debatable - report](https://news.google.com/rss/articles/CBMilwFBVV95cUxPQlBOSWxUSWZJU3lIYWFaMjl6aWtTaFdCbGR2Q3RmYTVVY0R1Rm5NeHNQQ3p4VER0QU5FUmxhYmdMRV8teGpVdEdQRTJ4LVhuNEFYNTBQOGhNQXNLajB0WDFhSHRQNUNDWG9la0tuc294N2NocG11OHFOWHFselFuNmdLMkpHVG5EZjNyQ1dlXzVmRkthdFFN?oc=5) — *Google News / Seeking Alpha*
- `11/12 00:49` [Fed's Goolsbee calls inflation report 'borderline disturbing' as central bank weighs more hikes](https://www.cnbc.com/2026/11/11/goolsbee-october-inflation-fed-rates-disturbing-distressing.html) — *CNBC Finance*：The consumer price index rose more than expected in October, raising alarm among Federal Reserve officials.

**總經與聯準會**
- `11/11 22:30` [US real estate stocks slide as hotter inflation fuels Fed rate-hike bets](https://news.google.com/rss/articles/CBMirwFBVV95cUxOVF9ZbThrZjRhSThEMTRzbmpaUHRULTVBWkFtR0FINEliOFRQSi1mQUMyNzhFMkdLQzEwVHRGNlVwRGZfN0pubEI0M24zeXpoUVpxa2d3OEUtLUd5azZfNzJaOTRfOGtONUhnNXROd2ZDQUtsTGhpV1ZpbGNiVU9DcVd0MnNVWkJPaXUzX1ZWZUdLaXBGNDVpLTBXRWJqOEZYUXg3UzNRSWRHQXdraDNz?oc=5) — *Google News / Anadolu Ajansı*
- `11/11 22:31` [Global bonds reel as hot U.S. CPI and hawkish Fed revive 50-bp hike fears](https://news.google.com/rss/articles/CBMiwgFBVV95cUxPNFFxb3VONWVqcTN4Z2NSYWdoT3ZFOXY0QXBkeFZwOFAzakJoUlpmOGl0RXZFdDFOcVNXUWVDZWhIZDFUMDE2cmNLWVlvaXVROERZYThXQVU0LXo1bHBYazQ4RXY1Y2N1b3dBN1REbHJxNDVQcjd5TzFzLVFZdnBXMHNJdEJvend3WkNBSnJ4Sk5LZk1mVGEwVU51M1NLU25mNTU0aG1VeUVPWlJMODNIZ1UyN2FSRVRvSWhiMjNfOWg5dw?oc=5) — *Google News / Investing.com*
- `11/11 23:12` [Gold struggles below $4,050 as Fed rate-hike bets boost US Dollar](https://news.google.com/rss/articles/CBMiogFBVV95cUxQWnBKQTBxWDhUZkNMZ2IwUUwwTTM1OXNuemNfRUEtb1FTVFcwcnlHVGJZbk5vLUttZU54LV9lVmx6LUR3dWFCMFI2MncwX09EQU5XampvUk52N0lybmdkczVIVzBpNzg4YWxfMUpxUVA0MTNTYjJ4QUtPbDRUTVZ0LW9qRTZ0d1ZQcFd1N1JUNDgtbWZNcFdUeVRQMkdfeEpLV2c?oc=5) — *Google News / FXStreet*
- `11/11 23:18` [Inflation Report Doesn't Warrant 50 Bps Fed Rate Hike, David Rosenberg Says](https://news.google.com/rss/articles/CBMijAFBVV95cUxNS3lPVUh0RzlBam4xdmx6ZXZUYlc2TWFPMEdya21QYzRaVnUxT2JXMFVMNnRELXJOUlVIVlR4elhCR1VZNmpOTWZoOUhuRjBYU2tzdXA2VVYzTGhIdkc0eE1aMEM3Z2VaNktSQUNYcmYzTFVNY2toZ0laVUh1UDdEWXljR1dJOHJGVjdGXw?oc=5) — *Google News / Yahoo! Finance Canada*
- `11/11 23:32` [Inflation data raises stakes for the Fed’s December decision](https://news.google.com/rss/articles/CBMilgFBVV95cUxPNC1OSjhnSDZpWml3YTgydDdEVnB3amU3LVhiYTVQUzdxRUN4Yjd3UzNjZTJzeWUta3ktN0NxY1FfYkU1Y0NWaFV4UjdBMzN4X2xmeXkyZjR4MDlCdThaTjFTYkRhYWMzM2hTM2s1TjBnVF9QSWFzbG52YTRtSUhMOUhqQmNnNVQyc3hZcndDdm9LMGNRMVE?oc=5) — *Google News / Baton Rouge Business Report*
- `11/11 23:45` [Dollar Rallies on Expectations of More Fed Rate Hikes](https://news.google.com/rss/articles/CBMiqAFBVV95cUxQc3dpWDdLVjRvQVFmYk94RlR3di10WWZtNXVUY3RLbXppQmFYMDZibXlPYTFudmFDWGF6cTB4OTB1Q1FLbFd2NFp1S3BSSHdsRVYtSFNqQ0FqMnNKd2R5elpaQ3pWc0lYOFBKbFg2QlFGR2lHeU96STZJZ2UwMGRFdXNLd3A3NmExeTdIYmtQYkJteG9KNGh1QTdldW50UmFsRHZPWlRhV2E?oc=5) — *Google News / Yahoo Finance*
- `11/12 00:32` [Why You Should Sell the Euro Now Before a December Fed Rate Hike](https://news.google.com/rss/articles/CBMirAFBVV95cUxQWVJNUWVsQS1jUk9SWDlDSXA5djZRSnN1WVBqMGdQS1NUNXBoeHA4NjJfZTQ5eTBzaldsT1BIdTVDc0xRTFVlZzQwZVRTVlVnMjJOSUVnOUtDZGVMVmZFTzNQNHVLc050cmxrZURZTEN3VUFzbjNPMllHUUtNNHhWcml3Tm5WMi1ZWlFIZF9RZGJYR1ZycDcyTTVmTzBDelpSdFplTW83TThpNWRq?oc=5) — *Google News / Barchart.com*
- `11/12 00:36` [Dollar advances as markets price in December Fed rate hike](https://news.google.com/rss/articles/CBMixwFBVV95cUxNWGdtalR2a0hLUGllQl9GUEdTRWVIVzFGc0xuMHBjV0sxMkhPUm5sMkNvNldJNDk0eUc4Y3ZiOFJfUkNDeGw3R0F1MUpkaGlUdHRXV3FmZEprcEtNcmN1bHpHU2RzQ1UzVkZ1SGJVNUVJZnhrb2NBelRqSjJHdGNDa3Z4R2diUms1ZEdpSzN2bWV0Z1hxWjdoR2xhUHR2MFRIZzM5cUk4S0pfTmxBYnhTOFowS3o0V0JGN01zbUFwcUJfNXY1a013?oc=5) — *Google News / TradingView*

**科技與 AI**
- `11/11 22:39` [Jensen Huang's affair with alleged Chinese spy further complicates Nvidia’s China problem](https://www.cnbc.com/2026/11/10/jensen-huang-nvidia-china-ties.html) — *CNBC Tech*：The allegations against Jensen Huang have brought new attention to the Nvidia CEO's potential influence over U.S. policy on chip sales to China.
- `11/11 23:10` [Nvidia stock rebounds, chip stocks surge as CoreWeave earnings boost AI trade](https://news.google.com/rss/articles/CBMizgFBVV95cUxOV3ZDNTlBT1d6a3V4VnJIYzdBaXZOWHBjTGlHUEl2QUw2RmlrUy1lLVZsS2lWYzVIbC1waF84VDNVellXQWpSby1CU0pscmg4YS1nTHI3QmJySGFscHJmdTB4UlhoemRScHA0UGhyUmdiV2tvMEM4M1JnMnVRSzJ0MXhCdHkyZXJJWjhpWUxtRFZOaVd2X2dZemhOSnVzSFNCbndLWTlGWmV3RkFCajJYQUJETXpJZzJabGZLYXY5MGtWNkpvbHFlN29BMmdiZw?oc=5) — *Google News / Yahoo Finance*
- `11/11 23:40` [Semiconductor stocks rejoice after CoreWeave hikes guidance](https://news.google.com/rss/articles/CBMipAFBVV95cUxNWmQxVUlwXzI0U2x1WVhwV3huckxxLVpiU3pjb194eldUSGpwLWJDQU81RUI3ZDFSRHBZRC1iNGlsRE9XSFRXU3JFOHMwU3EtblZKejFGeHlvSk5rSFNpT202Rk8tZXM2RU5CSk5rcDBJSUxWdWdJVzdydl8wVm9DRmRSSlI0b1JUWGthUjZmQUNyemladldKaktGcEMtMDBiYTNfcQ?oc=5) — *Google News / FXStreet*
- `11/11 23:44` [CoreWeave Leads Tech Stocks In Rally—Up 21%—After Earnings Eased AI Debt Fears](https://news.google.com/rss/articles/CBMiyAFBVV95cUxNSFpTRlVaOEwwRG44Z25UNlhYWFpIRVFUS2hvcnpKRHVaejFTc09JaHduZWhVbVVQOHBfMHp5dUplMWtlUDQ5ODBRUlE3Zlo0MUFuRW4tX3BUWElIZFRoLWFfMC1Yd1BzZlJnZkNuR1JxVHI0N1ktSjJCdUNtYnh4S1dKcUJQU2xBTnhkb1hhU1RZQmVQekV6aHA5OW1uN2ZPWHp5Ny1YeUpReFNDZnNFZk9kaHpoaDRMRkN1b2NZNHJYbWRhdHh5Rw?oc=5) — *Google News / Forbes*
- `11/11 23:53` [CoreWeave CEO Intrator tells CNBC he has canceled his planned share sale program](https://www.cnbc.com/2026/11/11/coreweave-ceo-election-data-center-buildout.html) — *CNBC Earnings*：After CoreWeave beat on revenue and upped its forecast, CEO Mike Intrator indicated the company will be fine despite last week's Democratic midterm sweep.
- `11/12 00:00` [Tech Stocks Add Over $500 Billion as CoreWeave's Blockbuster Results Rescue AI Rally](https://news.google.com/rss/articles/CBMiqwFBVV95cUxQek9wUW01SjFDSmE0dThwM01fWU1TSmdCeVpGSTQxcTJfak9XQ2FRdFRiLXFTckZaWHZYX2FhOTN3M2NCanBDSk5EeDBfbEhUbEw5eTlYNWZ3Y0FwVG9IVVFjajEyZ1h1ZUFBbVduVnNSeTQzdGxkWjlDSV9RV0d5cld1cnU1TU9SakJUdWpENURZT2xPX1RzQkFybkpxQWppdkVfYmEzSG5UYTQ?oc=5) — *Google News / Benzinga*
- `11/12 00:05` [Tired of spending time fixing other people's AI slop at work? How to talk to them about it](https://www.cnbc.com/make-it/2026/11/11/push-back-coworker-boss-extra-work-ai.html) — *CNBC*：Executive coach Melody Wilding shares four ways to address the extra work landing on your plate without the conversation becoming confrontational.
- `11/12 00:07` [Chip stocks reclaim 10,500, but the real breakout is still missing (SOX:) (SOX:)](https://news.google.com/rss/articles/CBMipAFBVV95cUxONFJxTmFGb0NKR21aeUhFc3BGeUt4NVc5WUJYbmE5cGlHM3NYZ19YSnFwdEdOSm1RWUM5SlpkQ2tlZE14aUtMcDdjeFZBUjZMdzdldDBWb2pyVEJFUm84V2NiNGhIeXNZaEtNVjNJd0YxRnhkRDBsU0RGREdqc2xRcnM3QUdyNEd2ejRmWUlzY1dYVUJWNlppYjFSZnB1ZTBtcWlkUA?oc=5) — *Google News / Seeking Alpha*
- `11/12 00:08` [CoreWeave's strong results lift chip stocks; Micron leads Quant rankings (SMH:NASDAQ)](https://news.google.com/rss/articles/CBMirAFBVV95cUxOM0Z1ZnJIUGFIMklpbVdQM1UtelR6Z0tJdGFObTZ4amVwd1pCZlZ3Mk1kMG9SSkJYc2pwZi1sNGVqdWZqMUd0T0xwTDlaTUljYTQxUThfY0FJUmhUZG1lc01fM2w0Z2U4M1BvWmZ3Z2dzVnZ0MklaXzhvUFE0U0VxV1phWS1JNTZWQmN5bDhSMi1PNVg4YUJJUXdTR3psbUFNZ1VGNzZOdGZkWVNu?oc=5) — *Google News / Seeking Alpha*
- `11/12 00:41` [Mortgage Rates Back Under Pressure as Tech Stocks Bounce](https://news.google.com/rss/articles/CBMimwFBVV95cUxNQmRuRUNFcVAwbHJESTJUUEUyU1BIVS1TUUU5N3Q5Z1ctdXR5VVRaREpfbDhwNmpwVTktMXRfVHlYdHQzek45eGZITUcxY1k4Z2VHY2VZYUg1bVdEOHJJaFlBanl0QS00cXREeXVkWkhQT2Foc2RTNkRIcDVmX3BXeEl3SHFKUV9weVU3UlIyYjJLREVoVEdmczlxOA?oc=5) — *Google News / The Truth About Mortgage*
- `11/12 00:51` [Schumer accuses Trump of defending Nvidia CEO Jensen Huang at cost of national security](https://www.cnbc.com/2026/11/10/moolenaar-nvidia-jensen-huang-affair-chinese-spy.html) — *CNBC Tech*：Rep. John Moolenaar, R-Mich., the Republican chairman of the Select Committee on China, told CNBC he expects answers about Jensen Huang's relationship.
- `11/12 00:59` [Jim Cramer's plan for our Intel position. Plus, a rotation loser to buy here](https://www.cnbc.com/investingclub/2026/11/11/jim-cramers-plan-for-our-intel-position-plus-a-rotation-loser-to-buy-here.html) — *Finnhub / CNBC*：The Investing Club holds its "Morning Meeting" every weekday at 10:20 a.m. ET.

**其他**
- `11/11 23:00` [JetBlue's CEO has pushed back on creditor fears and is betting on premium to return to profitability](https://www.cnbc.com/2026/11/11/jetblue-ceo-premium.html) — *Finnhub / CNBC*：JetBlue will give reporters a look at its new first class seats on Wednesday.
- `11/11 23:14` [Measles is surging in the U.S. — and outbreaks can cost millions](https://www.cnbc.com/2026/11/11/measles-outbreak-us-economic-cost.html) — *CNBC*：A new tool developed by Johns Hopkins researchers estimates that the 2025 measles outbreak in the U.S. carried an economic cost of more than $37 million.
- `11/11 23:20` [WNBA Finals MVP Gabby Williams: 'Our existence is political'](https://www.cnbc.com/2026/11/11/gabby-williams-embraces-role-in-wnbas-political-and-cultural-debates-.html) — *CNBC*：Golden State Valkyries star Gabby Williams speaks about winning the WNBA championship, being named Finals MVP, and the culture war over women's sports.
- `11/11 23:33` [FDA warns of vision loss risk linked to popular GLP-1 drugs like Wegovy, Zepbound](https://www.cnbc.com/2026/11/10/fda-warning-vision-loss-glp1-wegovy-zepbound.html) — *CNBC US*：The agency has found evidence of a potential increased risk of a rare eye condition linked to the use of GLP-1 drugs used for weight management and diabetes.
- `11/12 00:59` [Bessent moved to military base amid 'growing threats' from Iran](https://www.cnbc.com/2026/11/11/us-iran-war-trump-hormuz-bessent-threats.html) — *CNBC World*：Bessent's role leading Operation Economic Outcast has made him one of the Trump administration's top targets of Iranian threats, The New York Times reported.

**早盤個股新聞（Finnhub）**
- `11/11 23:45` [Nvidia is Now Down More Than 25% In 30 Days: This Wall Street Desk Predicts 205% Returns](https://finnhub.io/api/news?id=6b781d4135a96c03567322bee6d329f7697bff0dc5574640530ebaff29a090e5) — *NVDA / Yahoo*：NVIDIA posted a record quarter and still lost a quarter of its value in a month, while earnings estimates kept climbing. A swelling balance sheet and one looming report may explain why investors are b…
- `11/11 23:53` [Wall Street is cooling on Jensen Huang's Nvidia — but history says don't give up on the stock just yet](https://finnhub.io/api/news?id=8f33137fad56b117a727fd73024c2072cbebdeb8752f81ddf97d8c01807c2f3a) — *NVDA / Yahoo*：Morgan Stanley and Bank of America pulled Nvidia from their best-ideas lists, but nearly all analysts still rate the stock a Buy.
- `11/11 22:52` [Stock Market Today (Nov. 11, 2026): Nasdaq, S&P 500 gain after CoreWeave earnings beat; Apple hits new record](https://finnhub.io/api/news?id=3aa63f4e93691d6ce01bd0796b32d29265849fada10cc82dbb1a311c3a065f52) — *AAPL / Yahoo*：CoreWeave soars after the cloud provider reported better-than-expected quarterly revenue and raised its revenue outlook.
- `11/11 23:27` [Why the iPhone Duo Could Kick Off a ‘New Decade’ for Apple](https://finnhub.io/api/news?id=c9d8d94a905cf44630bf97b13336927b6252a449bb07179fb8bedcb9c93e0a97) — *AAPL / Yahoo*：Apple stock is having its best year since 2024. One firm believes the iPhone Duo could unlock a decade of potential.
- `11/12 00:00` [Sequoia Backs Microsoft Veterans' Edelbyte with $70M to Challenge Data Infrastructure Giants](https://finnhub.io/api/news?id=f958c34087dfbe67266997e4c6b9f78f62a91986468cb8f70b243c064a98637f) — *MSFT / Yahoo*：Four Microsoft veterans are targeting the $300B data market currently dominated by Snowflake, Databricks, and ClickHouse. Roughly a year after founding , the team has already secured leading enterpris…
- `11/12 00:54` [Microsoft Stocks Decline 2.5% as $1 Billion Anthropic Stake Tests Partner-Rival Balance](https://finnhub.io/api/news?id=44efdaf6a59686b8377175b41092bd1398aacceb048312ab6735fbd3c333e39b) — *MSFT / Yahoo*：Microsoft joins Anthropic's IPO placement while Azure collects fees on Claude sales under joint go-to-market arrangements.
- `11/12 00:00` [Will Rising AI Capex Weigh on Amazon's Free Cash Flow Prospects?](https://finnhub.io/api/news?id=de65b108c8f0f3c249c6a33385c933509c744132e71a285d568f22fcf2d5db37) — *AMZN / Yahoo*：AMZN's $220B AI capex plan is squeezing free cash flow despite surging AWS growth and stronger operating profits. Can its cloud momentum offset the strain?
- `11/12 00:35` [A Major Catalyst Could Send Amazon Stock to New Highs](https://finnhub.io/api/news?id=0019f903aa11dd59a66b3e5f43405c1e6c1d569763ee706d21ca00f10a047283) — *AMZN / Yahoo*：Amazon just posted its fastest cloud growth in nearly five years, yet a $220 billion spending binge has pushed free cash flow negative. Which force wins out, and how far could the stock run if the bul…
- `11/11 23:30` [Alphabet explores first Hong Kong and Singapore dollar bond sales worth up to $3 billion](https://finnhub.io/api/news?id=8552e50318542b49a04d15c40e77c496333b21cf4ce249aa899aacedf9acc47e) — *GOOGL / Yahoo*：Alphabet Inc (NASDAQ:GOOG), the Google parent company, is exploring its first Hong Kong and Singapore dollar bond sales worth up to $3 billion combined. The technology group is targeting between $1 bi…
- `11/11 23:55` [Eric Schmidt, Former Google CEO, Should Run Nvidia](https://finnhub.io/api/news?id=a564b5c6bbc0a10b556a2c347633ce2106c449372e431daf2a17f8542acbf87a) — *GOOGL / Yahoo*：A law firm probe into Jensen Huang's ties to Alice Huang could end his run at Nvidia, and the company has no obvious heir. Who could Wall Street, governments, and customers all accept in his place?
- `11/12 00:00` [Meta Wins CNCF End User Case Study Contest for Accelerating AI Infrastructure Deployments From Hours to Minutes](https://finnhub.io/api/news?id=7da5f82cfc007ed5ec623ee26c6def6f2d1e5799df6e7f9f24257aed7c70c1b2) — *META / Yahoo*：KubeCon + CloudNativeCon North America 2026 -- The Cloud Native Computing Foundation® (CNCF®), which builds sustainable ecosystems for cloud native software, announced Meta as the winner of the CNCF E…
- `11/12 00:24` [Reddit Rides on Strong Ad Growth: Can It Outpace META and SNAP?](https://finnhub.io/api/news?id=76fb9adb18675ffd961a8a47fe2fd2fb727d5b600abdc1b98e72fcc13cafca1d) — *META / Yahoo*：RDDT's surging ad revenues, AI-powered targeting, and expanding advertiser base fuel growth as it battles Meta Platforms and Snap for digital ad dominance.
- `11/11 23:45` [Elon Musk Warns Tesla May Have to Refund Cybertruck Deposits as Production Winds Down](https://finnhub.io/api/news?id=963b19ad86e4bff5221b10a13042ddff8a52fcb81f9f9c73b7ebe0b7b3cc5916) — *TSLA / Yahoo*：Some Tesla customers hoping to get a new cybertruck are going to be disappointed. CEO Elon Musk says Tesla is winding down production of the popular car at its Texas factory next year and there won’t…
- `11/12 00:40` [Tesla Clears Major Hurdle in European Self-Driving Push](https://finnhub.io/api/news?id=c90f3aafb63a339eeea0f92c9335be6408ef3af5ef8e5547a6c888539d141f28) — *TSLA / Yahoo*：A small regulatory victory could unlock much bigger opportunities.
- `11/12 00:07` [Something happened in October that changes the TSMC stock debate](https://finnhub.io/api/news?id=3aa25be6a3c9a02d5222bdc72aa0b37ee5b29a4e68c42cc89f29c2423bb6717d) — *TSM / Yahoo*：TSMC didn’t issue guidance with its October numbers because one number did all the talking.
- `11/12 00:55` [TSMC Stocks Climb 3.37% After Record NT$586.9 Billion October Revenue](https://finnhub.io/api/news?id=aa13a6525794366c993ea8409ee504ee238ccd6f794bf44b0b1af6908363e001) — *TSM / Yahoo*：October sales jumped 59.7% year over year, and TSMC's investment arm joins Anthropic's IPO placement with $500 million.
- `11/11 23:00` [AMD Rises 4% as CoreWeave’s Raised Outlook Revives the AI Chip Trade](https://finnhub.io/api/news?id=6d228c95118a908d0aac797143478b6ff8d81ea9b818f6b13cc8c93d4d7ab931) — *AMD / Yahoo*：Key TakeawaysAMD jumped 4% after CoreWeave raised its 2026 revenue forecast. CoreWeave’s growing backlog supports AMD’s guidance for server revenue to double next year.
- `11/11 23:28` [Supermicro Begins Shipping New AMD-Powered AI Servers — Claims Up To 1.8x Higher AI Throughput](https://finnhub.io/api/news?id=8eab8562ad2c7b262b9ac6a6ccab7e4c5af1632eba4376e8054b658b50f70f60) — *AMD / Yahoo*：The company is also expanding its AMD-based rack-scale offerings with the Helios and Helion platforms, though it didn’t disclose shipment volumes, orders, or expected revenue.
- `11/12 00:59` [Broadcom Stocks Climb 5.18% as Goldman Calls Chips Oversold and Anthropic Stake Lands](https://finnhub.io/api/news?id=22db75f1d54fb564eb26cb36504f4bf02ef537275ef2d772912ab036426fbab9) — *AVGO / Yahoo*：Goldman Sachs says Broadcom trades at 13 times a no-growth 2028 baseline, while Broadcom commits $1 billion to Anthropic's IPO.
- `11/12 00:03` [CoreWeave's Q3 Results Reinforce Growth, Financing Outlook, Morgan Stanley Says](https://finnhub.io/api/news?id=213a77fdad84e1e9e2a71f7a12ae969ac9176390705d37b18fcf16fcce87df61) — *CRWV / Yahoo*：CoreWeave's (CRWV) Q3 results topped expectations for revenue and adjusted operating income, while h
- `11/12 00:56` [Applied Digital Rallies 8% as CoreWeave’s Results Revive the AI Data Center Trade; Cipher Digital Jumps 10%, IREN Surges 9%](https://finnhub.io/api/news?id=bee13e8a28a71037b9a43e899b78aa50336896dd3503222c79692ba92c61b32d) — *CRWV / Yahoo*：After a brutal month, AI data center stocks are snapping back on a single piece of financing news from CoreWeave. But Applied Digital's debt load and a hawkish Fed could still decide whether this boun…